In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path(
    "/Volumes/T7/More Than A Hack 2027/Initial Challenge"
)

N_FRAMES = 199_319
FPS = 25

PATHS = {
    "kontakt_04": (
        PROJECT_DIR
        / "output/contact_classification/submissions"
        / "contact_state_shared_D.csv"
    ),
    "status_06": (
        PROJECT_DIR
        / "output/stoppage_detection/submissions"
        / "contact_state_stoppage_v1.csv"
    ),
    "ml_08": (
        PROJECT_DIR
        / "output/player_status_ml"
        / "full_match_status_diagnostics.csv"
    ),
    "ml_08_oof": (
        PROJECT_DIR
        / "output/player_status_ml"
        / "public_oof_predictions.csv"
    )
}

# Prüfen, ob alle Dateien vorhanden sind
for name, path in PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"{name}: {path}"
        )

# Gespeicherte Vorhersagen laden
contact_04 = pd.read_csv(PATHS["kontakt_04"])
status_06 = pd.read_csv(PATHS["status_06"])
ml_08 = pd.read_csv(
    PATHS["ml_08"],
    usecols=[
        "frame",
        "public_training_region",
        "AdaBoost_D_score",
        "AdaBoost_median91"
    ]
)

# Frame-Zuordnung prüfen
expected_frames = np.arange(N_FRAMES)

for name, table in [
    ("Kontakt 04", contact_04),
    ("Status 06", status_06),
    ("ML 08", ml_08)
]:
    assert np.array_equal(
        table["frame"].to_numpy(),
        expected_frames
    ), f"Frame-Zuordnung fehlerhaft: {name}"

print("Alle Vorhersagen erfolgreich geladen.")
print("Frames:", N_FRAMES)

print("\nLabelverteilung aus Notebook 06:")
display(
    status_06["label"]
    .value_counts()
    .rename("Frames")
    .to_frame()
)

# Vergleich der Statusentscheidungen innerhalb der
# Halbzeiten, für die das ML-Modell D/L ausgibt
ml_valid = ml_08["AdaBoost_median91"].isin(["D", "L"])

heuristic_status = np.where(
    status_06.loc[ml_valid, "label"] == "D",
    "D",
    "L"
)

print("\nSpielstatus: Heuristik 06 vs. ML 08")
display(
    pd.crosstab(
        pd.Series(
            heuristic_status,
            name="Heuristik 06"
        ),
        pd.Series(
            ml_08.loc[
                ml_valid, "AdaBoost_median91"
            ].to_numpy(),
            name="ML 08"
        ),
        margins=True
    )
)

print(
    "\nÖffentliche ML-Trainingsregion:",
    ml_08["public_training_region"].sum(),
    "Frames"
)


In [ ]:

# Out-of-Fold-Vorhersagen aus Notebook 08 laden
oof = pd.read_csv(PATHS["ml_08_oof"])

print("Shape:", oof.shape)

print("\nVorhandene Spalten:")
for col in oof.columns:
    print("-", col)

print("\nErste Beobachtungen:")
display(oof.head(10))

# Prüfen, welchen Videobereich die Datei enthält
if "frame" in oof.columns:
    print("\nFramebereich:")
    print("Minimum:", oof["frame"].min())
    print("Maximum:", oof["frame"].max())
    print("Eindeutige Frames:", oof["frame"].nunique())

# Verteilungen der kategorialen Spalten
for col in oof.select_dtypes(
    include=["object", "string", "category"]
).columns:
    print(f"\n{col}:")
    display(oof[col].value_counts(dropna=False).head(10))


In [ ]:

# ============================================================
# SCHRITT 3: OOF-VERGLEICH DER SPIELSTATUSMODELLE
# ============================================================

from sklearn.metrics import precision_recall_fscore_support

# Sicherstellen, dass alle erforderlichen Werte vorhanden sind
cols = ["frame", "GT", "Heuristik 06", "AdaBoost_median91"]

comparison_oof = oof[cols].copy()

assert comparison_oof[["GT", "Heuristik 06", "AdaBoost_median91"]].isin(
    ["D", "L"]
).all().all()

# 1. Vier Kombinationen von Heuristik und AdaBoost
agreement = (
    comparison_oof
    .groupby(["Heuristik 06", "AdaBoost_median91"])
    .agg(
        Frames=("GT", "size"),
        GT_D=("GT", lambda x: (x == "D").sum()),
        GT_L=("GT", lambda x: (x == "L").sum())
    )
    .reset_index()
)

agreement["GT_D_Anteil"] = (
    agreement["GT_D"] / agreement["Frames"]
)

print("Tatsächlicher Spielstatus nach Modellkombination:")
display(agreement.round(3))

# 2. D-Erkennungsqualität beider Modelle
performance = []

for model in ["Heuristik 06", "AdaBoost_median91"]:

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            comparison_oof["GT"],
            comparison_oof[model],
            labels=["D"],
            zero_division=0
        )
    )

    performance.append({
        "Modell": model,
        "D_Precision": precision[0],
        "D_Recall": recall[0],
        "D_F1": f1[0],
        "Accuracy": (
            comparison_oof["GT"] == comparison_oof[model]
        ).mean()
    })

print("\nOOF-Vergleich:")
display(pd.DataFrame(performance).round(4))

# 3. Nur widersprüchliche Vorhersagen untersuchen
disagreement = comparison_oof[
    comparison_oof["Heuristik 06"]
    != comparison_oof["AdaBoost_median91"]
]

print("\nWidersprüchliche öffentliche Frames:")
print(len(disagreement))

print("\nWelches Modell liegt bei Widersprüchen richtig?")

heuristic_correct = (
    disagreement["Heuristik 06"] == disagreement["GT"]
).sum()

ml_correct = (
    disagreement["AdaBoost_median91"] == disagreement["GT"]
).sum()

print("Heuristik 06 korrekt:", heuristic_correct)
print("AdaBoost 08 korrekt:", ml_correct)


In [ ]:

# Hierarchisches Modell – Analyse der D-Phasen

gt = oof["GT"].to_numpy()
heur = oof["Heuristik 06"].to_numpy()
ada = oof["AdaBoost_median91"].to_numpy()

status_variants = {
    "Ground Truth": gt,
    "Heuristik 06": heur,
    "AdaBoost 08": ada,
    "OR (mindestens ein D)": np.where(
        (heur == "D") | (ada == "D"), "D", "L"
    ),
    "AND (beide D)": np.where(
        (heur == "D") & (ada == "D"), "D", "L"
    )
}

def extract_d_spells(labels, frames):
    labels = np.asarray(labels)
    frames = np.asarray(frames)

    starts = np.flatnonzero(
        np.r_[True, (labels[1:] != labels[:-1])
                    | (np.diff(frames) != 1)]
    )
    ends = np.r_[starts[1:], len(labels)]

    d_spells = []

    for start, end in zip(starts, ends):
        if labels[start] == "D":
            d_spells.append({
                "start_frame": int(frames[start]),
                "end_frame": int(frames[end - 1]),
                "duration_s": (end - start) / FPS
            })

    return pd.DataFrame(
        d_spells,
        columns=["start_frame", "end_frame", "duration_s"]
    )

frames = oof["frame"].to_numpy()

results = []
d_spells_by_model = {}

for name, labels in status_variants.items():
    spells = extract_d_spells(labels, frames)
    d_spells_by_model[name] = spells

    durations = spells["duration_s"]

    results.append({
        "Modell": name,
        "D_Frames": int((labels == "D").sum()),
        "D_Phasen": len(spells),
        "Median_Dauer_s": (
            durations.median() if len(spells) else 0
        ),
        "Max_Dauer_s": (
            durations.max() if len(spells) else 0
        ),
        "Kurze_D_Phasen_unter_2s": int(
            (durations < 2).sum()
        )
    })

print("Zeitliche Struktur der Unterbrechungen:")
display(pd.DataFrame(results).round(2))

print("\nTatsächliche Unterbrechungen:")
display(d_spells_by_model["Ground Truth"])

print("\nOR-Kombination – erkannte Unterbrechungen:")
display(d_spells_by_model["OR (mindestens ein D)"])


In [ ]:

from tempfile import TemporaryDirectory
from zipfile import ZipFile
import subprocess
import sys

# ============================================================
# 1. Spielstatus-Vorhersagen vorbereiten
# ============================================================

# Bestehendes Modell aus Notebook 06
base_labels = status_06["label"].to_numpy().copy()

# AdaBoost-Vorhersagen für das komplette Spiel
ada_labels = ml_08["AdaBoost_median91"].to_numpy().copy()

# Im öffentlichen Bereich OOF-Vorhersagen einsetzen,
# damit dort nicht mit In-Sample-Predictions evaluiert wird
oof_frames = oof["frame"].to_numpy(dtype=int)

assert len(np.unique(oof_frames)) == len(oof_frames)
assert np.all((oof_frames >= 0) & (oof_frames < N_FRAMES))

ada_labels[oof_frames] = (
    oof["AdaBoost_median91"].to_numpy()
)

# ============================================================
# 2. Erste hierarchische Variante: Status OR
# ============================================================

hierarchical_or = base_labels.copy()

# Nur zusätzliche Unterbrechungen aus AdaBoost übernehmen
hierarchical_or[ada_labels == "D"] = "D"

models = {
    "hier_base06": base_labels,
    "hier_or_v0": hierarchical_or
}

print("Zusätzliche D-Frames im gesamten Video:")
print(
    np.sum(
        (base_labels != "D")
        & (hierarchical_or == "D")
    )
)

print("\nZusätzliche D-Frames im öffentlichen Bereich:")
print(
    np.sum(
        (base_labels[oof_frames] != "D")
        & (hierarchical_or[oof_frames] == "D")
    )
)

# ============================================================
# 3. Vollständige Submission-Dateien erstellen
# ============================================================

OUT = PROJECT_DIR / "output/hierarchical_v0"
SUBMISSIONS = OUT / "submissions"
EVALUATION = OUT / "evaluation"

SUBMISSIONS.mkdir(parents=True, exist_ok=True)
EVALUATION.mkdir(parents=True, exist_ok=True)

for name, labels in models.items():
    assert len(labels) == N_FRAMES
    assert np.isin(labels, ["H", "A", "D", "X"]).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUBMISSIONS / f"{name}.csv",
        index=False
    )

# ============================================================
# 4. Offizielle Evaluation
# ============================================================

ARCHIVE = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as temp:
    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(temp)

    root = Path(temp) / root_name

    validator = root / "evaluation/validate_submission.py"
    evaluator = root / "evaluation/run_evaluation.py"

    for name in models:
        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUBMISSIONS / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    result = subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUBMISSIONS),
            "--out", str(EVALUATION),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

# ============================================================
# 5. Vergleich der tatsächlichen Scores
# ============================================================

scores = pd.read_csv(
    EVALUATION / "scoreboard.csv"
).set_index("submission")

display(
    scores.loc[
        ["hier_base06", "hier_or_v0"],
        [
            "macro_f1",
            "spell_f1_mean",
            "f1_H",
            "f1_A",
            "f1_D",
            "poss",
            "dur"
        ]
    ].round(4)
)


In [ ]:

# ============================================================
# SCHRITT 6: ZUSÄTZLICHE D-INTERVALLE ANALYSIEREN
# ============================================================

base = status_06["label"].to_numpy()

# Nur Frames, die durch AdaBoost neu auf D gesetzt werden
additional_D = (
    (base != "D")
    & (ada_labels == "D")
)

# Zusammenhängende Intervalle bestimmen
starts = np.flatnonzero(
    additional_D
    & ~np.r_[False, additional_D[:-1]]
)

ends = np.flatnonzero(
    additional_D
    & ~np.r_[additional_D[1:], False]
)

# Öffentliche Referenz für Diagnose ergänzen
public_gt = np.full(N_FRAMES, "?", dtype="<U1")
public_gt[oof["frame"].to_numpy(dtype=int)] = (
    oof["GT"].to_numpy()
)

records = []

for start, end in zip(starts, ends):

    interval_gt = public_gt[start:end + 1]
    public_mask = interval_gt != "?"

    records.append({
        "start_frame": int(start),
        "end_frame": int(end),
        "duration_s": (end - start + 1) / FPS,
        "previous_label": (
            base[start - 1] if start > 0 else None
        ),
        "next_label": (
            base[end + 1] if end + 1 < N_FRAMES else None
        ),
        "public_frames": int(public_mask.sum()),
        "public_GT_D": int((interval_gt == "D").sum()),
        "public_GT_L": int((interval_gt == "L").sum())
    })

df_additional_D = pd.DataFrame(records)

print("Zusätzliche D-Intervalle:", len(df_additional_D))
print("Zusätzliche D-Frames:", additional_D.sum())

print("\nDauer der zusätzlichen Intervalle:")
display(
    df_additional_D["duration_s"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).round(2)
)

print("\nZusätzliche Intervalle unter 2 Sekunden:")
print(
    (df_additional_D["duration_s"] < 2).sum()
)

print("\nZusätzliche Intervalle im öffentlichen Fenster:")
display(
    df_additional_D.loc[
        df_additional_D["public_frames"] > 0
    ].round(2)
)

print("\nDie 10 längsten zusätzlichen Intervalle:")
display(
    df_additional_D.nlargest(
        10, "duration_s"
    ).round(2)
)


In [ ]:

# ============================================================
# SCHRITT 7: WIRKUNG ZUSÄTZLICHER D-INTERVALLE
# ============================================================

def classify_d_interval(row):
    before = row["previous_label"]
    after = row["next_label"]

    if before == "D" and after == "D":
        return "Verbindung bestehender D-Phasen"

    if before == "D" or after == "D":
        return "Erweiterung bestehender D-Phase"

    if before in ("H", "A") and after in ("H", "A"):
        if before == after:
            return "Neue D-Phase innerhalb gleicher Mannschaft"
        return "Neue D-Phase an H/A-Wechsel"

    return "Sonstiges / Videorand"


df_additional_D["interval_type"] = (
    df_additional_D.apply(
        classify_d_interval,
        axis=1
    )
)

summary = (
    df_additional_D
    .groupby("interval_type")
    .agg(
        n_intervals=("duration_s", "size"),
        total_duration_s=("duration_s", "sum"),
        median_duration_s=("duration_s", "median"),
        max_duration_s=("duration_s", "max")
    )
    .sort_values(
        "total_duration_s",
        ascending=False
    )
)

summary["share_duration"] = (
    summary["total_duration_s"]
    / df_additional_D["duration_s"].sum()
)

print("Typen zusätzlicher D-Intervalle:")
display(summary.round(3))

# Neue Unterbrechungen ohne direkten Anschluss
# an eine bestehende D-Phase
new_stoppages = df_additional_D[
    df_additional_D["interval_type"].str.startswith(
        "Neue D-Phase"
    )
].copy()

print("\nVollständig neue D-Intervalle:")
print(len(new_stoppages))

print("\nDavon mindestens 5 Sekunden:")
print((new_stoppages["duration_s"] >= 5).sum())

print("\nDie 15 längsten neuen D-Intervalle:")
display(
    new_stoppages.nlargest(
        15, "duration_s"
    )[
        [
            "start_frame",
            "end_frame",
            "duration_s",
            "previous_label",
            "next_label",
            "interval_type"
        ]
    ].round(2)
)


In [ ]:

# ============================================================
# SCHRITT 8: HIERARCHISCHE ABLATIONSVARIANTEN
# ============================================================

N = len(base_labels)

extension_mask = np.zeros(N, dtype=bool)
bridge_mask = np.zeros(N, dtype=bool)
new_mask = np.zeros(N, dtype=bool)

# Intervalle anhand ihrer zuvor bestimmten Kategorie zuordnen
for _, row in df_additional_D.iterrows():

    start = int(row["start_frame"])
    end = int(row["end_frame"]) + 1
    category = row["interval_type"]

    if category == "Erweiterung bestehender D-Phase":
        extension_mask[start:end] = True

    elif category == "Verbindung bestehender D-Phasen":
        bridge_mask[start:end] = True

    elif category.startswith("Neue D-Phase"):
        new_mask[start:end] = True

# Prüfen, ob alle zusätzlichen D-Frames erfasst sind
assert np.array_equal(
    extension_mask | bridge_mask | new_mask,
    additional_D
)

def apply_d_mask(mask):
    labels = base_labels.copy()
    labels[mask] = "D"
    return labels

ablation_models = {
    "hier_base06": base_labels.copy(),
    "hier_extensions": apply_d_mask(extension_mask),
    "hier_extensions_bridges": apply_d_mask(
        extension_mask | bridge_mask
    ),
    "hier_or_v0": apply_d_mask(
        extension_mask | bridge_mask | new_mask
    )
}

# Veränderung im gesamten Video und im öffentlichen Fenster
public_frames = oof["frame"].to_numpy(dtype=int)

rows = []

for name, labels in ablation_models.items():

    changed = labels != base_labels

    rows.append({
        "model": name,
        "total_new_D_frames": int(changed.sum()),
        "total_new_D_seconds": changed.sum() / FPS,
        "public_new_D_frames": int(
            changed[public_frames].sum()
        ),
        "full_match_D_frames": int(
            (labels == "D").sum()
        )
    })

display(pd.DataFrame(rows).round(2))

# Prüfen, ob die Varianten im öffentlichen
# Referenzbereich tatsächlich identisch sind
extensions_public = (
    ablation_models["hier_extensions"][public_frames]
)

for name in [
    "hier_extensions_bridges",
    "hier_or_v0"
]:
    print(
        f"Öffentlich identisch: hier_extensions vs {name}:",
        np.array_equal(
            extensions_public,
            ablation_models[name][public_frames]
        )
    )

# Alle Varianten für spätere Evaluation speichern
output_dir = (
    PROJECT_DIR
    / "output/hierarchical_ablation/submissions"
)
output_dir.mkdir(parents=True, exist_ok=True)

for name, labels in ablation_models.items():
    pd.DataFrame({
        "frame": np.arange(N),
        "label": labels
    }).to_csv(
        output_dir / f"{name}.csv",
        index=False
    )

print("\nAblationsvarianten gespeichert.")

In [ ]:

import shutil
import subprocess
from IPython.display import Video, display


video_path = "/Volumes/T7/More Than A Hack 2027/Initial Challenge/tactical/BAR-ATH_tactical.mp4"

ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    raise RuntimeError("ffmpeg wurde nicht gefunden.")

CLIP_DIR = PROJECT_DIR / "output/hierarchical_diagnostics"
CLIP_DIR.mkdir(parents=True, exist_ok=True)

cases = [
    ("case_01_H_to_A", 53142, 53914),
    ("case_02_A_to_H", 67524, 68156)
]

for name, first_frame, last_frame in cases:

    start_s = max(0, first_frame / FPS - 3)
    end_s = (last_frame + 1) / FPS + 3
    duration_s = end_s - start_s

    output_path = CLIP_DIR / f"{name}.mp4"

    subprocess.run(
        [
            ffmpeg,
            "-hide_banner",
            "-loglevel", "error",
            "-y",
            "-ss", f"{start_s:.3f}",
            "-i", str(video_path),
            "-t", f"{duration_s:.3f}",
            "-vf", "scale=960:-2",
            "-c:v", "libx264",
            "-preset", "veryfast",
            "-crf", "27",
            "-an",
            "-movflags", "+faststart",
            str(output_path)
        ],
        check=True
    )

    print(f"\n{name}")
    print(f"Frames: {first_frame}–{last_frame}")
    print(f"Video: {start_s:.2f}s–{end_s:.2f}s")
    print(f"Gespeichert: {output_path}")

    display(
        Video(
            filename=str(output_path),
            embed=True,
            width=850
        )
    )


In [ ]:

# Die drei längsten Verbindungen bestehender D-Phasen
bridge_cases = (
    df_additional_D[
        df_additional_D["interval_type"]
        == "Verbindung bestehender D-Phasen"
    ]
    .nlargest(3, "duration_s")
)

print("Drei längste Verbindungen:")
display(
    bridge_cases[
        ["start_frame", "end_frame", "duration_s"]
    ].round(2)
)

# Jeweils 3 Sekunden davor und danach zeigen
for i, (_, row) in enumerate(bridge_cases.iterrows(), 1):

    first_frame = int(row["start_frame"])
    last_frame = int(row["end_frame"])

    clip_start = max(0, first_frame / FPS - 3)
    clip_end = min(
        N_FRAMES / FPS,
        (last_frame + 1) / FPS + 3
    )

    clip_path = CLIP_DIR / f"bridge_case_{i:02d}.mp4"

    subprocess.run(
        [
            ffmpeg,
            "-hide_banner",
            "-loglevel", "error",
            "-y",
            "-ss", f"{clip_start:.3f}",
            "-i", str(video_path),
            "-t", f"{clip_end - clip_start:.3f}",
            "-vf", "scale=960:-2",
            "-c:v", "libx264",
            "-preset", "veryfast",
            "-crf", "27",
            "-an",
            "-movflags", "+faststart",
            str(clip_path)
        ],
        check=True
    )

    print(
        f"\nSzene {i}: Frames "
        f"{first_frame}–{last_frame}"
    )

    print(
        f"Neue D-Zeit: {row['duration_s']:.2f} s"
    )

    display(
        Video(
            filename=str(clip_path),
            embed=True,
            width=850
        )
    )

In [ ]:

# ============================================================
# SCHRITT 11: SPIELSTATUS-EVIDENZ UM DIE D-VERBINDUNGEN
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(
    3, 1,
    figsize=(14, 10),
    constrained_layout=True
)

for i, (_, case) in enumerate(bridge_cases.iterrows()):
    ax = axes[i]

    first = int(case["start_frame"])
    last = int(case["end_frame"])

    # Drei Sekunden Kontext davor und danach
    left = max(0, first - 3 * FPS)
    right = min(N_FRAMES, last + 3 * FPS + 1)

    frames = np.arange(left, right)
    time_s = (frames - left) / FPS

    score = ml_08.iloc[left:right][
        "AdaBoost_D_score"
    ].to_numpy()

    heuristic_D = (
        status_06.iloc[left:right]["label"]
        .to_numpy() == "D"
    ).astype(float)

    ada_D = (
        ada_labels[left:right] == "D"
    ).astype(float)

    ax.plot(
        time_s,
        score,
        label="AdaBoost D-Score",
        linewidth=1.6
    )

    ax.step(
        time_s,
        heuristic_D,
        where="post",
        label="Heuristik 06: D",
        alpha=0.8
    )

    ax.step(
        time_s,
        ada_D,
        where="post",
        label="AdaBoost: D",
        alpha=0.8
    )

    # Beginn und Ende des überbrückten Intervalls
    ax.axvline(
        (first - left) / FPS,
        color="gray",
        linestyle="--",
        alpha=0.7
    )

    ax.axvline(
        (last + 1 - left) / FPS,
        color="gray",
        linestyle="--",
        alpha=0.7
    )

    # Beobachteter Pfiff in Szene 1 (ungefähr)
    if i == 0:
        ax.axvline(
            18,
            color="red",
            linestyle=":",
            linewidth=2,
            label="Beobachteter Pfiff (ca.)"
        )

    ax.set_title(
        f"Szene {i+1}: Frames {first}–{last}"
    )
    ax.set_xlabel("Zeit innerhalb des Clips (s)")
    ax.set_ylabel("Score / D-Status")
    ax.set_ylim(-0.08, 1.08)
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")

plt.show()


In [ ]:

# ============================================================
# SCHRITT 13: BEWEGUNGSEVIDENZ RUND UM DEN PFIFF
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW_PATH = (
    PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
)

# AdaBoost-D-Beginn und beobachteter Pfiff
ada_start = 69845
whistle_frame = (46 * 60 + 48) * FPS

# Untersuchungsfenster mit Kontext
start = ada_start - 5 * FPS
end = whistle_frame + 8 * FPS

frames = np.arange(start, end + 1)
n = len(frames)

# 46 Spielerkanäle + 1 Ballkanal
xy = np.full((n, 47, 2), np.nan, dtype=np.float32)

# Nur die benötigten Trackingframes einlesen
with open(RAW_PATH, "r") as file:
    for line in file:
        parts = line.strip().split(":", 2)

        if len(parts) != 3:
            continue

        frame = int(parts[0]) - 1

        if frame < start:
            continue

        if frame > end:
            break

        values = np.fromstring(
            parts[1].replace(";", ",") + "," + parts[2],
            sep=",",
            dtype=np.float32
        )

        if len(values) == 47 * 3:
            xy[frame - start] = (
                values.reshape(47, 3)[:, :2]
            )

# Spielfeldabmessungen
pitch_scale = np.array([105.3, 68.0])

# ------------------------------------------------------------
# 1. Spielergeschwindigkeit
# ------------------------------------------------------------

players = xy[:, :46, :]

# Geschwindigkeit über 5 Frames statt nur 1 Frame
# zur Reduktion kurzfristigen Trackingrauschens
lag = 5

player_speed = np.full(
    (n, 46), np.nan, dtype=float
)

delta_players = (
    players[lag:] - players[:-lag]
) * pitch_scale

speed_values = (
    np.linalg.norm(delta_players, axis=2)
    * FPS / lag
)

# Extremwerte als mögliche Trackingfehler ausschließen
speed_values[speed_values > 12] = np.nan
player_speed[lag:] = speed_values

# Median über alle gültigen Spielerkanäle
median_player_speed = (
    pd.DataFrame(player_speed)
    .median(axis=1)
    .rolling(25, center=True, min_periods=5)
    .median()
    .to_numpy()
)

# Anteil bewegter Spieler unter gültigen Geschwindigkeiten
valid_players = np.isfinite(player_speed).sum(axis=1)

moving_players = (
    (player_speed >= 2.0)
    & np.isfinite(player_speed)
).sum(axis=1)

moving_fraction = np.divide(
    moving_players,
    valid_players,
    out=np.full(n, np.nan),
    where=valid_players > 0
)

moving_fraction = (
    pd.Series(moving_fraction)
    .rolling(25, center=True, min_periods=5)
    .mean()
    .to_numpy()
)

# ------------------------------------------------------------
# 2. Ballgeschwindigkeit und Tracking-Verfügbarkeit
# ------------------------------------------------------------

ball = xy[:, 46, :]

ball_valid = np.isfinite(ball).all(axis=1)

ball_speed = np.full(n, np.nan)

ball_displacement = (
    np.linalg.norm(
        (ball[1:] - ball[:-1]) * pitch_scale,
        axis=1
    ) * FPS
)

valid_ball_pairs = (
    ball_valid[1:] & ball_valid[:-1]
)

ball_speed[1:] = np.where(
    valid_ball_pairs
    & (ball_displacement <= 40),
    ball_displacement,
    np.nan
)

ball_speed_smooth = (
    pd.Series(ball_speed)
    .rolling(15, center=True, min_periods=5)
    .median()
    .to_numpy()
)

ball_coverage = (
    pd.Series(ball_valid.astype(float))
    .rolling(25, center=True, min_periods=1)
    .mean()
    .to_numpy()
)

# ------------------------------------------------------------
# 3. Ergebnisse zusammenführen
# ------------------------------------------------------------

motion = pd.DataFrame({
    "frame": frames,
    "video_time_s": frames / FPS,
    "player_speed_mps": median_player_speed,
    "moving_fraction": moving_fraction,
    "ball_speed_mps": ball_speed_smooth,
    "ball_coverage": ball_coverage
})

# Vorher / nachher vergleichen
before = motion[
    (motion["frame"] >= ada_start)
    & (motion["frame"] < whistle_frame)
]

after = motion[
    (motion["frame"] >= whistle_frame)
]

summary = pd.DataFrame({
    "Vor Pfiff (AdaBoost D)": before[
        ["player_speed_mps", "moving_fraction",
         "ball_speed_mps", "ball_coverage"]
    ].median(),
    "Nach Pfiff": after[
        ["player_speed_mps", "moving_fraction",
         "ball_speed_mps", "ball_coverage"]
    ].median()
}).T

print("Bewegungsmerkmale vor und nach dem Pfiff:")
display(summary.round(3))

# ------------------------------------------------------------
# 4. Zeitverlauf visualisieren
# ------------------------------------------------------------

t = (frames - start) / FPS

fig, axes = plt.subplots(
    4, 1,
    figsize=(14, 11),
    sharex=True,
    constrained_layout=True
)

signals = [
    ("player_speed_mps", "Median Spielergeschwindigkeit (m/s)"),
    ("moving_fraction", "Anteil Spieler >= 2 m/s"),
    ("ball_speed_mps", "Ballgeschwindigkeit (m/s)"),
    ("ball_coverage", "Balltracking-Abdeckung (1 s)")
]

for ax, (column, title) in zip(axes, signals):

    ax.plot(
        t,
        motion[column],
        linewidth=1.6
    )

    # Beginn der zusätzlichen AdaBoost-D-Vorhersage
    ax.axvline(
        (ada_start - start) / FPS,
        color="orange",
        linestyle="--",
        label="AdaBoost: D-Beginn"
    )

    # Beobachteter Pfiff
    ax.axvline(
        (whistle_frame - start) / FPS,
        color="red",
        linestyle="--",
        label="Pfiff (ca. 46:48)"
    )

    # Kritischer Bereich: AdaBoost meldet D,
    # obwohl offenbar noch gespielt wird
    ax.axvspan(
        (ada_start - start) / FPS,
        (whistle_frame - start) / FPS,
        color="orange",
        alpha=0.10
    )

    ax.set_ylabel(title)
    ax.grid(alpha=0.2)

axes[0].legend(loc="upper right")
axes[-1].set_xlabel(
    "Zeit seit Beginn des Untersuchungsfensters (s)"
)

plt.show()


In [ ]:

# ============================================================
# SCHRITT 14: BEWEGUNGSFEATURES FÜR DAS GESAMTE VIDEO
# ============================================================

import warnings
import numpy as np
import pandas as pd

# 46 Spielerkanäle + 1 Ball
xy_all = np.full(
    (N_FRAMES, 47, 2),
    np.nan,
    dtype=np.float32
)

# Tracking vollständig einlesen
with open(RAW_PATH, "r") as file:
    for line in file:
        parts = line.strip().split(":", 2)

        if len(parts) != 3:
            continue

        frame = int(parts[0]) - 1

        if not (0 <= frame < N_FRAMES):
            continue

        values = np.fromstring(
            parts[1].replace(";", ",") + "," + parts[2],
            sep=",",
            dtype=np.float32
        )

        if values.size == 47 * 3:
            xy_all[frame] = values.reshape(47, 3)[:, :2]

# ------------------------------------------------------------
# 1. Spielergeschwindigkeiten (5-Frame-Differenz)
# ------------------------------------------------------------

lag = 5

player_speed_all = np.full(
    (N_FRAMES, 46),
    np.nan,
    dtype=np.float32
)

delta = (
    xy_all[lag:, :46] - xy_all[:-lag, :46]
)

speed = np.hypot(
    delta[:, :, 0] * 105.3,
    delta[:, :, 1] * 68.0
) * FPS / lag

speed[speed > 12] = np.nan
player_speed_all[lag:] = speed

valid_count = np.isfinite(
    player_speed_all
).sum(axis=1)

moving_count = (
    player_speed_all >= 2
).sum(axis=1)

moving_fraction_all = np.divide(
    moving_count,
    valid_count,
    out=np.full(N_FRAMES, np.nan),
    where=valid_count > 0
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    median_speed_all = np.nanmedian(
        player_speed_all,
        axis=1
    )

# ------------------------------------------------------------
# 2. Ballbewegung
# ------------------------------------------------------------

ball = xy_all[:, 46]

ball_valid_all = np.isfinite(ball).all(axis=1)

ball_speed_all = np.full(
    N_FRAMES, np.nan, dtype=np.float32
)

ball_displacement = np.hypot(
    (ball[1:, 0] - ball[:-1, 0]) * 105.3,
    (ball[1:, 1] - ball[:-1, 1]) * 68.0
) * FPS

ball_speed_all[1:] = np.where(
    ball_valid_all[1:] & ball_valid_all[:-1]
    & (ball_displacement <= 40),
    ball_displacement,
    np.nan
)

# ------------------------------------------------------------
# 3. Zeitlich aggregierte Features
# ------------------------------------------------------------

movement_features = pd.DataFrame({
    "frame": np.arange(N_FRAMES),
    "player_speed_mps": median_speed_all,
    "moving_fraction": moving_fraction_all,
    "ball_speed_mps": ball_speed_all,
    "ball_valid": ball_valid_all
})

for col in [
    "player_speed_mps",
    "moving_fraction",
    "ball_speed_mps"
]:
    movement_features[col] = (
        movement_features[col]
        .rolling(
            25, center=True, min_periods=5
        )
        .median()
    )

movement_features["ball_coverage"] = (
    movement_features["ball_valid"]
    .astype(float)
    .rolling(25, center=True, min_periods=1)
    .mean()
)

# ------------------------------------------------------------
# 4. Bereits visuell untersuchte Szenen vergleichen
# ------------------------------------------------------------

cases = [
    ("Szene 1: Live vor Pfiff", 69845, 70199),
    ("Szene 1: Nach Pfiff", 70200, 70245),
    ("Foul Bilbao", 53142, 53914),
    ("Foul Barcelona", 67524, 68156),
    ("Halbzeitpause", 112926, 113282),
    ("Abseits", 124541, 124862)
]

results = []

for name, start, end in cases:

    section = movement_features.iloc[start:end + 1]

    results.append({
        "Szene": name,
        "Dauer_s": (end - start + 1) / FPS,
        "Spieler_mps": section[
            "player_speed_mps"
        ].median(),
        "Anteil_schnell": section[
            "moving_fraction"
        ].median(),
        "Ball_mps": section[
            "ball_speed_mps"
        ].median(),
        "Ball_Abdeckung": section[
            "ball_coverage"
        ].mean()
    })

comparison_motion = pd.DataFrame(results)

print("Bewegungsvergleich der untersuchten Szenen:")
display(comparison_motion.round(3))

# Für weitere Analysen speichern
FEATURE_OUT = (
    PROJECT_DIR
    / "output/hierarchical_v0/movement_features.csv"
)

FEATURE_OUT.parent.mkdir(
    parents=True, exist_ok=True
)

movement_features.to_csv(
    FEATURE_OUT, index=False
)

print("\nBewegungsfeatures gespeichert:", FEATURE_OUT)


In [ ]:

# ============================================================
# SCHRITT 15: FALSCHE VS. KORREKTE ADABOOST-D-ERKENNUNG
# ============================================================

# Öffentliche OOF-Vorhersagen mit Bewegungsfeatures verbinden
public_motion = oof[
    ["frame", "GT", "Heuristik 06", "AdaBoost_median91"]
].merge(
    movement_features[
        [
            "frame",
            "player_speed_mps",
            "moving_fraction",
            "ball_speed_mps",
            "ball_coverage"
        ]
    ],
    on="frame",
    how="left",
    validate="one_to_one"
)

# Nur Fälle, in denen AdaBoost zusätzlich D erkennt
# und die bisherige Heuristik Live vorhersagt
ada_additional = public_motion.loc[
    (public_motion["Heuristik 06"] == "L")
    & (public_motion["AdaBoost_median91"] == "D")
].copy()

assert len(ada_additional) == 350

# Tatsächlichen Spielstatus als Vergleich verwenden
summary = (
    ada_additional
    .groupby("GT")
    .agg(
        n_frames=("frame", "size"),

        median_player_speed=(
            "player_speed_mps", "median"
        ),

        moving_p25=(
            "moving_fraction",
            lambda x: x.quantile(0.25)
        ),

        moving_median=(
            "moving_fraction", "median"
        ),

        moving_p75=(
            "moving_fraction",
            lambda x: x.quantile(0.75)
        ),

        median_ball_speed=(
            "ball_speed_mps", "median"
        ),

        ball_speed_available=(
            "ball_speed_mps",
            lambda x: x.notna().mean()
        ),

        mean_ball_coverage=(
            "ball_coverage", "mean"
        )
    )
)

summary.index = summary.index.map({
    "D": "Korrekte zusätzliche D-Erkennung",
    "L": "Falsche zusätzliche D-Erkennung"
})

print("AdaBoost D / Heuristik Live:")
display(summary.round(3))


In [ ]:

# ============================================================
# SCHRITT 16: AKTIVITÄTSÄNDERUNG AN STATUSÜBERGÄNGEN
# ============================================================

import numpy as np
import pandas as pd

# Tatsächliche D-Beginne aus der öffentlichen GT
public_sorted = oof.sort_values("frame").reset_index(drop=True)

gt_starts = public_sorted.loc[
    public_sorted["GT"].eq("D")
    & public_sorted["GT"].shift().ne("D"),
    "frame"
].astype(int).tolist()

# Zusätzliche, bereits beobachtete Übergänge
# Nur für die Diagnose, nicht für die finale Vorhersage!
events = [
    (f"GT D-Beginn {f}", f, "Ground Truth")
    for f in gt_starts
]

events += [
    ("AdaBoost zu früh, öffentlich", 17260, "Frühe D-Vorhersage"),
    ("AdaBoost zu früh, Szene 46:48", 69845, "Frühe D-Vorhersage"),
    ("Beobachteter Pfiff 46:48", 70200, "Video-Beobachtung")
]

WINDOW = 3 * FPS

signals = {
    "moving_fraction": movement_features[
        "moving_fraction"
    ].to_numpy(dtype=float),
    "player_speed": movement_features[
        "player_speed_mps"
    ].to_numpy(dtype=float),
    "ball_coverage": movement_features[
        "ball_coverage"
    ].to_numpy(dtype=float)
}

def safe_median(values):
    valid = values[np.isfinite(values)]
    return float(np.median(valid)) if len(valid) else np.nan

records = []

for name, frame, event_type in events:

    before_start = max(0, frame - WINDOW)
    after_end = min(N_FRAMES, frame + WINDOW)

    record = {
        "Ereignis": name,
        "Typ": event_type,
        "Frame": frame
    }

    for signal_name, values in signals.items():

        before = safe_median(
            values[before_start:frame]
        )

        after = safe_median(
            values[frame:after_end]
        )

        record[f"{signal_name}_vor"] = before
        record[f"{signal_name}_nach"] = after
        record[f"{signal_name}_delta"] = after - before

    records.append(record)

df_transitions = pd.DataFrame(records)

print("Veränderung der Spieleraktivität an Übergängen:")

display(
    df_transitions[
        [
            "Ereignis",
            "moving_fraction_vor",
            "moving_fraction_nach",
            "moving_fraction_delta",
            "player_speed_delta",
            "ball_coverage_delta"
        ]
    ].round(3)
)


In [ ]:

# ============================================================
# SCHRITT 17: AUSSAGEKRAFT DER LIVE-EVIDENZ
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

WINDOW = 3 * FPS

# 1. Veränderung der Spieleraktivität berechnen
moving = movement_features["moving_fraction"]

# Median der vorherigen 3 Sekunden
moving_before = (
    moving.rolling(
        WINDOW, min_periods=25
    ).median().shift(1)
)

# Median der nachfolgenden 3 Sekunden
moving_after = (
    moving.iloc[::-1]
    .rolling(
        WINDOW, min_periods=25
    )
    .median()
    .shift(1)
    .iloc[::-1]
)

activity_delta = moving_after - moving_before

# 2. Features für den öffentlichen Bereich
public = oof[
    ["frame", "GT", "Heuristik 06", "AdaBoost_median91"]
].copy()

feature_data = movement_features[
    [
        "frame",
        "player_speed_mps",
        "moving_fraction",
        "ball_speed_mps",
        "ball_coverage"
    ]
].copy()

feature_data["activity_delta_3s"] = (
    activity_delta.to_numpy()
)

public = public.merge(
    feature_data,
    on="frame",
    validate="one_to_one"
)

# 3. Höherer Score bedeutet stärkere D-Evidenz.
# Vorzeichen sind nur Untersuchungshypothesen.
signal_columns = {
    "Geringe Spielergeschwindigkeit": -public["player_speed_mps"],
    "Wenige schnelle Spieler": -public["moving_fraction"],
    "Abnehmende Spieleraktivität": -public["activity_delta_3s"],
    "Geringe Ballgeschwindigkeit": -public["ball_speed_mps"],
    "Geringe Ballabdeckung": -public["ball_coverage"]
}

# 4. AUC für alle Frames und widersprüchliche Frames
subsets = {
    "Alle öffentlichen Frames": np.ones(
        len(public), dtype=bool
    ),
    "Nur Modellwidersprüche": (
        public["Heuristik 06"]
        != public["AdaBoost_median91"]
    ).to_numpy()
}

results = []

for subset_name, subset_mask in subsets.items():

    for signal_name, values in signal_columns.items():

        values = np.asarray(values, dtype=float)

        valid = (
            subset_mask & np.isfinite(values)
        )

        y = (
            public.loc[valid, "GT"] == "D"
        ).astype(int)

        if y.nunique() != 2:
            auc = np.nan
        else:
            auc = roc_auc_score(
                y, values[valid]
            )

        results.append({
            "Bereich": subset_name,
            "Signal": signal_name,
            "Frames": int(valid.sum()),
            "ROC_AUC": auc
        })

df_signal_auc = pd.DataFrame(results)

print("Aussagekraft der Bewegungsmerkmale:")
display(
    df_signal_auc.round(3)
)


In [ ]:

# ============================================================
# SCHRITT 18: HIERARCHISCHER STATUSDECODER
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

# ------------------------------------------------------------
# 1. Vorhersagen beider Modelle
# ------------------------------------------------------------

heur_d = base_labels == "D"
ada_d = ada_labels == "D"
ada_valid = np.isin(ada_labels, ["D", "L"])

# Vorläufige Evidenzwerte, ausdrücklich KEINE
# kalibrierten Wahrscheinlichkeiten.
#
# Werte sind feste Startannahmen und werden hier
# nicht auf öffentlichen Labels optimiert.

p_d = np.full(N_FRAMES, 0.04, dtype=float)

# Falls ML nicht verfügbar ist: Heuristik entscheidet
p_d[~ada_valid & heur_d] = 0.90
p_d[~ada_valid & ~heur_d] = 0.10

# Beide Modelle verfügbar
p_d[ada_valid & heur_d & ada_d] = 0.96
p_d[ada_valid & heur_d & ~ada_d] = 0.70
p_d[ada_valid & ~heur_d & ada_d] = 0.62
p_d[ada_valid & ~heur_d & ~ada_d] = 0.04

# Log-Odds als Evidenz für D gegenüber LIVE
base_logit = np.log(p_d / (1 - p_d))

# ------------------------------------------------------------
# 2. Optionale Bewegungsevidenz
# ------------------------------------------------------------

motion_logit = base_logit.copy()

moving = movement_features[
    "moving_fraction"
].to_numpy(dtype=float)

# Nur wenn Heuristik und AdaBoost widersprechen
disagreement = (
    ada_valid
    & (heur_d != ada_d)
    & np.isfinite(moving)
)

# Schwache Zusatzinformation:
# wenig schnelle Spieler -> etwas mehr D-Evidenz
# viele schnelle Spieler -> etwas mehr LIVE-Evidenz
#
# 0.20 und 0.45 sind feste Startparameter,
# keine empirisch optimierten Schwellenwerte.
motion_adjustment = np.zeros(N_FRAMES)

motion_adjustment[disagreement] = (
    0.45 * np.clip(
        (0.20 - moving[disagreement]) / 0.20,
        -1,
        1
    )
)

motion_logit += motion_adjustment

# ------------------------------------------------------------
# 3. Viterbi-Decoder mit Zustandswechselkosten
# ------------------------------------------------------------

def viterbi_status(logit_d, switch_penalty=4.0):
    """
    Zwei Zustände:
      0 = LIVE
      1 = D

    Maximiert die gesamte Evidenz über die Zeit
    und bestraft häufige Zustandswechsel.

    switch_penalty ist ein Startparameter
    (keine auf GT trainierte Größe).
    """

    logit_d = np.asarray(logit_d, dtype=float)
    n = len(logit_d)

    logp_d = -np.logaddexp(0, -logit_d)
    logp_l = -np.logaddexp(0, logit_d)

    back = np.zeros((n, 2), dtype=np.uint8)

    score_l = logp_l[0]
    score_d = logp_d[0]

    for t in range(1, n):

        # Neuer Zustand LIVE
        from_l = score_l
        from_d = score_d - switch_penalty

        if from_l >= from_d:
            new_l = from_l + logp_l[t]
            back[t, 0] = 0
        else:
            new_l = from_d + logp_l[t]
            back[t, 0] = 1

        # Neuer Zustand D
        from_l = score_l - switch_penalty
        from_d = score_d

        if from_d >= from_l:
            new_d = from_d + logp_d[t]
            back[t, 1] = 1
        else:
            new_d = from_l + logp_d[t]
            back[t, 1] = 0

        score_l, score_d = new_l, new_d

    # Optimalen Zustandsverlauf rekonstruieren
    states = np.zeros(n, dtype=np.uint8)
    states[-1] = 1 if score_d > score_l else 0

    for t in range(n - 1, 0, -1):
        states[t - 1] = back[t, states[t]]

    return np.where(states == 1, "D", "L")


status_viterbi = viterbi_status(base_logit)
status_viterbi_motion = viterbi_status(motion_logit)

# ------------------------------------------------------------
# 4. Gegen die öffentlichen OOF-Labels vergleichen
# ------------------------------------------------------------

public_frames = oof["frame"].to_numpy(dtype=int)
y_true = oof["GT"].to_numpy()

or_status = np.where(
    (base_labels == "D") | (ada_labels == "D"),
    "D", "L"
)

status_variants_v1 = {
    "Heuristik 06": np.where(heur_d, "D", "L"),
    "OR v0": or_status,
    "Viterbi Status": status_viterbi,
    "Viterbi Status + Motion": status_viterbi_motion
}

results = []

for name, labels in status_variants_v1.items():

    pred = labels[public_frames]

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true, pred,
            labels=["D"],
            zero_division=0
        )
    )

    # Anzahl D-Phasen im öffentlichen Bereich
    d_phases = extract_d_spells(
        pred, public_frames
    )

    results.append({
        "Modell": name,
        "D_Precision": precision[0],
        "D_Recall": recall[0],
        "D_F1": f1[0],
        "D_Phasen": len(d_phases),
        "D_Frames": int((pred == "D").sum())
    })

print("Erste Ergebnisse des zeitlichen Statusdecoders:")
display(pd.DataFrame(results).round(4))

In [ ]:

# ============================================================
# SCHRITT 19: VITERBI MIT ZEITLICH SKALIERTER EVIDENZ
# ============================================================

from sklearn.metrics import precision_recall_fscore_support

# Evidenz von Frames auf Sekunden umskalieren.
# Die Zustandswechselkosten beziehen sich damit
# auf eine Sekunde aggregierter Evidenz.
#
# Keine Optimierung anhand öffentlicher GT.
TIME_SCALE = 1.0 / FPS
SWITCH_COST = 1.0

status_viterbi_v2 = viterbi_status(
    base_logit * TIME_SCALE,
    switch_penalty=SWITCH_COST
)

status_viterbi_motion_v2 = viterbi_status(
    motion_logit * TIME_SCALE,
    switch_penalty=SWITCH_COST
)

# Vergleich mit bisherigem OR-Ansatz
variants = {
    "OR v0": or_status,
    "Viterbi v1": status_viterbi,
    "Viterbi v2": status_viterbi_v2,
    "Viterbi v2 + Motion": status_viterbi_motion_v2
}

public_idx = oof["frame"].to_numpy(dtype=int)
gt_public = oof["GT"].to_numpy()

results = []

for name, labels in variants.items():

    pred = labels[public_idx]

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            gt_public,
            pred,
            labels=["D"],
            zero_division=0
        )
    )

    spells = extract_d_spells(
        pred,
        public_idx
    )

    results.append({
        "Modell": name,
        "D_Precision": precision[0],
        "D_Recall": recall[0],
        "D_F1": f1[0],
        "D_Phasen": len(spells),
        "D_Frames": int((pred == "D").sum()),
        "Änderungen_öffentlich": int(
            np.count_nonzero(
                pred != or_status[public_idx]
            )
        ),
        "Änderungen_gesamt": int(
            np.count_nonzero(
                labels != or_status
            )
        )
    })

print("Vergleich der Zustandsdecoder:")
display(pd.DataFrame(results).round(4))

# Anzahl der Unterbrechungsphasen im gesamten Video
print("\nD-Phasen im gesamten Video:")

for name, labels in variants.items():

    full_spells = extract_d_spells(
        labels,
        np.arange(N_FRAMES)
    )

    print(
        f"{name}: {len(full_spells)} D-Phasen, "
        f"{(labels == 'D').sum() / FPS:.1f} s D"
    )


In [ ]:

# ============================================================
# SCHRITT 20: OFFIZIELLER SPELLF1 FÜR VITERBI-VARIANTEN
# ============================================================

from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import subprocess
import sys
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Spielstatus und Ballbesitz zusammenführen
# ------------------------------------------------------------

contact_labels = contact_04["label"].to_numpy()

assert len(contact_labels) == N_FRAMES
assert len(base_labels) == N_FRAMES

def combine_status_and_possession(status):
    """
    D aus dem Statusmodell übernehmen.

    Bei LIVE die Ballbesitzvorhersage aus Notebook 06
    verwenden. Falls diese D war, Kontaktmodell 04
    als Rückfalloption verwenden.

    Dies ist eine diagnostische Fusion, noch kein
    vollständiges hierarchisches Zustandsmodell.
    """

    labels = base_labels.copy()

    # Stufe 1: Spielstatus
    labels[status == "D"] = "D"

    # Stufe 2: Ballbesitz bei LIVE
    needs_possession = (
        (status == "L") & (labels == "D")
    )

    contact_available = np.isin(
        contact_labels, ["H", "A"]
    )

    recover = (
        needs_possession & contact_available
    )

    labels[recover] = contact_labels[recover]

    # Fehlende H/A-Evidenz bleibt vorläufig D
    unresolved = (
        needs_possession & ~contact_available
    )

    return labels, int(unresolved.sum())


status_models = {
    "hier_or_v0": or_status,
    "hier_viterbi_v1": status_viterbi,
    "hier_viterbi_v2": status_viterbi_v2,
    "hier_viterbi_v2_motion": status_viterbi_motion_v2
}

submission_models = {
    "hier_base06": base_labels.copy()
}

diagnostics = []

for name, status in status_models.items():

    labels, unresolved = combine_status_and_possession(
        status
    )

    submission_models[name] = labels

    diagnostics.append({
        "model": name,
        "unresolved_live_frames": unresolved,
        "changed_vs_or": int(
            np.count_nonzero(
                labels != hierarchical_or
            )
        ),
        "changed_public_vs_or": int(
            np.count_nonzero(
                labels[oof_frames]
                != hierarchical_or[oof_frames]
            )
        )
    })

# Bestehende OR-Variante exakt reproduzieren
assert np.array_equal(
    submission_models["hier_or_v0"],
    hierarchical_or
), "OR-Referenz wurde nicht exakt reproduziert."

print("Diagnostik der kombinierten Vorhersagen:")
display(pd.DataFrame(diagnostics))


# ------------------------------------------------------------
# 2. Vollständige Submissions speichern
# ------------------------------------------------------------

OUT_DIR = (
    PROJECT_DIR / "output/hierarchical_viterbi"
)

SUB_DIR = OUT_DIR / "submissions"
EVAL_DIR = OUT_DIR / "evaluation"

SUB_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

for name, labels in submission_models.items():

    assert len(labels) == N_FRAMES
    assert np.isin(
        labels, ["H", "A", "D", "X"]
    ).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUB_DIR / f"{name}.csv",
        index=False
    )


# ------------------------------------------------------------
# 3. Offizielle Evaluation
# ------------------------------------------------------------

ARCHIVE = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        archive_root = archive.namelist()[0]
        archive.extractall(tmp)

    official_root = Path(tmp) / archive_root

    validator = (
        official_root
        / "evaluation/validate_submission.py"
    )

    evaluator = (
        official_root
        / "evaluation/run_evaluation.py"
    )

    for name in submission_models:

        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB_DIR / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    result = subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB_DIR),
            "--out", str(EVAL_DIR),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

# ------------------------------------------------------------
# 4. Scorevergleich
# ------------------------------------------------------------

scores = pd.read_csv(
    EVAL_DIR / "scoreboard.csv"
)

columns = [
    "submission",
    "macro_f1",
    "spell_f1_mean",
    "f1_H",
    "f1_A",
    "f1_D",
    "poss",
    "dur"
]

print("\nOffizielle Evaluation:")
display(
    scores[columns]
    .sort_values(
        "spell_f1_mean",
        ascending=False
    )
    .round(4)
)


In [ ]:

# ============================================================
# SCHRITT 21: VERLORENE STATUSKORREKTUREN UNTERSUCHEN
# ============================================================

public_idx = oof["frame"].to_numpy(dtype=int)

# Frames, die Viterbi von D zu LIVE korrigiert
corrected = (
    (or_status[public_idx] == "D")
    & (status_viterbi_v2[public_idx] == "L")
)

corrected_frames = public_idx[corrected]

print("Korrigierte öffentliche Frames:", len(corrected_frames))

diagnostic = pd.DataFrame({
    "frame": corrected_frames,
    "GT": oof.loc[corrected, "GT"].to_numpy(),
    "base06": base_labels[corrected_frames],
    "contact04": contact_labels[corrected_frames],
    "viterbi_status": status_viterbi_v2[corrected_frames],
    "final_label": submission_models[
        "hier_viterbi_v2"
    ][corrected_frames]
})

print("\nTatsächlicher Status dieser Frames:")
display(diagnostic["GT"].value_counts())

# Zusammenhängende Intervalle bilden
diagnostic["group"] = (
    diagnostic["frame"].diff().ne(1)
).cumsum()

def nearest_team_before(frame, labels, horizon=125):
    values = labels[max(0, frame-horizon):frame]
    valid = values[np.isin(values, ["H", "A"])]
    return valid[-1] if len(valid) else "Unbekannt"

def nearest_team_after(frame, labels, horizon=125):
    values = labels[frame+1:min(N_FRAMES, frame+1+horizon)]
    valid = values[np.isin(values, ["H", "A"])]
    return valid[0] if len(valid) else "Unbekannt"

intervals = (
    diagnostic.groupby("group")
    .agg(
        start_frame=("frame", "min"),
        end_frame=("frame", "max"),
        n_frames=("frame", "size"),
        gt_live=("GT", lambda x: (x == "L").sum()),
        final_D=("final_label", lambda x: (x == "D").sum())
    )
    .reset_index(drop=True)
)

intervals["duration_s"] = intervals["n_frames"] / FPS

for label_name, labels in [
    ("Kontakt04", contact_labels),
    ("Basis06", base_labels)
]:
    intervals[f"{label_name}_vorher"] = [
        nearest_team_before(int(f), labels)
        for f in intervals["start_frame"]
    ]

    intervals[f"{label_name}_nachher"] = [
        nearest_team_after(int(f), labels)
        for f in intervals["end_frame"]
    ]

print("\nZusammenhängende Korrekturintervalle:")
display(intervals)


In [ ]:

# ============================================================
# SCHRITT 22: BALLBESITZ INNERHALB VON LIVE-PHASEN
# ============================================================

import numpy as np
import pandas as pd

def reconstruct_possession(status_labels, base_labels, contact_labels):
    """
    Hierarchische Rekonstruktion:

    Stufe 1: LIVE oder D.
    Stufe 2: H/A-Evidenz und zeitliche Besitzfortführung.

    Keine Ground-Truth-Informationen werden verwendet.
    """

    status_labels = np.asarray(status_labels)
    base_labels = np.asarray(base_labels)
    contact_labels = np.asarray(contact_labels)

    n = len(status_labels)

    live = status_labels == "L"

    # Verfügbare Evidenz für kontrollierenden Verein
    # Basis 06 hat Vorrang, Kontakt 04 ist Fallback.
    base_valid = np.isin(base_labels, ["H", "A"])
    contact_valid = np.isin(contact_labels, ["H", "A"])

    evidence = np.full(n, None, dtype=object)

    evidence[contact_valid] = contact_labels[contact_valid]
    evidence[base_valid] = base_labels[base_valid]

    # Keine Besitz-Evidenz während D übernehmen
    evidence[~live] = None

    # Zusammenhängende Live- und D-Segmente bestimmen
    segment_id = np.cumsum(
        np.r_[True, live[1:] != live[:-1]]
    )

    evidence_series = pd.Series(evidence)

    # Innerhalb jeder Live-Phase:
    # Letzten bekannten Ballbesitz fortführen
    possession = (
        evidence_series
        .groupby(segment_id)
        .ffill()
    )

    # Nur am Beginn einer Live-Phase, solange dort
    # noch keine H/A-Evidenz existiert:
    # erstes späteres H/A-Signal rückwärts übernehmen
    possession = (
        possession
        .groupby(segment_id)
        .bfill()
    )

    # Finale Labels
    labels = np.full(n, "D", dtype="<U1")

    reconstructed = possession.to_numpy()

    known = live & pd.notna(reconstructed)
    unknown = live & ~known

    labels[known] = reconstructed[known]
    labels[unknown] = "X"

    return labels, {
        "live_frames": int(live.sum()),
        "reconstructed_H": int((labels == "H").sum()),
        "reconstructed_A": int((labels == "A").sum()),
        "unresolved_X": int(unknown.sum())
    }


# Beide Viterbi-Varianten rekonstruieren
hier_v2, diag_v2 = reconstruct_possession(
    status_viterbi_v2,
    base_labels,
    contact_labels
)

hier_v2_motion, diag_motion = reconstruct_possession(
    status_viterbi_motion_v2,
    base_labels,
    contact_labels
)

# Ergebnisse vergleichen
diagnostics = pd.DataFrame([
    {"Modell": "Viterbi v2", **diag_v2},
    {"Modell": "Viterbi v2 + Motion", **diag_motion}
])

print("Hierarchische Ballbesitzrekonstruktion:")
display(diagnostics)

# Was passiert mit den 32 zuvor verlorenen Korrekturen?
corrected_frames = np.arange(16696, 16728)

print("\nRekonstruierte Labels im Diagnoseintervall:")
print(
    pd.Series(
        hier_v2[corrected_frames]
    ).value_counts()
)

# Änderungen gegenüber der OR-Baseline
public_idx = oof["frame"].to_numpy(dtype=int)

print("\nVeränderte finale Labels gegenüber OR:")

for name, labels in [
    ("Viterbi v2", hier_v2),
    ("Viterbi v2 + Motion", hier_v2_motion)
]:
    print(
        name,
        "| öffentlich:",
        np.sum(
            labels[public_idx]
            != hierarchical_or[public_idx]
        ),
        "| gesamtes Video:",
        np.sum(
            labels != hierarchical_or
        )
    )

# Vollständige Vorhersagen speichern
hier_out = (
    PROJECT_DIR / "output/hierarchical_v2"
)
hier_out.mkdir(parents=True, exist_ok=True)

for name, labels in [
    ("hier_v2_reconstructed", hier_v2),
    ("hier_v2_motion_reconstructed", hier_v2_motion)
]:
    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        hier_out / f"{name}.csv",
        index=False
    )

print("\nVorhersagen gespeichert:", hier_out)


In [ ]:

# ============================================================
# SCHRITT 23: OFFIZIELLER SPELLF1 – HIERARCHISCHES MODELL
# ============================================================

from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import subprocess
import sys
import numpy as np
import pandas as pd

# 1. Modelle zusammenstellen
models_final = {
    "hier_base06": base_labels,
    "hier_or_v0": hierarchical_or,
    "hier_v2_reconstructed": hier_v2,
    "hier_v2_motion_reconstructed": hier_v2_motion
}

OUT = PROJECT_DIR / "output/hierarchical_final_comparison"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

# 2. Vollständige Submission-Dateien schreiben
for name, labels in models_final.items():

    assert len(labels) == N_FRAMES
    assert np.isin(labels, ["H", "A", "D", "X"]).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUB / f"{name}.csv",
        index=False
    )

# 3. Offiziellen Evaluator ausführen
ARCHIVE = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official_root = Path(tmp) / root_name

    validator = (
        official_root / "evaluation/validate_submission.py"
    )
    evaluator = (
        official_root / "evaluation/run_evaluation.py"
    )

    for name in models_final:
        result = subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"Validierung fehlgeschlagen: {name}\n"
                + result.stdout + "\n" + result.stderr
            )

    result = subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Evaluation fehlgeschlagen:\n"
            + result.stdout + "\n" + result.stderr
        )

# 4. Ergebnisse vergleichen
scores = pd.read_csv(
    EVAL / "scoreboard.csv"
)

columns = [
    "submission",
    "macro_f1",
    "spell_f1_mean",
    "f1_H",
    "f1_A",
    "f1_D",
    "poss",
    "dur"
]

scores = scores[columns].sort_values(
    "spell_f1_mean",
    ascending=False
)

print("Offizielle öffentliche Evaluation:")
display(scores.round(4))

# 5. Verbesserung gegenüber unserer OR-Baseline
baseline = scores.loc[
    scores["submission"] == "hier_or_v0"
].iloc[0]

comparison = scores[
    ["submission", "macro_f1", "spell_f1_mean", "poss"]
].copy()

comparison["delta_spellF1_vs_OR"] = (
    comparison["spell_f1_mean"]
    - baseline["spell_f1_mean"]
)

comparison["delta_macroF1_vs_OR"] = (
    comparison["macro_f1"]
    - baseline["macro_f1"]
)

print("\nVeränderung gegenüber OR v0:")
display(comparison.round(4))


In [ ]:

# ============================================================
# SCHRITT 24: AUDIT DER BALLBESITZREKONSTRUKTION
# ============================================================

import numpy as np
import pandas as pd

live = status_viterbi_v2 == "L"

# Direkte Evidenz wie in unserer Rekonstruktionsfunktion
base_valid = np.isin(base_labels, ["H", "A"])
contact_valid = np.isin(contact_labels, ["H", "A"])

evidence = np.full(N_FRAMES, None, dtype=object)

evidence[live & contact_valid] = contact_labels[
    live & contact_valid
]
evidence[live & base_valid] = base_labels[
    live & base_valid
]

# Zusammenhängende Live-/D-Segmente
segment_id = np.cumsum(
    np.r_[True, live[1:] != live[:-1]]
)

evidence_series = pd.Series(evidence)

forward = evidence_series.groupby(segment_id).ffill()
reconstructed = forward.groupby(segment_id).bfill()

forward_valid = forward.notna().to_numpy()
reconstructed_valid = reconstructed.notna().to_numpy()
direct_valid = pd.notna(evidence)

# Herkunft jedes Labels
source = np.full(N_FRAMES, "D", dtype=object)

source[live & direct_valid] = "Direkte Evidenz"

source[
    live & ~direct_valid & forward_valid
] = "Vorwärtsfortführung"

source[
    live & ~direct_valid
    & ~forward_valid & reconstructed_valid
] = "Rückwärtsauffüllung"

source[
    live & ~reconstructed_valid
] = "Ohne Evidenz"

# Rekonstruktion muss zur gespeicherten Vorhersage passen
check_labels = np.full(N_FRAMES, "D", dtype="<U1")

known = live & reconstructed_valid
check_labels[known] = reconstructed.to_numpy()[known]
check_labels[live & ~known] = "X"

assert np.array_equal(check_labels, hier_v2), (
    "Audit reproduziert hier_v2 nicht."
)

# ------------------------------------------------------------
# 1. Herkunft der H/A-Labels
# ------------------------------------------------------------

summary = (
    pd.Series(source[live])
    .value_counts()
    .rename_axis("Evidenzherkunft")
    .reset_index(name="Frames")
)

summary["Sekunden"] = summary["Frames"] / FPS
summary["Anteil_LIVE"] = summary["Frames"] / live.sum()

print("Herkunft der H/A-Labels:")
display(summary.round(3))

# ------------------------------------------------------------
# 2. Zusammenhängende rekonstruierte Abschnitte
# ------------------------------------------------------------

source_df = pd.DataFrame({
    "frame": np.arange(N_FRAMES),
    "source": source
})

source_df["run"] = (
    source_df["source"]
    .ne(source_df["source"].shift())
    .cumsum()
)

inferred_runs = (
    source_df[
        source_df["source"].isin([
            "Vorwärtsfortführung",
            "Rückwärtsauffüllung"
        ])
    ]
    .groupby("run")
    .agg(
        source=("source", "first"),
        start_frame=("frame", "min"),
        end_frame=("frame", "max"),
        n_frames=("frame", "size")
    )
    .reset_index(drop=True)
)

inferred_runs["duration_s"] = (
    inferred_runs["n_frames"] / FPS
)

print("\nDauer der rekonstruierten Abschnitte:")
display(
    inferred_runs.groupby("source")["duration_s"]
    .describe()
    .round(2)
)

print("\nDie 10 längsten rekonstruierten Abschnitte:")
display(
    inferred_runs.nlargest(
        10, "duration_s"
    ).round(2)
)

# ------------------------------------------------------------
# 3. Öffentlicher vs. übriger Videobereich
# ------------------------------------------------------------

public_mask = np.zeros(N_FRAMES, dtype=bool)
public_mask[oof["frame"].to_numpy(dtype=int)] = True

print("\nEvidenzherkunft nach Videobereich:")
display(
    pd.crosstab(
        np.where(
            public_mask & live,
            "Öffentlich LIVE",
            np.where(live, "Übriges LIVE", "D")
        ),
        source
    )
)


In [ ]:

# ============================================================
# SCHRITT 25: ORIGINALEN KONTAKTALGORITHMUS ÜBERNEHMEN
# Quelle: 06_spielunterbrechungen_untersuchen.ipynb
# ============================================================

import json
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

# ------------------------------------------------------------
# 1. Originalparameter aus Notebook 04/06
# ------------------------------------------------------------

CONTACT_DIR = PROJECT_DIR / "output/contact_classification"

config = json.loads(
    (CONTACT_DIR / "run_manifest.json").read_text()
)

PITCH = np.asarray(config["pitch_m"], dtype=float)

assert config["fps"] == FPS
assert config["n_frames"] == N_FRAMES

NEAR_DISTANCE_M = config["near_distance_m"]
PEAK_ACCELERATION_M_S2 = config["peak_acceleration_m_s2"]
MIN_RUN_SECONDS = config["min_run_seconds"]
EDGE_GUARD_SECONDS = config["edge_guard_seconds"]

MEASUREMENT_SIGMA_M = config["measurement_sigma_m"]
PROCESS_SIGMA_M_S2 = config["process_sigma_m_s2"]

META_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"

# ------------------------------------------------------------
# 2. Originale Track-Reihenfolge und Teamzuordnung
# ------------------------------------------------------------

root = ET.parse(META_PATH).getroot()

team_nodes = root.findall("./Metadata/Teams/Team")

team_map = {
    team.get("id"): ("H" if i == 0 else "A")
    for i, team in enumerate(team_nodes)
}

channel_to_track = {
    channel.get("id"): channel.get("playerId")
    for channel in root.findall(
        "./Metadata/PlayerChannels/PlayerChannel"
    )
}

spec = root.find(
    "./DataFormatSpecifications/DataFormatSpecification"
)

track_ids = [
    channel_to_track[
        group.find("PlayerChannelRef").get("playerChannelId")
    ]
    for group in spec.findall("./SplitRegister/SplitRegister")
]

player_team = {
    player.get("id"): team_map[player.get("teamId")]
    for player in root.findall("./Metadata/Players/Player")
}

track_sides = np.asarray(
    [player_team[track_id] for track_id in track_ids]
)

# ------------------------------------------------------------
# 3. Trackingdaten in Spielfeldmetern einlesen
# ------------------------------------------------------------

positions = np.empty(
    (N_FRAMES, len(track_ids) + 1, 2),
    dtype=float
)

with RAW_PATH.open() as handle:
    for frame, line in enumerate(handle):

        raw_frame, players_raw, ball_raw = (
            line.strip().split(":")
        )

        assert int(raw_frame) - 1 == frame

        values = np.fromstring(
            players_raw.replace(";", ",") + "," + ball_raw,
            sep=","
        ).reshape(-1, 3)

        positions[frame] = values[:, :2] * PITCH

assert frame + 1 == N_FRAMES

player_xy_contact = positions[:, :-1]
ball_xy_contact = positions[:, -1]

ball_seen = np.isfinite(ball_xy_contact).all(axis=1)

# ------------------------------------------------------------
# 4. Originaler Rauch-Tung-Striebel-Smoother
# ------------------------------------------------------------

def smooth_ball_rts(points, fps,
                    measurement_sigma, process_sigma):

    dt = 1 / fps

    transition = np.eye(4)
    transition[0, 2] = dt
    transition[1, 3] = dt

    observation = np.eye(2, 4)

    acceleration_map = np.r_[
        np.eye(2) * dt**2 / 2,
        np.eye(2) * dt
    ]

    process_noise = (
        acceleration_map @ acceleration_map.T
        * process_sigma**2
    )

    measurement_noise = (
        np.eye(2) * measurement_sigma**2
    )

    state = np.r_[
        points[0],
        (points[1] - points[0]) / dt
    ]

    covariance = np.diag([
        measurement_sigma**2,
        measurement_sigma**2,
        100.0,
        100.0
    ])

    predicted = np.empty((len(points), 4))
    filtered = np.empty_like(predicted)

    predicted_cov = np.empty((len(points), 4, 4))
    filtered_cov = np.empty_like(predicted_cov)

    for i, point in enumerate(points):

        if i:
            state = transition @ state
            covariance = (
                transition @ covariance @ transition.T
                + process_noise
            )

        predicted[i] = state
        predicted_cov[i] = covariance

        gain = np.linalg.solve(
            observation @ covariance @ observation.T
            + measurement_noise,
            (covariance @ observation.T).T
        ).T

        state = state + gain @ (point - observation @ state)

        residual_map = np.eye(4) - gain @ observation

        covariance = (
            residual_map @ covariance @ residual_map.T
            + gain @ measurement_noise @ gain.T
        )

        filtered[i] = state
        filtered_cov[i] = covariance

    smoothed = filtered.copy()

    for i in range(len(points) - 2, -1, -1):

        gain = np.linalg.solve(
            predicted_cov[i + 1].T,
            (filtered_cov[i] @ transition.T).T
        ).T

        smoothed[i] += gain @ (
            smoothed[i + 1] - predicted[i + 1]
        )

    return smoothed


# ------------------------------------------------------------
# 5. Originale Kontaktkandidaten berechnen
# ------------------------------------------------------------

starts = np.flatnonzero(
    ball_seen & ~np.r_[False, ball_seen[:-1]]
)

ends = np.flatnonzero(
    ball_seen & ~np.r_[ball_seen[1:], False]
)

acceleration = np.full(N_FRAMES, np.nan)
nearest_distance = np.full(N_FRAMES, np.nan)
nearest_offset = np.full(N_FRAMES, np.nan)

eligible = np.zeros(N_FRAMES, dtype=bool)

edge_guard = round(EDGE_GUARD_SECONDS * FPS)

for first, last in zip(starts, ends):

    idx = np.arange(first, last + 1)

    if len(idx) / FPS < MIN_RUN_SECONDS:
        continue

    state = smooth_ball_rts(
        ball_xy_contact[idx],
        FPS,
        MEASUREMENT_SIGMA_M,
        PROCESS_SIGMA_M_S2
    )

    acceleration[idx] = np.linalg.norm(
        np.gradient(
            state[:, 2:],
            1 / FPS,
            axis=0
        ),
        axis=1
    )

    distances = np.linalg.norm(
        player_xy_contact[idx] - state[:, None, :2],
        axis=2
    )

    distances = np.where(
        np.isfinite(distances),
        distances,
        np.inf
    )

    distance_min = distances.min(axis=1)
    offset_min = distances.argmin(axis=1).astype(float)

    offset_min[~np.isfinite(distance_min)] = np.nan
    distance_min[~np.isfinite(distance_min)] = np.nan

    nearest_distance[idx] = distance_min
    nearest_offset[idx] = offset_min

    if len(idx) > 2 * edge_guard:
        eligible[idx[edge_guard:-edge_guard]] = True


acc_series = pd.Series(acceleration)

local_peak = (
    acc_series.gt(acc_series.shift())
    & acc_series.ge(acc_series.shift(-1))
).to_numpy()

candidate_mask = (
    eligible
    & local_peak
    & (acceleration >= PEAK_ACCELERATION_M_S2)
    & (nearest_distance <= NEAR_DISTANCE_M)
)

contact_candidate_side = np.full(
    N_FRAMES, "", dtype="<U1"
)

contact_candidate_side[candidate_mask] = (
    track_sides[
        nearest_offset[candidate_mask].astype(int)
    ]
)

# ------------------------------------------------------------
# 6. Originaler Kontakt-Zustandsautomat aus Notebook 06
# ------------------------------------------------------------

HALVES = [
    ("1. Halbzeit", 11185, 90651),
    ("2. Halbzeit", 110808, 190197)
]

def classify_from_contacts(baseline, event_side, halves):

    prediction = baseline.copy()

    uses_contact_state = np.zeros(
        len(baseline), dtype=bool
    )

    for _, first, last in halves:

        last_team = None

        for frame in range(first, last + 1):

            if baseline[frame] == "D":
                last_team = None
                continue

            if event_side[frame] in ("H", "A"):
                last_team = event_side[frame]

            if last_team is not None:
                prediction[frame] = last_team
                uses_contact_state[frame] = True

    return prediction, uses_contact_state


# ------------------------------------------------------------
# 7. Originalvorhersage aus Notebook 04 reproduzieren
# ------------------------------------------------------------

baseline_original = pd.read_csv(
    CONTACT_DIR / "submissions/baseline_smooth50.csv"
)["label"].to_numpy()

replayed_contact, uses_contact = classify_from_contacts(
    baseline_original,
    contact_candidate_side,
    HALVES
)

original_contact = contact_04["label"].to_numpy()

n_differences = np.count_nonzero(
    replayed_contact != original_contact
)

print("Kontaktkandidaten:", candidate_mask.sum())
print("Davon Barcelona:", (contact_candidate_side == "H").sum())
print("Davon Athletic:", (contact_candidate_side == "A").sum())
print("Abweichungen zu Notebook 04:", n_differences)

assert n_differences == 0, (
    "Die Kontaktvorhersagen wurden nicht exakt reproduziert."
)

print("\nOriginaler Kontaktalgorithmus erfolgreich reproduziert.")


In [ ]:

# ============================================================
# SCHRITT 26: KONTAKT + VITERBI -> OFFIZIELLER SPELLF1
# ============================================================

import numpy as np
import pandas as pd
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

# 1. Viterbi bestimmt ausschliesslich LIVE / D
status = status_viterbi_v2
live = status == "L"

# 2. Originale Baseline als initiale H/A-Evidenz
seed = baseline_original.copy()
seed[live & ~np.isin(seed, ["H", "A"])] = "X"
seed[~live] = "D"

# 3. Originalen Kontakt-Zustandsautomaten NEU ausführen
contact_new, contact_used = classify_from_contacts(
    seed,
    contact_candidate_side,
    HALVES
)

# 4. Fehlende H/A-Zuordnung innerhalb der Live-Phasen
#    rekonstruieren, aber nie über D hinweg
segments = np.cumsum(
    np.r_[True, live[1:] != live[:-1]]
)

pos = pd.Series(
    np.where(
        live & np.isin(contact_new, ["H", "A"]),
        contact_new,
        None
    )
)

pos = pos.groupby(segments).ffill()
pos = pos.groupby(segments).bfill()

hier_contact_v3 = np.full(N_FRAMES, "D", dtype="<U1")

known = live & pos.notna().to_numpy()
hier_contact_v3[known] = pos.to_numpy()[known]
hier_contact_v3[live & ~known] = "X"

print("Neue Kontakt-Hierarchie:")
print("Kontaktgesteuerte Frames:", int(contact_used.sum()))
print("Unbekannte Live-Frames:", int(
    (hier_contact_v3 == "X").sum()
))
print("Änderungen zu Viterbi v2 (alt):", int(
    np.count_nonzero(hier_contact_v3 != hier_v2)
))
print("Änderungen öffentlich:", int(
    np.count_nonzero(
        hier_contact_v3[oof_frames] != hier_v2[oof_frames]
    )
))

# 5. Vergleichsmodelle
models = {
    "hier_or_v0": hierarchical_or,
    "hier_v2_reconstructed": hier_v2,
    "hier_v3_contact_rerun": hier_contact_v3
}

OUT = PROJECT_DIR / "output/hierarchical_contact_test"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

for name, labels in models.items():
    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(SUB / f"{name}.csv", index=False)

# 6. Offiziellen Evaluator starten
ARCHIVE = (
    PROJECT_DIR /
    "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:
    with ZipFile(ARCHIVE) as archive:
        archive_root = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / archive_root

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:
        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

# 7. Ergebnisse
scores = pd.read_csv(EVAL / "scoreboard.csv")

cols = [
    "submission", "macro_f1", "spell_f1_mean",
    "f1_H", "f1_A", "f1_D", "poss"
]

display(
    scores[cols]
    .sort_values("spell_f1_mean", ascending=False)
    .round(4)
)

old_score = scores.loc[
    scores["submission"] == "hier_v2_reconstructed",
    "spell_f1_mean"
].iloc[0]

new_score = scores.loc[
    scores["submission"] == "hier_v3_contact_rerun",
    "spell_f1_mean"
].iloc[0]

print(f"\nBisheriger spellF1: {old_score:.4f}")
print(f"Neuer spellF1:      {new_score:.4f}")
print(f"Veränderung:        {new_score - old_score:+.4f}")


In [ ]:

# ============================================================
# SCHRITT 27: FEHLERANALYSE DER HIERARCHISCHEN MODELLE
# ============================================================

import numpy as np
import pandas as pd

public = oof.sort_values("frame").reset_index(drop=True)

frames = public["frame"].to_numpy(dtype=int)
gt = public["GT"].to_numpy()

# 1. Tatsächliche Unterschiede der Kontaktvarianten
different = hier_contact_v3 != hier_v2

print("Unterschiede Kontakt-Neuberechnung vs. bisherige H/A:")
print("Gesamtes Video:", int(different.sum()))
print("Öffentlich:", int(different[frames].sum()))

if different[frames].any():
    print("\nÖffentliche Label-Unterschiede:")
    display(pd.crosstab(
        pd.Series(hier_v2[frames][different[frames]], name="Alt"),
        pd.Series(hier_contact_v3[frames][different[frames]], name="Neu")
    ))

# 2. Falsch vorhergesagte Unterbrechungen finden
status_public = status_viterbi_v2[frames]

false_d = (status_public == "D") & (gt == "L")

error_frames = frames[false_d]

print("\nFalsch als D klassifizierte öffentliche Frames:",
      len(error_frames))

# Zusammenhängende Fehlerintervalle
if len(error_frames):

    groups = np.cumsum(
        np.r_[True, np.diff(error_frames) != 1]
    )

    records = []

    for _, group_frames in pd.Series(
        error_frames
    ).groupby(groups):

        start = int(group_frames.iloc[0])
        end = int(group_frames.iloc[-1])

        idx = np.arange(start, end + 1)

        contacts = contact_candidate_side[idx]
        moving = movement_features.loc[
            idx, "moving_fraction"
        ].to_numpy()

        records.append({
            "start_frame": start,
            "end_frame": end,
            "duration_s": len(idx) / FPS,
            "contact_H": int((contacts == "H").sum()),
            "contact_A": int((contacts == "A").sum()),
            "moving_fraction_median": float(
                np.nanmedian(moving)
            ) if np.isfinite(moving).any() else np.nan,
            "heuristic06_D_frames": int(
                (base_labels[idx] == "D").sum()
            ),
            "ada08_D_frames": int(
                (ada_labels[idx] == "D").sum()
            )
        })

    df_false_d = pd.DataFrame(records)

    print("\nFalsch erkannte D-Intervalle:")
    display(
        df_false_d.sort_values(
            "duration_s", ascending=False
        ).round(3)
    )

# 3. Öffentliche H/A-Phasen des besten Modells
labels = hier_v2[frames]

run_start = np.r_[
    True,
    (labels[1:] != labels[:-1])
    | (np.diff(frames) != 1)
]

run_id = np.cumsum(run_start)

spell_df = (
    pd.DataFrame({
        "frame": frames,
        "label": labels,
        "run": run_id
    })
    .groupby("run")
    .agg(
        label=("label", "first"),
        start_frame=("frame", "min"),
        end_frame=("frame", "max"),
        n_frames=("frame", "size")
    )
    .reset_index(drop=True)
)

spell_df = spell_df[
    spell_df["label"].isin(["H", "A"])
].copy()

spell_df["duration_s"] = (
    spell_df["n_frames"] / FPS
)

print("\nVorhergesagte öffentliche Ballbesitzphasen:")
display(spell_df.round(2))


In [ ]:

# ============================================================
# SCHRITT 28: UNTERBRECHUNGSGRENZEN ANALYSIEREN
# ============================================================

import numpy as np
import pandas as pd

public = oof.sort_values("frame").reset_index(drop=True)
frames = public["frame"].to_numpy(dtype=int)

gt_status = public["GT"].to_numpy()
pred_status = status_viterbi_v2[frames]


def get_d_intervals(frame_ids, labels):
    """Zusammenhängende D-Intervalle extrahieren."""
    mask = labels == "D"
    selected = frame_ids[mask]

    if len(selected) == 0:
        return pd.DataFrame(columns=["start", "end"])

    group = np.cumsum(
        np.r_[True, np.diff(selected) != 1]
    )

    return (
        pd.DataFrame({"frame": selected, "group": group})
        .groupby("group")
        .agg(start=("frame", "min"),
             end=("frame", "max"))
        .reset_index(drop=True)
    )


gt_intervals = get_d_intervals(frames, gt_status)
pred_intervals = get_d_intervals(frames, pred_status)

rows = []
matched_pred = set()

for i, gt_row in gt_intervals.iterrows():

    start_gt = int(gt_row["start"])
    end_gt = int(gt_row["end"])

    overlaps = np.maximum(
        0,
        np.minimum(pred_intervals["end"].to_numpy(), end_gt)
        - np.maximum(pred_intervals["start"].to_numpy(), start_gt)
        + 1
    )

    if len(overlaps) == 0 or overlaps.max() == 0:
        rows.append({
            "GT_Start": start_gt,
            "GT_Ende": end_gt,
            "Pred_Start": np.nan,
            "Pred_Ende": np.nan,
            "Startfehler_s": np.nan,
            "Endfehler_s": np.nan,
            "Erkannte_D_s": 0
        })
        continue

    j = int(np.argmax(overlaps))
    matched_pred.add(j)

    pred_row = pred_intervals.iloc[j]

    rows.append({
        "GT_Start": start_gt,
        "GT_Ende": end_gt,
        "Pred_Start": int(pred_row["start"]),
        "Pred_Ende": int(pred_row["end"]),
        "Startfehler_s": (
            int(pred_row["start"]) - start_gt
        ) / FPS,
        "Endfehler_s": (
            int(pred_row["end"]) - end_gt
        ) / FPS,
        "Erkannte_D_s": overlaps[j] / FPS
    })

boundary_results = pd.DataFrame(rows)

print("Zeitliche Abweichungen an tatsächlichen D-Phasen:")
display(boundary_results.round(2))

print("\nZusätzliche vorhergesagte D-Phasen ohne GT-Überlappung:")

unmatched = pred_intervals.loc[
    ~pred_intervals.index.isin(matched_pred)
].copy()

unmatched["duration_s"] = (
    unmatched["end"] - unmatched["start"] + 1
) / FPS

display(unmatched.round(2))

print("\nGesamtfehler der Spielstatusklassifikation:")
print(
    "Falsch D:",
    int(((pred_status == "D") & (gt_status == "L")).sum()),
    "Frames"
)
print(
    "Übersehene D:",
    int(((pred_status == "L") & (gt_status == "D")).sum()),
    "Frames"
)


In [ ]:

# ============================================================
# SCHRITT 29: KONTINUIERLICHE STATUS-EVIDENZ + SPELLF1
# ============================================================

import numpy as np
import pandas as pd
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

# ------------------------------------------------------------
# 1. AdaBoost-Scores laden
# ------------------------------------------------------------

ada_score = ml_08["AdaBoost_D_score"].to_numpy(
    dtype=float
).copy()

# Öffentliche Frames: zwingend OOF statt In-Sample
public_idx = oof["frame"].to_numpy(dtype=int)

ada_score[public_idx] = oof[
    "AdaBoost_D_score"
].to_numpy(dtype=float)

valid = np.isfinite(ada_score)

assert np.all(
    (ada_score[valid] >= 0)
    & (ada_score[valid] <= 1)
)

# ------------------------------------------------------------
# 2. Score-Abstände zu 0.5 normalisieren
#    Ohne Ground Truth und ohne Parameteroptimierung
# ------------------------------------------------------------

public_mask = np.zeros(N_FRAMES, dtype=bool)
public_mask[public_idx] = True

scale_mask = valid & ~public_mask

score_scale = np.quantile(
    np.abs(ada_score[scale_mask] - 0.5),
    0.90
)

assert score_scale > 1e-8, (
    "AdaBoost-Scores haben zu geringe Streuung."
)

# Zentrierter und begrenzter ML-Evidenzwert
ml_evidence = np.zeros(N_FRAMES, dtype=float)

ml_evidence[valid] = 2.0 * np.tanh(
    (ada_score[valid] - 0.5) / score_scale
)

# Heuristik: eigenständige, binäre Evidenz
heur_evidence = np.where(
    base_labels == "D", 1.0, -1.0
)

# ------------------------------------------------------------
# 3. Zwei neue Decoder-Varianten
# ------------------------------------------------------------

# Nur ML, bei fehlenden ML-Scores Heuristik
evidence_ml = ml_evidence.copy()
evidence_ml[~valid] = 2.0 * heur_evidence[~valid]

# ML + schwächer gewichtete Heuristik
evidence_fusion = (
    ml_evidence + 0.75 * heur_evidence
)
evidence_fusion[~valid] = (
    2.0 * heur_evidence[~valid]
)

# Zeitliche Skalierung wie bei unserem Viterbi v2
status_ml_soft = viterbi_status(
    evidence_ml / FPS,
    switch_penalty=1.0
)

status_fusion_soft = viterbi_status(
    evidence_fusion / FPS,
    switch_penalty=1.0
)

# ------------------------------------------------------------
# 4. H/A innerhalb der Live-Phasen rekonstruieren
# ------------------------------------------------------------

hier_ml_soft, diag_ml = reconstruct_possession(
    status_ml_soft,
    base_labels,
    contact_labels
)

hier_fusion_soft, diag_fusion = reconstruct_possession(
    status_fusion_soft,
    base_labels,
    contact_labels
)

# ------------------------------------------------------------
# 5. Modelle vergleichen
# ------------------------------------------------------------

models = {
    "hier_v2_reference": hier_v2,
    "hier_ml_soft": hier_ml_soft,
    "hier_fusion_soft": hier_fusion_soft
}

status_models = {
    "hier_ml_soft": status_ml_soft,
    "hier_fusion_soft": status_fusion_soft
}

print("AdaBoost-Skalierung:", round(score_scale, 6))

diagnostics = []

for name, status in status_models.items():

    labels = models[name]

    diagnostics.append({
        "Modell": name,
        "D_Phasen_gesamt": len(
            extract_d_spells(
                status,
                np.arange(N_FRAMES)
            )
        ),
        "D_Phasen_public": len(
            extract_d_spells(
                status[public_idx],
                public_idx
            )
        ),
        "Statuswechsel_public": int(
            np.count_nonzero(
                status[public_idx]
                != status_viterbi_v2[public_idx]
            )
        ),
        "Labelwechsel_public": int(
            np.count_nonzero(
                labels[public_idx]
                != hier_v2[public_idx]
            )
        ),
        "Labelwechsel_gesamt": int(
            np.count_nonzero(labels != hier_v2)
        )
    })

print("\nDiagnostik:")
display(pd.DataFrame(diagnostics))

# ------------------------------------------------------------
# 6. Offizielle Evaluation
# ------------------------------------------------------------

OUT = PROJECT_DIR / "output/hierarchical_soft_evidence"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

for name, labels in models.items():

    assert len(labels) == N_FRAMES
    assert np.isin(
        labels, ["H", "A", "D", "X"]
    ).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUB / f"{name}.csv",
        index=False
    )

ARCHIVE = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root_name

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:

        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\nOffizieller spellF1-Vergleich:")

display(
    scores[
        [
            "submission",
            "macro_f1",
            "spell_f1_mean",
            "f1_H",
            "f1_A",
            "f1_D",
            "poss"
        ]
    ]
    .sort_values("spell_f1_mean", ascending=False)
    .round(4)
)


In [ ]:

# ============================================================
# SCHRITT 30: D-PHASEN DER DREI STATUSMODELLE VERGLEICHEN
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

public = oof.sort_values("frame").reset_index(drop=True)

frames = public["frame"].to_numpy(dtype=int)
gt = public["GT"].to_numpy()

status_models = {
    "Viterbi v2": status_viterbi_v2,
    "ML Soft": status_ml_soft,
    "Fusion Soft": status_fusion_soft
}


def get_intervals(frame_ids, labels, target="D"):
    selected = frame_ids[labels == target]

    if len(selected) == 0:
        return []

    boundaries = np.flatnonzero(
        np.diff(selected) > 1
    ) + 1

    chunks = np.split(selected, boundaries)

    return [
        (int(chunk[0]), int(chunk[-1]))
        for chunk in chunks
    ]


gt_spells = get_intervals(frames, gt)

results = []
summary = []

for model_name, full_status in status_models.items():

    pred = full_status[frames]
    pred_spells = get_intervals(frames, pred)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            gt,
            pred,
            labels=["D"],
            zero_division=0
        )
    )

    hit_spells = 0

    for event_id, (gs, ge) in enumerate(gt_spells, 1):

        overlaps = []

        for ps, pe in pred_spells:
            intersection = max(
                0, min(ge, pe) - max(gs, ps) + 1
            )

            if intersection:
                overlaps.append((ps, pe, intersection))

        total_covered = sum(
            1 for f, label in zip(frames, pred)
            if gs <= f <= ge and label == "D"
        )

        coverage = total_covered / (ge - gs + 1)

        if overlaps:
            hit_spells += 1
            ps, pe, overlap = max(
                overlaps, key=lambda x: x[2]
            )

            start_error = (ps - gs) / FPS
            end_error = (pe - ge) / FPS
        else:
            ps = pe = None
            start_error = end_error = np.nan

        results.append({
            "Modell": model_name,
            "GT_Ereignis": event_id,
            "GT_Start": gs,
            "GT_Ende": ge,
            "Pred_Start": ps,
            "Pred_Ende": pe,
            "D_Abdeckung_%": 100 * coverage,
            "Startfehler_s": start_error,
            "Endfehler_s": end_error
        })

    false_d = int(
        ((pred == "D") & (gt == "L")).sum()
    )

    missed_d = int(
        ((pred == "L") & (gt == "D")).sum()
    )

    summary.append({
        "Modell": model_name,
        "D_Phasen": len(pred_spells),
        "GT_Phasen_getroffen": hit_spells,
        "D_Precision": precision[0],
        "D_Recall": recall[0],
        "D_F1": f1[0],
        "Falsche_D_s": false_d / FPS,
        "Übersehene_D_s": missed_d / FPS
    })


print("1. Gesamtergebnis:")
display(pd.DataFrame(summary).round(3))

print("\n2. Abdeckung der einzelnen GT-Unterbrechungen:")
display(pd.DataFrame(results).round(2))


In [ ]:

# ============================================================
# SCHRITT 31: STATUS-MODELLWIDERSPRÜCHE IM ÜBRIGEN SPIEL
# ============================================================

import numpy as np
import pandas as pd

# Nur aktive Spielhälften berücksichtigen
active = np.zeros(N_FRAMES, dtype=bool)

for _, start, end in HALVES:
    active[start:end + 1] = True

# Öffentlichen GT-Bereich vollständig ausschließen
public_region = np.zeros(N_FRAMES, dtype=bool)
public_region[oof["frame"].to_numpy(dtype=int)] = True

# D-Intervalle innerhalb der aktiven Spielhälften
status_active = np.where(
    active & (status_viterbi_v2 == "D"),
    "D", "L"
)

d_spells = get_intervals(
    np.arange(N_FRAMES),
    status_active
)

records = []

for start, end in d_spells:

    idx = np.arange(start, end + 1)
    duration = len(idx) / FPS

    # Nur vollständige Intervalle außerhalb Public-GT.
    # Extreme Längen zunächst aus dem Szenen-Audit entfernen.
    if public_region[idx].any():
        continue
    if not (2 <= duration <= 90):
        continue

    heur_share = np.mean(base_labels[idx] == "D")

    ada_known = np.isin(ada_labels[idx], ["D", "L"])
    ada_share = (
        np.mean(ada_labels[idx][ada_known] == "D")
        if ada_known.any() else np.nan
    )

    # Mehrheitliche Modellunterstützung
    if np.isnan(ada_share):
        category = "ML nicht verfügbar"
    elif heur_share >= 0.7 and ada_share < 0.3:
        category = "Nur Heuristik"
    elif ada_share >= 0.7 and heur_share < 0.3:
        category = "Nur AdaBoost"
    elif heur_share >= 0.7 and ada_share >= 0.7:
        category = "Beide Modelle"
    else:
        category = "Gemischte Evidenz"

    contacts = contact_candidate_side[idx]

    records.append({
        "start_frame": start,
        "end_frame": end,
        "video_time": pd.to_timedelta(
            start / FPS, unit="s"
        ),
        "duration_s": duration,
        "category": category,
        "heur_D_share": heur_share,
        "ada_D_share": ada_share,
        "contact_events": int(
            np.isin(contacts, ["H", "A"]).sum()
        ),
        "median_moving_fraction": float(
            np.nanmedian(
                movement_features["moving_fraction"]
                .to_numpy()[idx]
            )
        )
    })

df_d_audit = pd.DataFrame(records)

print("D-Intervalle nach Modellunterstützung:")
display(
    df_d_audit["category"]
    .value_counts()
    .rename_axis("Kategorie")
    .reset_index(name="Intervalle")
)

# Reproduzierbare Stichprobe statt nur Extremfälle
rng = np.random.default_rng(2027)

samples = []

for category, group in df_d_audit.groupby("category"):

    n = min(4, len(group))

    selected = rng.choice(
        group.index.to_numpy(),
        size=n,
        replace=False
    )

    samples.append(df_d_audit.loc[selected])

audit_sample = (
    pd.concat(samples, ignore_index=True)
    .sort_values(["category", "start_frame"])
)

print("\nStichprobe für die Videoüberprüfung:")
display(
    audit_sample.round(3)
)

# Ergebnisse lokal speichern
audit_dir = (
    PROJECT_DIR / "output/hierarchical_boundary_audit"
)
audit_dir.mkdir(parents=True, exist_ok=True)

df_d_audit.to_csv(
    audit_dir / "all_d_intervals.csv",
    index=False
)

audit_sample.to_csv(
    audit_dir / "video_review_sample.csv",
    index=False
)


In [ ]:

# ============================================================
# SCHRITT 32: LIVE-EVIDENZ AM BEGINN VON D-INTERVALLEN
# ============================================================

import numpy as np
import pandas as pd

motion = movement_features.set_index("frame")
window = 5 * FPS

def safe_median(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(np.median(values)) if len(values) else np.nan

rows = []

for _, event in df_d_audit.iterrows():

    start = int(event["start_frame"])
    end = int(event["end_frame"])

    before = motion.iloc[max(0, start-window):start]
    early = motion.iloc[start:min(end+1, start+window)]
    late = motion.iloc[min(end+1, start+window):end+1]

    moving_before = safe_median(before["moving_fraction"])
    moving_early = safe_median(early["moving_fraction"])

    rows.append({
        "start_frame": start,
        "end_frame": end,
        "video_time": event["video_time"],
        "category": event["category"],
        "duration_s": event["duration_s"],
        "moving_before5": moving_before,
        "moving_first5": moving_early,
        "moving_delta": moving_early - moving_before,
        "moving_later": safe_median(late["moving_fraction"]),
        "ball_speed_first5": safe_median(early["ball_speed_mps"]),
        "ball_coverage_first5": safe_median(early["ball_coverage"]),
        "contacts_first5": int(np.isin(
            contact_candidate_side[
                start:min(end+1, start+window)
            ],
            ["H", "A"]
        ).sum())
    })

df_boundary_features = pd.DataFrame(rows)

# Visuell geprüfte Szenen:
# Nur diagnostische Kennzeichnung, KEINE Trainingslabels
# und KEINE manuelle Korrektur der Submission.
reviewed = {
    19772: "Beginn zu früh",
    83578: "Falsche D-Phase",
    143920: "Falsche D-Phase",
    54785: "Falsche D-Phase",
    31170: "Bestätigte D-Phase",
    142570: "D-Phase, Grenzen ungenau"
}

df_boundary_features["video_review"] = (
    df_boundary_features["start_frame"]
    .map(reviewed)
    .fillna("Nicht geprüft")
)

# Aktivitätsrang innerhalb aller 94 Intervalle
df_boundary_features["activity_percentile"] = (
    df_boundary_features["moving_first5"]
    .rank(pct=True) * 100
)

print("1. Sechs visuell geprüfte Szenen:")
display(
    df_boundary_features.loc[
        df_boundary_features["video_review"] != "Nicht geprüft",
        [
            "start_frame",
            "video_review",
            "moving_before5",
            "moving_first5",
            "moving_delta",
            "ball_speed_first5",
            "ball_coverage_first5",
            "activity_percentile"
        ]
    ].round(3)
)

print("\n2. Aktivität am D-Beginn nach Modellkategorie:")
display(
    df_boundary_features.groupby("category").agg(
        n=("start_frame", "size"),
        moving_first5_median=("moving_first5", "median"),
        moving_delta_median=("moving_delta", "median"),
        ball_speed_median=("ball_speed_first5", "median"),
        ball_coverage_median=("ball_coverage_first5", "median")
    ).round(3)
)

print("\n3. Zwölf D-Intervalle mit höchster anfänglicher Aktivität:")
display(
    df_boundary_features.nlargest(12, "moving_first5")[
        [
            "start_frame",
            "video_time",
            "category",
            "duration_s",
            "moving_first5",
            "moving_delta",
            "ball_speed_first5",
            "video_review"
        ]
    ].round(3)
)

# Nur Auditdaten speichern – keine Submission ändern
audit_dir = PROJECT_DIR / "output/hierarchical_boundary_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

df_boundary_features.to_csv(
    audit_dir / "boundary_motion_features.csv",
    index=False
)


In [ ]:

# ============================================================
# SCHRITT 33: KONSERVATIVER LIVE-VETO + OFFIZIELLER SPELLF1
# ============================================================

import numpy as np
import pandas as pd
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

# 1. Feste Testannahmen (nicht auf GT optimiert)
ACTIVITY_THRESHOLD = 0.60
MODEL_SUPPORT = 0.70
MODEL_OPPOSITION = 0.30
ONSET_SECONDS = 5

moving = movement_features[
    "moving_fraction"
].to_numpy(dtype=float)

# 2. Statusvorhersage kopieren
status_veto = status_viterbi_v2.copy()

active = np.zeros(N_FRAMES, dtype=bool)
for _, start, end in HALVES:
    active[start:end + 1] = True

intervals = get_intervals(
    np.arange(N_FRAMES),
    np.where(active & (status_vito := (
        status_viterbi_v2 == "D"
    )), "D", "L")
)

changes = []

# 3. Regel auf alle aktiven D-Intervalle anwenden
for start, end in intervals:

    idx = np.arange(start, end + 1)

    heur_share = np.mean(base_labels[idx] == "D")

    ada_known = np.isin(
        ada_labels[idx], ["D", "L"]
    )

    if not ada_known.any():
        continue

    ada_share = np.mean(
        ada_labels[idx][ada_known] == "D"
    )

    # Nur klar einseitig unterstützte D-Phasen
    single_support = (
        (
            heur_share >= MODEL_SUPPORT
            and ada_share <= MODEL_OPPOSITION
        )
        or
        (
            ada_share >= MODEL_SUPPORT
            and heur_share <= MODEL_OPPOSITION
        )
    )

    if not single_support:
        continue

    first = moving[
        start:min(end + 1, start + ONSET_SECONDS * FPS)
    ]

    first = first[np.isfinite(first)]

    if len(first) == 0:
        continue

    if np.median(first) < ACTIVITY_THRESHOLD:
        continue

    # Nur zusammenhängende aktive Sekunden am Beginn
    new_end = start - 1

    for chunk_start in range(start, end + 1, FPS):

        chunk_end = min(end + 1, chunk_start + FPS)

        activity = moving[chunk_start:chunk_end]
        activity = activity[np.isfinite(activity)]

        if len(activity) == 0:
            break

        if np.median(activity) < ACTIVITY_THRESHOLD:
            break

        status_veto[chunk_start:chunk_end] = "L"
        new_end = chunk_end - 1

    if new_end >= start:
        changes.append({
            "start_frame": start,
            "end_frame": end,
            "corrected_until": new_end,
            "corrected_seconds": (new_end-start+1)/FPS,
            "heur_D_share": heur_share,
            "ada_D_share": ada_share,
            "first5_activity": np.median(first)
        })

df_veto_changes = pd.DataFrame(changes)

# 4. Ballbesitz innerhalb neuer LIVE-Phasen rekonstruieren
hier_veto, veto_diagnostics = reconstruct_possession(
    status_veto,
    base_labels,
    contact_labels
)

print("Geänderte D-Intervalle:", len(changes))
print("Geänderte Statusframes:", int(
    np.count_nonzero(status_veto != status_viterbi_v2)
))
print("Ungeklärte Live-Frames:", veto_diagnostics["unresolved_X"])

print("\nVeränderte Intervalle:")
display(df_veto_changes.round(3))

# 5. Diagnostik an bereits überprüften Szenen
# Diese Frames dienen NUR zur Anzeige.
reviewed_starts = [
    19772, 31170, 54785,
    83578, 142570, 143920
]

print("\nVideo-Audit: Auswirkungen der allgemeinen Regel:")

if len(df_veto_changes):
    display(
        df_veto_changes[
            df_veto_changes["start_frame"].isin(
                reviewed_starts
            )
        ].round(3)
    )

# 6. Offizieller spellF1
OUT = PROJECT_DIR / "output/hierarchical_live_veto"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

models = {
    "hier_v2_reference": hier_v2,
    "hier_live_veto_v1": hier_veto
}

for name, labels in models.items():
    assert len(labels) == N_FRAMES
    assert np.isin(labels, ["H", "A", "D", "X"]).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(SUB / f"{name}.csv", index=False)

ARCHIVE = (
    PROJECT_DIR /
    "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:
        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\nOffizieller öffentlicher Score:")
display(
    scores[
        [
            "submission", "macro_f1",
            "spell_f1_mean", "f1_H",
            "f1_A", "f1_D", "poss"
        ]
    ].sort_values(
        "spell_f1_mean", ascending=False
    ).round(4)
)


In [ ]:

# ============================================================
# SCHRITT 34: DYNAMISCHER LIVE-VETO MIT HYSTERESE
# ============================================================

import numpy as np
import pandas as pd

ACTIVITY_START = 0.60
ACTIVITY_STOP = 0.25
QUIET_SECONDS = 2

moving = movement_features[
    "moving_fraction"
].to_numpy(dtype=float)

status_veto_v2 = status_viterbi_v2.copy()

active = np.zeros(N_FRAMES, dtype=bool)

for _, start, end in HALVES:
    active[start:end + 1] = True

d_intervals = get_intervals(
    np.arange(N_FRAMES),
    np.where(
        active & (status_viterbi_v2 == "D"),
        "D", "L"
    )
)

records = []

for start, end in d_intervals:

    idx = np.arange(start, end + 1)

    heur_share = np.mean(base_labels[idx] == "D")

    ada_known = np.isin(
        ada_labels[idx], ["D", "L"]
    )

    if not ada_known.any():
        continue

    ada_share = np.mean(
        ada_labels[idx][ada_known] == "D"
    )

    # Gleiche Modellwiderspruchsregel wie in Schritt 33
    single_support = (
        (heur_share >= 0.70 and ada_share <= 0.30)
        or
        (ada_share >= 0.70 and heur_share <= 0.30)
    )

    if not single_support:
        continue

    first5 = moving[
        start:min(end + 1, start + 5 * FPS)
    ]

    first5 = first5[np.isfinite(first5)]

    if len(first5) == 0:
        continue

    if np.median(first5) < ACTIVITY_START:
        continue

    # Hysterese: vereinzelte ruhige Sekunden zulassen,
    # bei zwei aufeinanderfolgenden ruhigen Sekunden stoppen.
    quiet_count = 0
    last_active_end = start - 1

    for second_start in range(start, end + 1, FPS):

        second_end = min(
            end + 1,
            second_start + FPS
        )

        values = moving[second_start:second_end]
        values = values[np.isfinite(values)]

        # Fehlende Evidenz -> konservativ abbrechen
        if len(values) == 0:
            break

        activity = np.median(values)

        if activity < ACTIVITY_STOP:
            quiet_count += 1

            if quiet_count >= QUIET_SECONDS:
                break
        else:
            quiet_count = 0
            last_active_end = second_end - 1

    # Nur bis zur letzten aktiv gestützten Sekunde
    if last_active_end < start:
        continue

    status_veto_v2[
        start:last_active_end + 1
    ] = "L"

    records.append({
        "start_frame": start,
        "end_frame": end,
        "corrected_until": last_active_end,
        "corrected_seconds": (
            last_active_end - start + 1
        ) / FPS,
        "original_duration_s": len(idx) / FPS,
        "heur_D_share": heur_share,
        "ada_D_share": ada_share
    })

df_veto_v2 = pd.DataFrame(records)

# H/A neu rekonstruieren
hier_veto_v2, diag = reconstruct_possession(
    status_veto_v2,
    base_labels,
    contact_labels
)

# Gesamtdiagnostik
print("Dynamischer Live-Veto:")
print("Veränderte D-Intervalle:", len(df_veto_v2))
print("Korrigierte Frames:", int(
    np.count_nonzero(
        status_veto_v2 != status_viterbi_v2
    )
))
print("Korrigierte Sekunden:", round(
    np.count_nonzero(
        status_veto_v2 != status_viterbi_v2
    ) / FPS, 2
))
print("Ungeklärte Live-Frames:", diag["unresolved_X"])

# Vergleich mit bisheriger Veto-Version
comparison = df_veto_v2.merge(
    df_veto_changes[
        ["start_frame", "corrected_seconds"]
    ].rename(
        columns={
            "corrected_seconds": "v1_seconds"
        }
    ),
    on="start_frame",
    how="left"
)

comparison["additional_seconds"] = (
    comparison["corrected_seconds"]
    - comparison["v1_seconds"].fillna(0)
)

print("\nVergleich alter und neuer Live-Veto:")
display(
    comparison[
        [
            "start_frame",
            "original_duration_s",
            "v1_seconds",
            "corrected_seconds",
            "additional_seconds"
        ]
    ].round(2)
)

# Wichtig: öffentliche Vorhersagen unverändert?
public_idx = oof["frame"].to_numpy(dtype=int)

print("\nÄnderungen im öffentlichen Bereich:", int(
    np.count_nonzero(
        hier_veto_v2[public_idx] != hier_v2[public_idx]
    )
))

# Experiment speichern, Referenz nicht überschreiben
OUT = PROJECT_DIR / "output/hierarchical_live_veto"
OUT.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "frame": np.arange(N_FRAMES),
    "label": hier_veto_v2
}).to_csv(
    OUT / "hier_live_veto_v2.csv",
    index=False
)


In [ ]:

# ============================================================
# SCHRITT 35: LIVE-VETO V2 – OFFIZIELLE EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
from sklearn.metrics import precision_recall_fscore_support

public = oof.sort_values("frame").reset_index(drop=True)
frames = public["frame"].to_numpy(dtype=int)
gt = public["GT"].to_numpy()

# ------------------------------------------------------------
# 1. Tatsächlich veränderte Statusframes
# ------------------------------------------------------------

old_status = status_viterbi_v2[frames]
new_status = status_veto_v2[frames]

changed = old_status != new_status

print("Öffentlich veränderte Frames:", int(changed.sum()))

print("\nGround Truth der veränderten Frames:")
display(
    pd.Series(gt[changed], name="GT")
    .value_counts()
    .rename_axis("Tatsächlicher Status")
    .reset_index(name="Frames")
)

# Zusammenhängende Intervalle
changed_frames = frames[changed]

if len(changed_frames):

    groups = np.cumsum(
        np.r_[True, np.diff(changed_frames) != 1]
    )

    records = []

    for _, grp in pd.DataFrame({
        "frame": changed_frames,
        "GT": gt[changed],
        "group": groups
    }).groupby("group"):

        records.append({
            "start": int(grp["frame"].min()),
            "end": int(grp["frame"].max()),
            "duration_s": len(grp) / FPS,
            "GT_L_frames": int((grp["GT"] == "L").sum()),
            "GT_D_frames": int((grp["GT"] == "D").sum())
        })

    print("\nVeränderte öffentliche Intervalle:")
    display(pd.DataFrame(records))

# ------------------------------------------------------------
# 2. D-Precision / Recall / F1
# ------------------------------------------------------------

metrics = []

for name, pred in {
    "Viterbi v2": old_status,
    "Live-Veto v2": new_status
}.items():

    p, r, f1, _ = precision_recall_fscore_support(
        gt, pred, labels=["D"], zero_division=0
    )

    metrics.append({
        "Modell": name,
        "D_Precision": p[0],
        "D_Recall": r[0],
        "D_F1": f1[0]
    })

print("\nSpielstatusqualität:")
display(pd.DataFrame(metrics).round(4))

# ------------------------------------------------------------
# 3. Offizielle spellF1-Evaluation
# ------------------------------------------------------------

OUT = PROJECT_DIR / "output/hierarchical_veto_v2_evaluation"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

models = {
    "hier_v2_reference": hier_v2,
    "hier_live_veto_v1": hier_veto,
    "hier_live_veto_v2": hier_veto_v2
}

for name, labels in models.items():

    assert len(labels) == N_FRAMES
    assert np.isin(labels, ["H", "A", "D", "X"]).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(SUB / f"{name}.csv", index=False)

ARCHIVE = (
    PROJECT_DIR /
    "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root_name

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:

        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\nOffizielle öffentliche Evaluation:")

display(
    scores[
        [
            "submission",
            "macro_f1",
            "spell_f1_mean",
            "f1_H",
            "f1_A",
            "f1_D",
            "poss"
        ]
    ]
    .sort_values("spell_f1_mean", ascending=False)
    .round(4)
)


In [ ]:

# ============================================================
# SCHRITT 36: BALL- UND SPIELEREVIDENZ BEI D-ENTSCHEIDUNGEN
# ============================================================

import numpy as np
import pandas as pd

# Bereits visuell überprüfte Situationen.
# Ausschließlich Diagnose, keine Trainingslabels.
audit_cases = [
    (18026, "Echte D-Phase (öffentlich)"),
    (19772, "D beginnt zu früh"),
    (31170, "Echte D-Phase (Foul)"),
    (42853, "Falsche D-Phase"),
    (54785, "Falsche D-Phase"),
    (67524, "Echte D-Phase (Freistoß)"),
    (83578, "Falsche D-Phase"),
    (86237, "Falsche D-Phase"),
    (136650, "Falsche D-Phase"),
    (142570, "D beginnt zu früh"),
    (143920, "Falsche D-Phase"),
    (151893, "Falsche D-Phase")
]

WINDOW = 5 * FPS

motion = movement_features.set_index("frame")

def finite_median(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

rows = []

for start, observation in audit_cases:

    idx = np.arange(
        start, min(start + WINDOW, N_FRAMES)
    )

    before = np.arange(
        max(0, start - WINDOW), start
    )

    # Ball-Tracking und tatsächliche Nähe zu Spielern
    ball_valid = ball_seen[idx]
    nearest = nearest_distance[idx]

    proximity_valid = np.isfinite(nearest)

    near_player = (
        np.mean(nearest[proximity_valid] <= 3.0)
        if proximity_valid.any() else np.nan
    )

    moving_first = finite_median(
        motion.loc[idx, "moving_fraction"]
    )

    moving_before = finite_median(
        motion.loc[before, "moving_fraction"]
    )

    rows.append({
        "start_frame": start,
        "video_time_s": start / FPS,
        "Beobachtung": observation,
        "moving_first5": moving_first,
        "moving_delta": moving_first - moving_before,
        "ball_visible_share": float(np.mean(ball_valid)),
        "ball_speed_mps": finite_median(
            motion.loc[idx, "ball_speed_mps"]
        ),
        "nearest_player_m": finite_median(nearest),
        "ball_within_3m_share": near_player,
        "contact_candidates": int(np.isin(
            contact_candidate_side[idx], ["H", "A"]
        ).sum()),
        "heur_D_share": float(
            np.mean(base_labels[idx] == "D")
        ),
        "ada_D_share": float(
            np.mean(ada_labels[idx] == "D")
        )
    })

df_physical_evidence = pd.DataFrame(rows)

print("Ball- und Bewegungsevidenz, erste 5 Sekunden:")
display(df_physical_evidence.round(3))

print("\nGruppenvergleich (nur eindeutig überprüfte Fälle):")

clear = df_physical_evidence[
    df_physical_evidence["Beobachtung"].isin([
        "Falsche D-Phase",
        "Echte D-Phase (öffentlich)",
        "Echte D-Phase (Foul)",
        "Echte D-Phase (Freistoß)"
    ])
].copy()

clear["Gruppe"] = np.where(
    clear["Beobachtung"] == "Falsche D-Phase",
    "Falsch D",
    "Echt D"
)

display(
    clear.groupby("Gruppe").agg(
        n=("start_frame", "size"),
        moving=("moving_first5", "median"),
        ball_visible=("ball_visible_share", "median"),
        ball_speed=("ball_speed_mps", "median"),
        nearest_player=("nearest_player_m", "median"),
        ball_near_player=("ball_within_3m_share", "median"),
        contacts=("contact_candidates", "median")
    ).round(3)
)


In [ ]:

# ============================================================
# SCHRITT 37: PHYSIKALISCHE LIVE-EVIDENZ IN VITERBI
# ============================================================

import numpy as np
import pandas as pd

# 1. Sicherheitscheck: Referenz exakt reproduzieren
assert np.array_equal(
    viterbi_status(base_logit / FPS, switch_penalty=1.0),
    status_viterbi_v2
)

# 2. Frameweise Bewegungsmerkmale
moving = movement_features["moving_fraction"].to_numpy(float)
ball_speed = movement_features["ball_speed_mps"].to_numpy(float)
ball_coverage = movement_features["ball_coverage"].to_numpy(float)

nearest_smooth = (
    pd.Series(nearest_distance)
    .rolling(FPS, center=True, min_periods=5)
    .median()
    .to_numpy()
)

# 3. Nur bei widersprüchlichen Statusmodellen
heur_d = base_labels == "D"
ada_d = ada_labels == "D"
ada_known = np.isin(ada_labels, ["D", "L"])

disagreement = ada_known & (heur_d != ada_d)

active = np.zeros(N_FRAMES, dtype=bool)
for _, start, end in HALVES:
    active[start:end + 1] = True

# 4. Gemeinsame Live-Evidenz
# Feste experimentelle Annahmen, NICHT kalibriert.
#
# Balltracking muss ausreichend vorhanden sein.
# Fehlende Werte erzeugen keine positive Live-Evidenz.
physical_live = (
    active
    & disagreement
    & np.isfinite(moving)
    & np.isfinite(ball_speed)
    & np.isfinite(nearest_smooth)
    & (moving >= 0.40)
    & (ball_speed >= 6.0)
    & (ball_coverage >= 0.50)
    & (nearest_smooth <= 3.5)
)

# 5. Physikalische Evidenz schwächt D ab,
# erzwingt aber keinen Statuswechsel.
PHYSICS_WEIGHT = 1.5

physics_logit = base_logit.copy()
physics_logit[physical_live] -= PHYSICS_WEIGHT

status_physics_v1 = viterbi_status(
    physics_logit / FPS,
    switch_penalty=1.0
)

# 6. Ballbesitz rekonstruieren
hier_physics_v1, physics_diag = reconstruct_possession(
    status_physics_v1,
    base_labels,
    contact_labels
)

# 7. Änderungen gegenüber Viterbi v2
changed = status_physics_v1 != status_viterbi_v2
changed_frames = np.flatnonzero(changed)

records = []

if len(changed_frames):
    groups = np.cumsum(
        np.r_[True, np.diff(changed_frames) != 1]
    )

    for _, group in pd.DataFrame({
        "frame": changed_frames,
        "group": groups
    }).groupby("group"):

        idx = group["frame"].to_numpy()
        start, end = int(idx[0]), int(idx[-1])

        records.append({
            "start_frame": start,
            "end_frame": end,
            "video_time_s": start / FPS,
            "duration_s": len(idx) / FPS,
            "old_status": status_viterbi_v2[start],
            "new_status": status_physics_v1[start],
            "public": np.isin(idx, oof_frames).any()
        })

df_physics_changes = pd.DataFrame(records)

print("Frames mit physikalischer Live-Evidenz:",
      int(physical_live.sum()))
print("Geänderte Statusframes:", len(changed_frames))
print("Ungeklärte Live-Frames:",
      physics_diag["unresolved_X"])

print("\nVeränderte Intervalle:")
display(df_physics_changes.round(2))

# 8. Offizielle Evaluation
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import subprocess
import sys
from pathlib import Path

OUT = PROJECT_DIR / "output/hierarchical_physics_v1"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

models = {
    "hier_v2_reference": hier_v2,
    "hier_physics_v1": hier_physics_v1
}

for name, labels in models.items():
    assert len(labels) == N_FRAMES
    assert np.isin(labels, ["H", "A", "D", "X"]).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(SUB / f"{name}.csv", index=False)

ARCHIVE = (
    PROJECT_DIR /
    "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:
    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root_name

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:
        subprocess.run(
            [
                sys.executable, str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable, str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\nOffizieller öffentlicher Vergleich:")
display(
    scores[
        [
            "submission", "macro_f1",
            "spell_f1_mean", "f1_H",
            "f1_A", "f1_D", "poss"
        ]
    ]
    .sort_values("spell_f1_mean", ascending=False)
    .round(4)
)


In [ ]:

# ============================================================
# SCHRITT 38: VIDEO-AUDIT DER NEUEN PHYSICS-KORREKTUREN
# ============================================================

import numpy as np
import pandas as pd

audit_starts = [
    164019,  # Größte Korrektur
    132924,  # Lange Korrektur
    21884,   # Ungeprüfte Korrektur
    147953,  # Ungeprüfte Korrektur
    53142,   # Bereits bekanntes Foul
]

audit = df_physics_changes.loc[
    df_physics_changes["start_frame"].isin(audit_starts)
].copy()

assert len(audit) == len(audit_starts), (
    "Nicht alle Audit-Intervalle gefunden."
)

audit["video_start"] = audit["start_frame"].apply(
    lambda f: str(pd.to_timedelta(f / FPS, unit="s"))
)

audit["video_end"] = audit["end_frame"].apply(
    lambda f: str(pd.to_timedelta(f / FPS, unit="s"))
)

moving = movement_features["moving_fraction"].to_numpy()
ball_speed = movement_features["ball_speed_mps"].to_numpy()
coverage = movement_features["ball_coverage"].to_numpy()

def median_valid(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

records = []

for row in audit.itertuples():

    start = int(row.start_frame)
    end = int(row.end_frame)

    idx = np.arange(start, end + 1)

    records.append({
        "start_frame": start,
        "video_start": row.video_start,
        "video_end": row.video_end,
        "duration_s": row.duration_s,
        "moving_median": median_valid(moving[idx]),
        "ball_speed_median": median_valid(ball_speed[idx]),
        "ball_coverage": median_valid(coverage[idx]),
        "heur_D_share": float(
            np.mean(base_labels[idx] == "D")
        ),
        "ada_D_share": float(
            np.mean(ada_labels[idx] == "D")
        ),
        "contacts": int(
            np.isin(
                contact_candidate_side[idx], ["H", "A"]
            ).sum()
        )
    })

audit_table = pd.DataFrame(records)

print("Physikalische Merkmale der Korrekturintervalle:")
display(audit_table.round(3).sort_values("start_frame"))

print("\nZum Abspielen: jeweils ca. 5 s vor video_start beginnen.")


In [ ]:

# ============================================================
# SCHRITT 39: VERBLINDETE VIDEO-STICHPROBE
# ============================================================

import numpy as np
import pandas as pd

rng = np.random.default_rng(2027)

# Bereits kontrollierte Szenen ausschließen
reviewed_starts = {
    19772, 31170, 42853, 53142, 54785,
    67524, 83578, 86237, 136650,
    142570, 143920, 151893,
    164019, 132924, 21884, 147953,
    18026, 69845
}

public_mask = np.zeros(N_FRAMES, dtype=bool)
public_mask[oof["frame"].to_numpy(dtype=int)] = True

# 1. Weitere korrigierte Intervalle
corrected = df_physics_changes[
    ~df_physics_changes["start_frame"].isin(
        reviewed_starts
    )
].copy()

corrected = corrected[
    ~corrected.apply(
        lambda r: public_mask[
            int(r.start_frame):int(r.end_frame)+1
        ].any(),
        axis=1
    )
]

# 2. Unveränderte D-Intervalle als Kontrollfälle
active = np.zeros(N_FRAMES, dtype=bool)
for _, first, last in HALVES:
    active[first:last+1] = True

all_d = get_intervals(
    np.arange(N_FRAMES),
    np.where(
        active & (status_viterbi_v2 == "D"),
        "D", "L"
    )
)

changed_mask = (
    status_physics_v1 != status_viterbi_v2
)

controls = []

for start, end in all_d:

    idx = slice(start, end+1)
    duration = (end-start+1)/FPS

    if not 2 <= duration <= 45:
        continue

    if public_mask[idx].any() or changed_mask[idx].any():
        continue

    if any(
        abs(start - old) < 10*FPS
        for old in reviewed_starts
    ):
        continue

    controls.append({
        "start_frame": start,
        "end_frame": end,
        "duration_s": duration
    })

controls = pd.DataFrame(controls)

def sample_rows(df, n):
    if df.empty:
        return df.copy()

    idx = rng.choice(
        len(df),
        size=min(n, len(df)),
        replace=False
    )
    return df.iloc[idx].copy()

# 3. Unterschiedliche Intervalllängen berücksichtigen
corrected_long = corrected[
    corrected["duration_s"] >= 2
]
corrected_short = corrected[
    (corrected["duration_s"] >= 0.3)
    & (corrected["duration_s"] < 2)
]

control_long = controls[
    controls["duration_s"] >= 10
]
control_short = controls[
    controls["duration_s"] < 10
]

parts = []

for df, n, kind in [
    (corrected_long, 4, "Korrektur"),
    (corrected_short, 2, "Korrektur"),
    (control_long, 2, "Kontrolle"),
    (control_short, 2, "Kontrolle")
]:
    chosen = sample_rows(df, n)

    if len(chosen):
        chosen["kind"] = kind
        parts.append(chosen)

audit = pd.concat(
    parts, ignore_index=True
).sample(frac=1, random_state=2027).reset_index(drop=True)

audit["audit_id"] = [
    f"S{i+1:02d}" for i in range(len(audit))
]

def video_time(frame):
    seconds = frame / FPS
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

audit["video_start"] = audit["start_frame"].apply(video_time)
audit["video_end"] = audit["end_frame"].apply(video_time)

# Vollständige interne Zuordnung lokal aufbewahren
audit_dir = (
    PROJECT_DIR / "output/hierarchical_physics_v1"
)

audit_dir.mkdir(parents=True, exist_ok=True)

audit.to_csv(
    audit_dir / "blinded_video_audit.csv",
    index=False
)

# Kategorie bei der Sichtung nicht anzeigen
print("Neue Szenen für die Videoüberprüfung:")

display(
    audit[
        [
            "audit_id",
            "video_start",
            "video_end",
            "duration_s"
        ]
    ].round(2)
)

print(
    "\nBitte jeweils ca. 5 Sekunden vorher starten "
    "und den tatsächlichen LIVE/D-Verlauf notieren."
)


In [ ]:

# ============================================================
# SCHRITT 40: KONTINUIERLICHE PHYSIKALISCHE LIVE-EVIDENZ
# ============================================================

import numpy as np
import pandas as pd
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

# ------------------------------------------------------------
# 1. Ausgangsdaten
# ------------------------------------------------------------

moving = movement_features[
    "moving_fraction"
].to_numpy(dtype=float)

ball_speed = movement_features[
    "ball_speed_mps"
].to_numpy(dtype=float)

ball_coverage = movement_features[
    "ball_coverage"
].to_numpy(dtype=float)

# Bereits in Schritt 37 berechnet
nearest = np.asarray(nearest_smooth, dtype=float)

heur_d = base_labels == "D"
ada_d = ada_labels == "D"
ada_valid = np.isin(ada_labels, ["D", "L"])

disagreement = ada_valid & (heur_d != ada_d)

active = np.zeros(N_FRAMES, dtype=bool)

for _, start, end in HALVES:
    active[start:end + 1] = True

# ------------------------------------------------------------
# 2. Kontinuierliche Einzelevidenz
# ------------------------------------------------------------

def sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))

# Ausgangspunkte entsprechen Physics v1.
# Die Breiten steuern weiche statt harte Übergänge.
#
# Diese Werte sind nicht auf den öffentlichen
# Ground-Truth-Frames optimiert.

MOVE_CENTER = 0.40
MOVE_WIDTH = 0.12

BALL_CENTER = 6.0
BALL_WIDTH = 2.0

DIST_CENTER = 3.5
DIST_WIDTH = 0.8

COVERAGE_REFERENCE = 0.50

PHYSICS_WEIGHT = 1.5

valid_physics = (
    active
    & disagreement
    & np.isfinite(moving)
    & np.isfinite(ball_speed)
    & np.isfinite(nearest)
    & np.isfinite(ball_coverage)
)

# Fehlende Beobachtungen geben keine Live-Evidenz.
live_strength = np.zeros(N_FRAMES, dtype=float)

idx = np.flatnonzero(valid_physics)

if len(idx):

    # Anteil schneller Spieler
    move_signal = sigmoid(
        (moving[idx] - MOVE_CENTER) / MOVE_WIDTH
    )

    # Ballbewegung
    ball_signal = sigmoid(
        (ball_speed[idx] - BALL_CENTER) / BALL_WIDTH
    )

    # Nähe eines Spielers zum Ball
    distance_signal = sigmoid(
        (DIST_CENTER - nearest[idx]) / DIST_WIDTH
    )

    # Verfügbarkeit der Ballbeobachtung
    coverage_signal = np.clip(
        ball_coverage[idx] / COVERAGE_REFERENCE,
        0.0,
        1.0
    )

    # Konservative Kombination:
    # Das schwächste Signal begrenzt die Live-Evidenz.
    live_strength[idx] = np.minimum.reduce([
        move_signal,
        ball_signal,
        distance_signal,
        coverage_signal
    ])

# ------------------------------------------------------------
# 3. Live-Evidenz in Viterbi integrieren
# ------------------------------------------------------------

physics_v2_logit = (
    base_logit
    - PHYSICS_WEIGHT * live_strength
)

status_physics_v2 = viterbi_status(
    physics_v2_logit / FPS,
    switch_penalty=1.0
)

# ------------------------------------------------------------
# 4. Ballbesitz innerhalb von LIVE-Phasen
# ------------------------------------------------------------

hier_physics_v2, diag_v2 = reconstruct_possession(
    status_physics_v2,
    base_labels,
    contact_labels
)

# ------------------------------------------------------------
# 5. Diagnostik
# ------------------------------------------------------------

public_idx = oof["frame"].to_numpy(dtype=int)

changed = (
    status_physics_v2 != status_viterbi_v2
)

changed_vs_v1 = (
    status_physics_v2 != status_physics_v1
)

print("Kontinuierliche Live-Evidenz:")
print(
    "Frames mit verfügbarer physikalischer Evidenz:",
    int(valid_physics.sum())
)
print(
    "Frames mit Evidenzstärke > 0.5:",
    int((live_strength > 0.5).sum())
)
print(
    "Mittlere Evidenzstärke (gültige Frames):",
    round(float(live_strength[valid_physics].mean()), 4)
    if valid_physics.any() else np.nan
)

print("\nVeränderte Statusframes:")
print("Gegenüber Viterbi v2:", int(changed.sum()))
print("Gegenüber Physics v1:", int(changed_vs_v1.sum()))
print(
    "Öffentlich gegenüber Viterbi v2:",
    int(changed[public_idx].sum())
)
print(
    "Ungeklärte LIVE-Frames:",
    diag_v2["unresolved_X"]
)

# Zusammenhängende Änderungen gegenüber Physics v1
frames_changed = np.flatnonzero(changed_vs_v1)

records = []

if len(frames_changed):

    boundaries = np.flatnonzero(
        np.diff(frames_changed) > 1
    ) + 1

    for group in np.split(frames_changed, boundaries):

        start = int(group[0])
        end = int(group[-1])

        records.append({
            "start_frame": start,
            "end_frame": end,
            "video_time_s": start / FPS,
            "duration_s": len(group) / FPS,
            "physics_v1": status_physics_v1[start],
            "physics_v2": status_physics_v2[start],
            "mean_live_strength": float(
                live_strength[group].mean()
            ),
            "public": bool(
                np.isin(group, public_idx).any()
            )
        })

df_physics_v2_changes = pd.DataFrame(records)

print("\nGrößte Unterschiede zu Physics v1:")

if len(df_physics_v2_changes):
    display(
        df_physics_v2_changes.nlargest(
            15, "duration_s"
        ).round(3)
    )
else:
    print("Keine Änderungen.")

# ------------------------------------------------------------
# 6. Submission und offizielle Evaluation
# ------------------------------------------------------------

OUT = PROJECT_DIR / "output/hierarchical_physics_v2"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

models = {
    "hier_v2_reference": hier_v2,
    "hier_physics_v1": hier_physics_v1,
    "hier_physics_v2_continuous": hier_physics_v2
}

for name, labels in models.items():

    assert len(labels) == N_FRAMES
    assert np.isin(
        labels, ["H", "A", "D", "X"]
    ).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUB / f"{name}.csv",
        index=False
    )

ARCHIVE = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root_name

    validator = (
        official / "evaluation/validate_submission.py"
    )

    evaluator = (
        official / "evaluation/run_evaluation.py"
    )

    for name in models:

        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\nOffizieller öffentlicher Vergleich:")

display(
    scores[
        [
            "submission",
            "macro_f1",
            "spell_f1_mean",
            "f1_H",
            "f1_A",
            "f1_D",
            "poss"
        ]
    ]
    .sort_values(
        "spell_f1_mean",
        ascending=False
    )
    .round(4)
)


In [ ]:

# ============================================================
# SCHRITT 41: PHYSICS V1 + NO-BALL LIVE-EVIDENCE
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import subprocess
import sys

moving = movement_features[
    "moving_fraction"
].to_numpy(dtype=float)

coverage = movement_features[
    "ball_coverage"
].to_numpy(dtype=float)

active = np.zeros(N_FRAMES, dtype=bool)
for _, start, end in HALVES:
    active[start:end + 1] = True

# ------------------------------------------------------------
# 1. Ergänzende Evidenz bei fehlendem Balltracking
# ------------------------------------------------------------

# Bewusst konservative Testbedingungen:
# - Heuristik meldet D
# - AdaBoost meldet LIVE
# - Balltracking weitgehend nicht vorhanden
# - deutliche kollektive Spielerbewegung
#
# Keine GT-basierten Sonderregeln für einzelne Szenen.

no_ball_live = (
    active
    & (base_labels == "D")
    & (ada_labels == "L")
    & np.isfinite(moving)
    & np.isfinite(coverage)
    & (coverage <= 0.20)
    & (moving >= 0.60)
)

# Physikalische Evidenz aus Physics v1 unverändert
# übernehmen und nur im No-Ball-Regime ergänzen.
NO_BALL_WEIGHT = 1.5

combined_logit = base_logit.copy()

combined_logit[physical_live] -= PHYSICS_WEIGHT
combined_logit[no_ball_live] -= NO_BALL_WEIGHT

# ------------------------------------------------------------
# 2. Zeitlichen Decoder neu ausführen
# ------------------------------------------------------------

status_physics_v3 = viterbi_status(
    combined_logit / FPS,
    switch_penalty=1.0
)

hier_physics_v3, diag_v3 = reconstruct_possession(
    status_physics_v3,
    base_labels,
    contact_labels
)

# Sicherstellen, dass Physics v1 reproduziert wird
assert np.array_equal(
    viterbi_status(
        (base_logit - PHYSICS_WEIGHT
         * physical_live.astype(float)) / FPS,
        switch_penalty=1.0
    ),
    status_physics_v1
)

# ------------------------------------------------------------
# 3. Unterschiede zu Physics v1
# ------------------------------------------------------------

changed = status_physics_v3 != status_physics_v1
changed_frames = np.flatnonzero(changed)

records = []

if len(changed_frames):
    boundaries = np.flatnonzero(
        np.diff(changed_frames) > 1
    ) + 1

    for group in np.split(changed_frames, boundaries):
        start = int(group[0])
        end = int(group[-1])

        records.append({
            "start_frame": start,
            "end_frame": end,
            "video_time_s": start / FPS,
            "duration_s": len(group) / FPS,
            "v1_status": status_physics_v1[start],
            "v3_status": status_physics_v3[start],
            "mean_moving": float(
                np.nanmean(moving[group])
            ),
            "mean_ball_coverage": float(
                np.nanmean(coverage[group])
            ),
            "public": bool(
                np.isin(group, oof_frames).any()
            )
        })

df_physics_v3_changes = pd.DataFrame(records)

print("No-Ball-Live-Evidenz:")
print("Frames mit zusätzlicher Evidenz:", int(no_ball_live.sum()))
print("Geänderte Statusframes:", int(changed.sum()))
print("Ungeklärte Live-Frames:", diag_v3["unresolved_X"])

print("\nVeränderte Intervalle:")
display(df_physics_v3_changes.round(3))

# ------------------------------------------------------------
# 4. Offizielle Evaluation
# ------------------------------------------------------------

OUT = PROJECT_DIR / "output/hierarchical_physics_v3"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

models = {
    "physics_v1_reference": hier_physics_v1,
    "physics_v3_no_ball": hier_physics_v3
}

for name, labels in models.items():
    assert len(labels) == N_FRAMES
    assert np.isin(labels, ["H", "A", "D", "X"]).all()

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUB / f"{name}.csv",
        index=False
    )

ARCHIVE = (
    PROJECT_DIR /
    "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root_name

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:
        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\nOffizieller öffentlicher Score:")
display(
    scores[
        [
            "submission",
            "macro_f1",
            "spell_f1_mean",
            "f1_H",
            "f1_A",
            "f1_D",
            "poss"
        ]
    ]
    .sort_values("spell_f1_mean", ascending=False)
    .round(4)
)


In [ ]:

# ============================================================
# SCHRITT 42: AUDIT DER NO-BALL-EVIDENZ UND VITERBI-AUSBREITUNG
# ============================================================

import numpy as np
import pandas as pd

changed = status_physics_v3 != status_physics_v1

assert np.all(
    status_physics_v3[changed] == "L"
), "Es wurden auch LIVE->D-Änderungen gefunden."

# 1. Unmittelbar durch No-Ball-Evidenz gestützte Frames
direct = changed & no_ball_live
propagated = changed & ~no_ball_live

print("Änderungen Physics v1 -> v3")
print("Gesamte Änderungen:      ", int(changed.sum()))
print("Direkte No-Ball-Evidenz:  ", int(direct.sum()))
print("Ohne lokale No-Ball-Evidenz:", int(propagated.sum()))

print(
    "Anteil direkt gestützt:",
    f"{100 * direct.sum() / max(changed.sum(), 1):.1f}%"
)

# 2. Analyse pro Änderungsintervall
rows = []

for row in df_physics_v3_changes.itertuples():

    start = int(row.start_frame)
    end = int(row.end_frame)

    idx = np.arange(start, end + 1)

    n_direct = int(no_ball_live[idx].sum())

    rows.append({
        "start_frame": start,
        "video_time": str(
            pd.to_timedelta(start / FPS, unit="s")
        ),
        "duration_s": len(idx) / FPS,
        "direct_frames": n_direct,
        "propagated_frames": len(idx) - n_direct,
        "direct_pct": 100 * n_direct / len(idx),
        "moving_median": float(
            np.nanmedian(moving[idx])
        ),
        "coverage_median": float(
            np.nanmedian(coverage[idx])
        ),
        "heur_D_pct": 100 * np.mean(
            base_labels[idx] == "D"
        ),
        "ada_L_pct": 100 * np.mean(
            ada_labels[idx] == "L"
        ),
        "public": bool(
            np.isin(idx, oof_frames).any()
        )
    })

df_no_ball_audit = pd.DataFrame(rows)

print("\nEvidenz pro Änderungsintervall:")

display(
    df_no_ball_audit.sort_values(
        ["propagated_frames", "duration_s"],
        ascending=False
    ).round(2)
)

# 3. Öffentlich: Wurden wirklich nur LIVE-Frames korrigiert?
public_frames = oof["frame"].to_numpy(dtype=int)
public_changed = changed[public_frames]

gt_changed = oof["GT"].to_numpy()[public_changed]

print("\nGround Truth der öffentlich korrigierten Frames:")

display(
    pd.Series(gt_changed, name="GT")
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Frames")
)

# 4. Diagnostik speichern
audit_path = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "no_ball_evidence_audit.csv"
)

audit_path.parent.mkdir(parents=True, exist_ok=True)

df_no_ball_audit.to_csv(
    audit_path, index=False
)

print("\nAudit gespeichert:", audit_path)


In [ ]:

# ============================================================
# SCHRITT 43: SENSITIVITÄT DER NO-BALL-EVIDENZ
# ============================================================

import numpy as np
import pandas as pd

WEIGHTS = [1.2, 1.5, 1.8]

variants = {}

# Ballgestützte Physics-v1-Evidenz bleibt unverändert.
# Ausschließlich das No-Ball-Gewicht wird variiert.
for weight in WEIGHTS:

    logits = (
        base_logit.copy()
        - 1.5 * physical_live.astype(float)
        - weight * no_ball_live.astype(float)
    )

    variants[weight] = viterbi_status(
        logits / FPS,
        switch_penalty=1.0
    )

# Reproduzierbarkeit der bestehenden Version
assert np.array_equal(
    variants[1.5],
    status_physics_v3
), "Physics v3 wird nicht exakt reproduziert."

# ------------------------------------------------------------
# 1. Globale Stabilität
# ------------------------------------------------------------

base = status_physics_v1

change_masks = {
    weight: variants[weight] != base
    for weight in WEIGHTS
}

stable_all = np.logical_and.reduce(
    [change_masks[w] for w in WEIGHTS]
)

current_changes = change_masks[1.5]

global_results = []

for weight in WEIGHTS:

    changed = change_masks[weight]

    global_results.append({
        "weight": weight,
        "changed_vs_v1": int(changed.sum()),
        "changed_vs_v3": int(
            np.count_nonzero(
                variants[weight] != status_physics_v3
            )
        ),
        "new_D_to_L": int(np.sum(
            (base == "D")
            & (variants[weight] == "L")
        )),
        "new_L_to_D": int(np.sum(
            (base == "L")
            & (variants[weight] == "D")
        ))
    })

print("Globale Sensitivität:")
display(pd.DataFrame(global_results))

print(
    "\nAnteil aktueller v3-Korrekturen, "
    "die bei allen drei Gewichten bestehen bleiben:"
)

print(
    f"{100 * np.sum(stable_all & current_changes) / max(current_changes.sum(), 1):.2f}%"
)

# ------------------------------------------------------------
# 2. Robustheit der bestehenden 16 Korrekturintervalle
# ------------------------------------------------------------

records = []

for row in df_physics_v3_changes.itertuples():

    start = int(row.start_frame)
    end = int(row.end_frame)

    idx = np.arange(start, end + 1)

    # Anteil des v3-Korrekturintervalls, der auch
    # bei schwächerem / stärkerem Gewicht LIVE wird.
    low_live = np.mean(
        variants[1.2][idx] == "L"
    )

    current_live = np.mean(
        variants[1.5][idx] == "L"
    )

    high_live = np.mean(
        variants[1.8][idx] == "L"
    )

    direct_share = np.mean(
        no_ball_live[idx]
    )

    records.append({
        "start_frame": start,
        "video_time_s": start / FPS,
        "duration_s": len(idx) / FPS,
        "direct_evidence_pct": 100 * direct_share,
        "live_pct_weight_1_2": 100 * low_live,
        "live_pct_weight_1_5": 100 * current_live,
        "live_pct_weight_1_8": 100 * high_live,
        "fully_stable": bool(
            np.all(stable_all[idx])
        )
    })

df_physics_v3_stability = pd.DataFrame(records)

print("\nRobustheit der 16 Korrekturintervalle:")

display(
    df_physics_v3_stability
    .sort_values(
        "live_pct_weight_1_2",
        ascending=True
    )
    .round(2)
)

# ------------------------------------------------------------
# 3. Zusätzliche Änderungen beim höheren Gewicht
# ------------------------------------------------------------

additional_high = (
    change_masks[1.8]
    & ~change_masks[1.5]
)

frames = np.flatnonzero(additional_high)

extra_records = []

if len(frames):

    boundaries = np.flatnonzero(
        np.diff(frames) > 1
    ) + 1

    for group in np.split(frames, boundaries):

        start = int(group[0])
        end = int(group[-1])

        extra_records.append({
            "start_frame": start,
            "end_frame": end,
            "video_time_s": start / FPS,
            "duration_s": len(group) / FPS
        })

print("\nZusätzliche Korrekturen bei Gewicht 1.8:")

display(
    pd.DataFrame(extra_records)
    .sort_values(
        "duration_s",
        ascending=False
    )
    .head(20)
    .round(2)
    if extra_records
    else pd.DataFrame()
)

# ------------------------------------------------------------
# 4. Speichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
)

OUT.mkdir(parents=True, exist_ok=True)

df_physics_v3_stability.to_csv(
    OUT / "no_ball_weight_stability.csv",
    index=False
)

print("\nSensitivitätsanalyse abgeschlossen.")


In [ ]:

# ============================================================
# SCHRITT 44: H/A-SPELLS UND URSACHEN DER FRAGMENTIERUNG
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Öffentlichen Bereich vorbereiten
# ------------------------------------------------------------

public = oof.sort_values("frame").reset_index(drop=True)

frames = public["frame"].to_numpy(dtype=int)
gt_status = public["GT"].to_numpy()

contacts = np.asarray(contact_candidate_side)

def video_time(frame):
    sec = frame / FPS
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    return f"{h:02d}:{m:02d}:{sec % 60:05.2f}"

def extract_runs(labels):
    labels = np.asarray(labels)

    boundaries = np.flatnonzero(
        (labels[1:] != labels[:-1])
        | (np.diff(frames) != 1)
    ) + 1

    groups = np.split(
        np.arange(len(frames)),
        boundaries
    )

    return [
        {
            "indices": group,
            "start_frame": int(frames[group[0]]),
            "end_frame": int(frames[group[-1]]),
            "label": str(labels[group[0]]),
            "duration_s": len(group) / FPS
        }
        for group in groups
    ]

# ------------------------------------------------------------
# 2. Vergleich der Besitzphasen
# ------------------------------------------------------------

models = {
    "Viterbi v2": hier_v2,
    "Physics v1": hier_physics_v1,
    "Physics v3": hier_physics_v3
}

overview = []

for name, full_labels in models.items():

    labels = np.asarray(full_labels)[frames]
    runs = extract_runs(labels)

    ha_runs = [
        r for r in runs
        if r["label"] in ("H", "A")
    ]

    d_runs = [
        r for r in runs
        if r["label"] == "D"
    ]

    direct_switches = 0
    restarts = 0

    for i, run in enumerate(runs):

        if run["label"] not in ("H", "A"):
            continue

        if i == 0:
            continue

        previous = runs[i - 1]["label"]

        if previous in ("H", "A"):
            direct_switches += 1

        elif previous == "D":
            restarts += 1

    overview.append({
        "Modell": name,
        "H/A-Runs": len(ha_runs),
        "Direkte H/A-Wechsel": direct_switches,
        "Neustarts nach D": restarts,
        "D-Intervalle": len(d_runs),
        "D-Frames bei GT=LIVE": int(np.sum(
            (labels == "D") & (gt_status == "L")
        ))
    })

print("Ballbesitzstruktur im öffentlichen Bereich:")
display(pd.DataFrame(overview))

# ------------------------------------------------------------
# 3. Physics-v3-Ballbesitzphasen im Detail
# ------------------------------------------------------------

pred = np.asarray(hier_physics_v3)[frames]
runs = extract_runs(pred)

ha_records = []

for i, run in enumerate(runs):

    side = run["label"]

    if side not in ("H", "A"):
        continue

    start = run["start_frame"]
    end = run["end_frame"]

    if i == 0:
        cause = "Ausschnittbeginn"
    else:
        previous = runs[i - 1]["label"]

        if previous == "D":
            cause = "Nach D"
        elif previous in ("H", "A"):
            cause = "Direkter H/A-Wechsel"
        else:
            cause = "Nach X"

    # Kontaktkandidaten im Fenster +/- 2 Sekunden
    lo = max(0, start - 2 * FPS)
    hi = min(N_FRAMES, start + 2 * FPS + 1)

    nearby = contacts[lo:hi]

    same_contact_frames = np.flatnonzero(
        nearby == side
    ) + lo

    opposite = "A" if side == "H" else "H"

    opposite_contact_frames = np.flatnonzero(
        nearby == opposite
    ) + lo

    if len(same_contact_frames):
        nearest_contact_s = (
            same_contact_frames[
                np.argmin(
                    np.abs(same_contact_frames - start)
                )
            ] - start
        ) / FPS
    else:
        nearest_contact_s = np.nan

    ha_records.append({
        "start_frame": start,
        "video_time": video_time(start),
        "team": side,
        "duration_s": run["duration_s"],
        "start_cause": cause,
        "matching_contacts_2s": len(
            same_contact_frames
        ),
        "opposite_contacts_2s": len(
            opposite_contact_frames
        ),
        "nearest_matching_contact_s": nearest_contact_s,
        "GT_D_frames": int(np.sum(
            gt_status[run["indices"]] == "D"
        ))
    })

df_ha_spells = pd.DataFrame(ha_records)

print("\nPhysics v3: Vorhergesagte H/A-Phasen:")
display(df_ha_spells.round(2))

# ------------------------------------------------------------
# 4. D-Intervalle: Welche unterbrechen echtes LIVE?
# ------------------------------------------------------------

d_records = []

for i, run in enumerate(runs):

    if run["label"] != "D":
        continue

    idx = run["indices"]

    true_d = int(np.sum(gt_status[idx] == "D"))
    true_live = int(np.sum(gt_status[idx] == "L"))

    previous_team = (
        runs[i - 1]["label"]
        if i > 0 else "-"
    )

    next_team = (
        runs[i + 1]["label"]
        if i + 1 < len(runs) else "-"
    )

    d_records.append({
        "start_frame": run["start_frame"],
        "video_time": video_time(run["start_frame"]),
        "duration_s": run["duration_s"],
        "GT_D_frames": true_d,
        "GT_L_frames": true_live,
        "false_D_pct": (
            100 * true_live / len(idx)
        ),
        "previous_team": previous_team,
        "next_team": next_team,
        "same_team_around_D": (
            previous_team == next_team
            and previous_team in ("H", "A")
        )
    })

df_d_splits = pd.DataFrame(d_records)

print("\nPhysics v3: D-Unterbrechungen:")
display(df_d_splits.round(2))

# ------------------------------------------------------------
# 5. Unmittelbare Teamwechsel ohne passenden Kontakt
# ------------------------------------------------------------

direct = df_ha_spells[
    df_ha_spells["start_cause"]
    == "Direkter H/A-Wechsel"
].copy()

unexplained = direct[
    direct["matching_contacts_2s"] == 0
]

print("\nDirekte H/A-Wechsel:", len(direct))
print(
    "Ohne passenden Kontakt innerhalb +/- 2 s:",
    len(unexplained)
)

display(unexplained.round(2))

# ------------------------------------------------------------
# 6. Ergebnisse speichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_ha_spells.to_csv(
    OUT / "ha_spells_public.csv",
    index=False
)

df_d_splits.to_csv(
    OUT / "d_splits_public.csv",
    index=False
)

print("\nDiagnostik gespeichert:", OUT)


In [ ]:

# ============================================================
# SCHRITT 45: GROUND-TRUTH-QUELLEN UND KURZE H/A-PHASEN
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from zipfile import ZipFile

# 1. Ground-Truth-Spalten im bestehenden OOF-Datensatz
print("=== OOF-SPALTEN ===")
print(oof.columns.tolist())

print("\n=== SPALTEN MIT WENIGEN KATEGORIEN ===")

for col in oof.columns:
    if not (
        pd.api.types.is_object_dtype(oof[col])
        or isinstance(oof[col].dtype, pd.CategoricalDtype)
    ):
        continue

    values = oof[col].dropna().astype(str).unique()

    if len(values) <= 12:
        print(f"{col}: {values.tolist()}")

# 2. Öffentliche Referenzdateien im offiziellen Archiv suchen
archive_path = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

print("\n=== MÖGLICHE GROUND-TRUTH-DATEIEN ===")

keywords = (
    "ground", "truth", "annot",
    "label", "public", "possession", "gt"
)

with ZipFile(archive_path) as archive:

    candidates = [
        info for info in archive.infolist()
        if not info.is_dir()
        and any(
            keyword in Path(info.filename).name.lower()
            for keyword in keywords
        )
        and info.filename.lower().endswith(
            (".csv", ".tsv", ".json", ".parquet")
        )
    ]

    print(f"Gefundene Kandidaten: {len(candidates)}")

    for info in candidates[:30]:
        print(
            f"\n{info.filename} "
            f"({info.file_size / 1024:.1f} KB)"
        )

        if info.filename.lower().endswith((".csv", ".tsv")):
            try:
                sep = (
                    "\t" if info.filename.endswith(".tsv")
                    else ","
                )

                with archive.open(info) as f:
                    preview = pd.read_csv(
                        f, sep=sep, nrows=3
                    )

                print("Spalten:", preview.columns.tolist())
                print(preview.to_string(index=False))

            except Exception as exc:
                print("Vorschau nicht lesbar:", str(exc)[:160])

# 3. Kurze vorhergesagte H/A-Spells
print("\n=== KURZE H/A-PHASEN (< 4 SEKUNDEN) ===")

short_spells = (
    df_ha_spells[
        df_ha_spells["duration_s"] < 4
    ]
    .sort_values("start_frame")
    .copy()
)

display(
    short_spells[
        [
            "start_frame",
            "video_time",
            "team",
            "duration_s",
            "start_cause",
            "matching_contacts_2s",
            "opposite_contacts_2s",
            "GT_D_frames"
        ]
    ].round(2)
)

print("\nKurze H/A-Phasen:", len(short_spells))
print(
    "Anteil aller H/A-Phasen:",
    f"{100 * len(short_spells) / len(df_ha_spells):.1f}%"
)

# 4. Besonderer Problemabschnitt
print("\n=== STATUS IM PROBLEMBEREICH ===")

for start, end in [
    (17097, 17143),
    (17144, 17228),
    (17229, 17259),
    (17260, 17379),
    (17380, 17837)
]:
    labels = hier_physics_v3[start:end + 1]

    counts = pd.Series(labels).value_counts().to_dict()

    print(
        f"{start}–{end}: "
        f"{(end-start+1)/FPS:.2f}s | {counts}"
    )


In [ ]:

# ============================================================
# SCHRITT 46: OFFIZIELLE H/A/D-GROUND-TRUTH-ANALYSE
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from zipfile import ZipFile

# ------------------------------------------------------------
# 1. Vollständige öffentliche H/A/D-Ground Truth laden
# ------------------------------------------------------------

archive_path = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with ZipFile(archive_path) as archive:
    gt_file = next(
        name for name in archive.namelist()
        if name.endswith("BAR-ATH_possession_gt_public.csv")
    )

    with archive.open(gt_file) as f:
        gt_full = pd.read_csv(f)

public = oof.sort_values("frame").reset_index(drop=True)
frames = public["frame"].to_numpy(dtype=int)

gt_lookup = (
    gt_full
    .drop_duplicates("frame")
    .set_index("frame")["label"]
)

gt = gt_lookup.reindex(frames).to_numpy()
pred = np.asarray(hier_physics_v3)[frames]

print("Offizielle Ground-Truth-Labels:")
print(pd.Series(gt).value_counts(dropna=False))

print("\nÜbereinstimmung mit bisherigem D/L-GT:")
status_gt = np.where(
    gt == "D", "D",
    np.where(np.isin(gt, ["H", "A"]), "L", "X")
)

valid_status = np.isin(status_gt, ["D", "L"])

print(
    "Abweichende Statusframes:",
    int(np.sum(
        status_gt[valid_status]
        != public["GT"].to_numpy()[valid_status]
    ))
)

# ------------------------------------------------------------
# 2. Konfusionsmatrix H/A/D
# ------------------------------------------------------------

valid = np.isin(gt, ["H", "A", "D"])

confusion = pd.crosstab(
    pd.Series(gt[valid], name="Ground Truth"),
    pd.Series(pred[valid], name="Vorhersage")
).reindex(
    index=["H", "A", "D"],
    columns=["H", "A", "D", "X"],
    fill_value=0
)

print("\nKonfusionsmatrix:")
display(confusion)

# ------------------------------------------------------------
# 3. Zusammenhängende Phasen extrahieren
# ------------------------------------------------------------

def extract_runs(labels, frame_ids):
    labels = np.asarray(labels)
    frame_ids = np.asarray(frame_ids)

    boundaries = np.flatnonzero(
        (labels[1:] != labels[:-1])
        | (np.diff(frame_ids) != 1)
    ) + 1

    groups = np.split(
        np.arange(len(labels)),
        boundaries
    )

    runs = []

    for group in groups:
        runs.append({
            "start": int(frame_ids[group[0]]),
            "end": int(frame_ids[group[-1]]),
            "label": str(labels[group[0]]),
            "duration_s": len(group) / FPS,
            "indices": group
        })

    return runs

gt_runs = extract_runs(gt, frames)
pred_runs = extract_runs(pred, frames)

def count_ha(runs):
    return sum(r["label"] in ("H", "A") for r in runs)

print("\nH/A-Phasen:")
print("Ground Truth:", count_ha(gt_runs))
print("Physics v3:", count_ha(pred_runs))

# ------------------------------------------------------------
# 4. Kurze vorhergesagte Besitzphasen bewerten
# ------------------------------------------------------------

short_records = []

for run in pred_runs:

    if run["label"] not in ("H", "A"):
        continue

    if run["duration_s"] >= 4:
        continue

    actual = gt[run["indices"]]
    predicted_side = run["label"]

    counts = {
        label: int(np.sum(actual == label))
        for label in ["H", "A", "D", "X"]
    }

    short_records.append({
        "start_frame": run["start"],
        "duration_s": run["duration_s"],
        "predicted": predicted_side,
        "GT_H_pct": 100 * counts["H"] / len(actual),
        "GT_A_pct": 100 * counts["A"] / len(actual),
        "GT_D_pct": 100 * counts["D"] / len(actual),
        "correct_team_pct": (
            100 * counts[predicted_side] / len(actual)
        )
    })

df_short_gt = pd.DataFrame(short_records)

print("\nKurze H/A-Phasen gegen Ground Truth:")
display(df_short_gt.round(1))

# ------------------------------------------------------------
# 5. Direkte H/A-Wechsel identifizieren
# ------------------------------------------------------------

def direct_switches(runs):
    result = []

    for i in range(1, len(runs)):
        previous = runs[i - 1]
        current = runs[i]

        if (
            previous["label"] in ("H", "A")
            and current["label"] in ("H", "A")
            and previous["label"] != current["label"]
        ):
            result.append({
                "frame": current["start"],
                "from": previous["label"],
                "to": current["label"]
            })

    return result

gt_switches = direct_switches(gt_runs)
pred_switches = direct_switches(pred_runs)

print("\nDirekte H/A-Wechsel:")
print("Ground Truth:", len(gt_switches))
print("Physics v3:", len(pred_switches))

# ------------------------------------------------------------
# 6. Wechsel mit +/- 2 Sekunden Toleranz zuordnen
#    Eins-zu-eins-Matching, keine Doppelzählung
# ------------------------------------------------------------

TOLERANCE_FRAMES = 2 * FPS

candidates = []

for i, p in enumerate(pred_switches):
    for j, g in enumerate(gt_switches):

        if p["from"] != g["from"] or p["to"] != g["to"]:
            continue

        delta = p["frame"] - g["frame"]

        if abs(delta) <= TOLERANCE_FRAMES:
            candidates.append((abs(delta), i, j, delta))

used_pred = set()
used_gt = set()
matches = {}

for _, i, j, delta in sorted(candidates):
    if i in used_pred or j in used_gt:
        continue

    used_pred.add(i)
    used_gt.add(j)
    matches[i] = (j, delta)

rows = []

for i, p in enumerate(pred_switches):

    match = matches.get(i)

    rows.append({
        "start_frame": p["frame"],
        "from": p["from"],
        "to": p["to"],
        "matched_2s": match is not None,
        "timing_error_s": (
            match[1] / FPS
            if match is not None else np.nan
        ),
        "nearest_GT_same_direction_s": (
            min(
                [
                    (p["frame"] - g["frame"]) / FPS
                    for g in gt_switches
                    if g["from"] == p["from"]
                    and g["to"] == p["to"]
                ],
                key=abs,
                default=np.nan
            )
        )
    })

df_switch_audit = pd.DataFrame(rows)

print("\nAbgleich der vorhergesagten direkten H/A-Wechsel:")
display(df_switch_audit.round(2))

print("\nWechsel-Matching (+/- 2 Sekunden):")
print("Zugeordnet:", len(matches))
print("Vorhergesagt ohne Zuordnung:", len(pred_switches) - len(matches))
print("GT-Wechsel ohne Zuordnung:", len(gt_switches) - len(matches))

# ------------------------------------------------------------
# 7. Falsche D-Phase: Tatsächliches Team?
# ------------------------------------------------------------

problem = (
    (frames >= 17097)
    & (frames <= 17379)
)

print("\nGround Truth im Problemabschnitt 17097–17379:")

display(
    pd.crosstab(
        pd.Series(
            gt[problem],
            name="Ground Truth"
        ),
        pd.Series(
            pred[problem],
            name="Physics v3"
        )
    )
)

# ------------------------------------------------------------
# 8. Diagnostik speichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_short_gt.to_csv(
    OUT / "short_spells_vs_gt.csv",
    index=False
)

df_switch_audit.to_csv(
    OUT / "direct_switches_vs_gt.csv",
    index=False
)

print("\nDiagnostik gespeichert:", OUT)


In [ ]:

# ============================================================
# SCHRITT 47: ORACLE-STATUS UND H/A-FEHLERZERLEGUNG
# ============================================================

import numpy as np
import pandas as pd

frames = oof.sort_values("frame")["frame"].to_numpy(dtype=int)
gt = gt_lookup.reindex(frames).to_numpy()

pred_v3 = np.asarray(hier_physics_v3)[frames]

assert np.isin(gt, ["H", "A", "D"]).all()
assert len(frames) == 7500

# ------------------------------------------------------------
# 1. Diagnostische Variante: Status durch GT ersetzen
# ------------------------------------------------------------

oracle_status = status_physics_v3.copy()

oracle_status[frames] = np.where(
    gt == "D", "D", "L"
)

oracle_full, oracle_diag = reconstruct_possession(
    oracle_status,
    base_labels,
    contact_labels
)

pred_oracle = np.asarray(oracle_full)[frames]

# Keine Speicherung als Submission:
# Diese Variante verwendet ausdrücklich Ground Truth.

# ------------------------------------------------------------
# 2. Fehler nach Ursachen aufschlüsseln
# ------------------------------------------------------------

def summarize_errors(name, pred):
    true_live = np.isin(gt, ["H", "A"])
    true_dead = gt == "D"

    return {
        "Modell": name,
        "LIVE korrektes Team": int(np.sum(
            true_live & (pred == gt)
        )),
        "LIVE falsches Team": int(np.sum(
            true_live
            & np.isin(pred, ["H", "A"])
            & (pred != gt)
        )),
        "LIVE fälschlich D": int(np.sum(
            true_live & (pred == "D")
        )),
        "LIVE ungeklärt X": int(np.sum(
            true_live & (pred == "X")
        )),
        "D korrekt": int(np.sum(
            true_dead & (pred == "D")
        )),
        "D fälschlich LIVE": int(np.sum(
            true_dead & np.isin(pred, ["H", "A"])
        ))
    }

summary = pd.DataFrame([
    summarize_errors("Physics v3", pred_v3),
    summarize_errors("Oracle D/L (nur Diagnose)", pred_oracle)
])

print("=== FEHLERZERLEGUNG ===")
display(summary)

# ------------------------------------------------------------
# 3. Welche H/A-Fehler werden durch richtigen Status gelöst?
# ------------------------------------------------------------

true_live = np.isin(gt, ["H", "A"])

v3_wrong_team = (
    true_live
    & np.isin(pred_v3, ["H", "A"])
    & (pred_v3 != gt)
)

oracle_wrong_team = (
    true_live
    & np.isin(pred_oracle, ["H", "A"])
    & (pred_oracle != gt)
)

fixed_team = (
    v3_wrong_team & (pred_oracle == gt)
)

new_team_errors = (
    ~v3_wrong_team
    & oracle_wrong_team
)

print("\n=== AUSWIRKUNG PERFEKTER D/L-ERKENNUNG ===")

print("Bisher falsches Team:", int(v3_wrong_team.sum()))
print("Davon durch Oracle korrigiert:", int(fixed_team.sum()))
print("Neue falsche H/A-Zuordnungen:", int(new_team_errors.sum()))
print("Verbleibende falsche H/A-Frames:",
      int(oracle_wrong_team.sum()))

# ------------------------------------------------------------
# 4. Struktur der vorhergesagten Besitzphasen
# ------------------------------------------------------------

def run_statistics(labels):
    runs = extract_runs(labels, frames)

    ha = [
        r for r in runs
        if r["label"] in ("H", "A")
    ]

    direct = sum(
        runs[i - 1]["label"] in ("H", "A")
        and runs[i]["label"] in ("H", "A")
        and runs[i - 1]["label"] != runs[i]["label"]
        for i in range(1, len(runs))
    )

    return {
        "H/A-Phasen": len(ha),
        "Direkte H/A-Wechsel": direct,
        "D-Intervalle": sum(
            r["label"] == "D" for r in runs
        )
    }

print("\n=== BALLBESITZSTRUKTUR ===")

display(pd.DataFrame([
    {"Modell": "Ground Truth", **run_statistics(gt)},
    {"Modell": "Physics v3", **run_statistics(pred_v3)},
    {
        "Modell": "Oracle D/L",
        **run_statistics(pred_oracle)
    }
]))

# ------------------------------------------------------------
# 5. Größte echte H/A-Verwechslungen finden
# ------------------------------------------------------------

wrong_idx = np.flatnonzero(v3_wrong_team)
episodes = []

if len(wrong_idx):

    boundaries = np.flatnonzero(
        (np.diff(frames[wrong_idx]) != 1)
        | (gt[wrong_idx[1:]] != gt[wrong_idx[:-1]])
        | (
            pred_v3[wrong_idx[1:]]
            != pred_v3[wrong_idx[:-1]]
        )
    ) + 1

    for group in np.split(wrong_idx, boundaries):

        start = int(frames[group[0]])
        end = int(frames[group[-1]])

        episodes.append({
            "start_frame": start,
            "end_frame": end,
            "video_time_s": round(start / FPS, 2),
            "duration_s": len(group) / FPS,
            "GT_team": gt[group[0]],
            "pred_team": pred_v3[group[0]],
            "oracle_correct_pct": (
                100 * np.mean(
                    pred_oracle[group] == gt[group]
                )
            )
        })

df_ha_errors = pd.DataFrame(episodes)

print("\n=== GRÖSSTE H/A-VERWECHSLUNGEN ===")

if len(df_ha_errors):
    display(
        df_ha_errors.nlargest(
            15, "duration_s"
        ).round(2)
    )
else:
    print("Keine direkten H/A-Verwechslungen.")

# ------------------------------------------------------------
# 6. Problemabschnitt 17097–17379
# ------------------------------------------------------------

mask = (frames >= 17097) & (frames <= 17379)

print("\n=== PROBLEMBEREICH 17097–17379 ===")

display(pd.DataFrame({
    "Ground Truth": gt[mask],
    "Physics v3": pred_v3[mask],
    "Oracle D/L": pred_oracle[mask]
}).value_counts().reset_index(name="Frames"))

print(
    "\nHinweis: Oracle D/L ist nur eine diagnostische "
    "Obergrenzenanalyse, kein gültiges Modell."
)


In [ ]:

# ============================================================
# SCHRITT 48: ZUVERLÄSSIGKEIT DER H/A-KONTAKTWECHSEL
# ============================================================

import numpy as np
import pandas as pd

frames = oof.sort_values("frame")["frame"].to_numpy(dtype=int)
gt = gt_lookup.reindex(frames).to_numpy()
pred = np.asarray(hier_physics_v3)[frames]
contacts = np.asarray(contact_candidate_side)

assert np.all(np.diff(frames) == 1)
assert np.isin(gt, ["H", "A", "D"]).all()

FIRST = int(frames[0])
LAST = int(frames[-1])

# ------------------------------------------------------------
# 1. Direkte vorhergesagte H/A-Wechsel
# ------------------------------------------------------------

switch_mask = (
    np.isin(pred[1:], ["H", "A"])
    & np.isin(pred[:-1], ["H", "A"])
    & (pred[1:] != pred[:-1])
)

switch_frames = set(
    frames[1:][switch_mask].tolist()
)

# ------------------------------------------------------------
# 2. Hilfsfunktionen
# ------------------------------------------------------------

def gt_mode(start, end):
    """Häufigstes H/A-Label, D wird ausgeschlossen."""
    start = max(FIRST, int(start))
    end = min(LAST + 1, int(end))

    if start >= end:
        return "-"

    values = gt[start - FIRST:end - FIRST]
    values = values[np.isin(values, ["H", "A"])]

    if not len(values):
        return "-"

    counts = pd.Series(values).value_counts()
    return str(counts.index[0])


def gt_team_share(start, end, side):
    """Anteil eines Teams unter tatsächlichen LIVE-Frames."""
    start = max(FIRST, int(start))
    end = min(LAST + 1, int(end))

    if start >= end:
        return np.nan

    values = gt[start - FIRST:end - FIRST]
    live = values[np.isin(values, ["H", "A"])]

    if not len(live):
        return np.nan

    return float(np.mean(live == side))


# ------------------------------------------------------------
# 3. Alle öffentlichen Kontaktkandidaten untersuchen
# ------------------------------------------------------------

contact_frames = frames[
    np.isin(contacts[frames], ["H", "A"])
]

records = []

for frame in contact_frames:

    side = str(contacts[frame])
    opposite = "A" if side == "H" else "H"

    # Nachfolgende Kontakte, aktuellen Kontakt ausschließen
    end3 = min(N_FRAMES, frame + 3 * FPS + 1)
    following = contacts[frame + 1:end3]

    next_same = int(np.sum(following == side))
    next_opposite = int(np.sum(following == opposite))

    future_events = np.flatnonzero(
        np.isin(following, ["H", "A"])
    )

    if len(future_events):
        next_contact_side = str(
            following[future_events[0]]
        )
        next_contact_delay_s = (
            future_events[0] + 1
        ) / FPS
    else:
        next_contact_side = "-"
        next_contact_delay_s = np.nan

    records.append({
        "frame": int(frame),
        "video_time_s": frame / FPS,
        "contact_side": side,
        "triggered_HA_switch": frame in switch_frames,
        "GT_at_contact": str(gt[frame - FIRST]),
        "GT_before_2s": gt_mode(
            frame - 2 * FPS, frame
        ),
        "GT_after_2s": gt_mode(
            frame, frame + 2 * FPS
        ),
        "GT_after_5s": gt_mode(
            frame, frame + 5 * FPS
        ),
        "GT_side_share_next2s": gt_team_share(
            frame, frame + 2 * FPS, side
        ),
        "same_contacts_next3s": next_same,
        "opposite_contacts_next3s": next_opposite,
        "next_contact_side": next_contact_side,
        "next_contact_delay_s": next_contact_delay_s
    })

df_contact_quality = pd.DataFrame(records)

# ------------------------------------------------------------
# 4. Direkt ausgelöste Wechsel gegen GT vergleichen
# ------------------------------------------------------------

switch_quality = df_contact_quality[
    df_contact_quality["triggered_HA_switch"]
].copy()

# Referenz aus Schritt 46: Wechselmatching +/- 2s
match_lookup = df_switch_audit.set_index(
    "start_frame"
)["matched_2s"]

switch_quality["GT_switch_matched_2s"] = (
    switch_quality["frame"]
    .map(match_lookup)
    .fillna(False)
    .astype(bool)
)

print("=== DIREKTE H/A-WECHSEL UND KONTAKTFOLGE ===")

display(
    switch_quality[
        [
            "frame",
            "contact_side",
            "GT_before_2s",
            "GT_at_contact",
            "GT_after_2s",
            "GT_after_5s",
            "GT_side_share_next2s",
            "same_contacts_next3s",
            "opposite_contacts_next3s",
            "next_contact_side",
            "next_contact_delay_s",
            "GT_switch_matched_2s"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 5. Vergleichen: Zugeordnete vs. nicht zugeordnete Wechsel
# ------------------------------------------------------------

print("\n=== ZUGEORDNETE / NICHT ZUGEORDNETE WECHSEL ===")

if len(switch_quality):

    summary = (
        switch_quality
        .groupby("GT_switch_matched_2s")
        .agg(
            n=("frame", "size"),
            median_GT_side_next2s=(
                "GT_side_share_next2s", "median"
            ),
            median_same_contacts_next3s=(
                "same_contacts_next3s", "median"
            ),
            median_opposite_contacts_next3s=(
                "opposite_contacts_next3s", "median"
            ),
            median_next_contact_delay_s=(
                "next_contact_delay_s", "median"
            )
        )
        .reset_index()
    )

    display(summary.round(3))

# ------------------------------------------------------------
# 6. Wie viele Ballkontakte führen überhaupt zu Wechseln?
# ------------------------------------------------------------

print("\n=== ALLE KONTAKTKANDIDATEN IM ÖFFENTLICHEN BEREICH ===")

print("Kontaktkandidaten:", len(df_contact_quality))
print("Davon mit direktem H/A-Wechsel:", len(switch_quality))

df_contact_quality["side_matches_GT"] = (
    df_contact_quality["contact_side"]
    == df_contact_quality["GT_at_contact"]
)

display(
    df_contact_quality
    .groupby("triggered_HA_switch")
    .agg(
        n=("frame", "size"),
        contact_matches_GT_pct=(
            "side_matches_GT",
            lambda x: 100 * x.mean()
        ),
        median_GT_side_next2s=(
            "GT_side_share_next2s", "median"
        ),
        median_same_contacts_next3s=(
            "same_contacts_next3s", "median"
        ),
        median_opposite_contacts_next3s=(
            "opposite_contacts_next3s", "median"
        )
    )
    .round(3)
)

# ------------------------------------------------------------
# 7. Diagnostik speichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_contact_quality.to_csv(
    OUT / "contact_event_quality_public.csv",
    index=False
)

switch_quality.to_csv(
    OUT / "contact_switch_quality_public.csv",
    index=False
)

print("\nKontaktaudit abgeschlossen.")


In [ ]:

# ============================================================
# SCHRITT 49: KONTAKTBASIERTER H/A-HMM
# ============================================================

import numpy as np
import pandas as pd
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

# ------------------------------------------------------------
# 1. Feste Modellparameter
# ------------------------------------------------------------

CONTACT_MISMATCH_COST = 1.4
TEAM_SWITCH_COST = 1.8
INITIAL_CHANGE_COST = 0.8

SIDES = np.array(["H", "A"])

contacts = np.asarray(contact_candidate_side)
status = np.asarray(status_physics_v3)
reference = np.asarray(hier_physics_v3)

hier_contact_hmm_v1 = reference.copy()

active = np.zeros(N_FRAMES, dtype=bool)
for _, start, end in HALVES:
    active[start:end+1] = True

live = active & (status == "L")

# Zusammenhängende LIVE-Abschnitte
boundaries = np.flatnonzero(
    live[1:] != live[:-1]
) + 1

segments = np.split(
    np.arange(N_FRAMES),
    boundaries
)

diagnostics = []

# ------------------------------------------------------------
# 2. HMM für jeden LIVE-Abschnitt
# ------------------------------------------------------------

for segment in segments:

    if not len(segment):
        continue

    start = int(segment[0])
    end = int(segment[-1]) + 1

    if not live[start]:
        continue

    local_contacts = contacts[start:end]

    event_offsets = np.flatnonzero(
        np.isin(local_contacts, SIDES)
    )

    if not len(event_offsets):
        continue

    event_frames = event_offsets + start
    observed_sides = contacts[event_frames]

    n = len(event_frames)

    # Emissionskosten: Kontakt mit gegnerischem Team
    emission = np.column_stack([
        (observed_sides != "H").astype(float),
        (observed_sides != "A").astype(float)
    ]) * CONTACT_MISMATCH_COST

    # Ausgangsteam aus bisheriger Rekonstruktion
    initial_side = reference[start]

    if initial_side not in SIDES:
        initial_side = observed_sides[0]

    dp = np.full((n, 2), np.inf)
    back = np.zeros((n, 2), dtype=np.int8)

    # Anfangszustand
    for state_idx, side in enumerate(SIDES):
        dp[0, state_idx] = (
            emission[0, state_idx]
            + (
                INITIAL_CHANGE_COST
                if side != initial_side else 0.0
            )
        )

    # Viterbi auf der Kontaktsequenz
    for t in range(1, n):

        for state_idx in range(2):

            options = (
                dp[t-1]
                + TEAM_SWITCH_COST
            )

            # Kein Wechsel -> keine Wechselstrafe
            options[state_idx] = dp[t-1, state_idx]

            previous = int(np.argmin(options))

            dp[t, state_idx] = (
                options[previous]
                + emission[t, state_idx]
            )

            back[t, state_idx] = previous

    # Rückverfolgung
    decoded = np.zeros(n, dtype=np.int8)
    decoded[-1] = int(np.argmin(dp[-1]))

    for t in range(n-2, -1, -1):
        decoded[t] = back[t+1, decoded[t+1]]

    decoded_sides = SIDES[decoded]

    # --------------------------------------------------------
    # 3. Dekodierte Kontakte auf Frames übertragen
    # --------------------------------------------------------

    # Bis zum ersten Kontakt bisherigen Zustand behalten.
    # Ab jedem Kontakt gilt der dekodierte Zustand bis zum
    # nächsten Kontakt beziehungsweise Ende der LIVE-Phase.
    for t, frame in enumerate(event_frames):

        next_frame = (
            int(event_frames[t+1])
            if t+1 < n else end
        )

        hier_contact_hmm_v1[
            frame:next_frame
        ] = decoded_sides[t]

    diagnostics.append({
        "start_frame": start,
        "end_frame": end-1,
        "duration_s": (end-start)/FPS,
        "contacts": n,
        "contact_observations_overruled": int(
            np.sum(decoded_sides != observed_sides)
        ),
        "decoded_event_switches": int(
            np.sum(decoded_sides[1:] != decoded_sides[:-1])
        )
    })

# ------------------------------------------------------------
# 4. Sicherheitsprüfungen
# ------------------------------------------------------------

assert len(hier_contact_hmm_v1) == N_FRAMES

# Spielstatus darf nicht verändert werden
assert np.array_equal(
    hier_contact_hmm_v1[~live],
    reference[~live]
)

assert np.array_equal(
    hier_contact_hmm_v1[status == "D"],
    reference[status == "D"]
)

assert np.isin(
    hier_contact_hmm_v1, ["H", "A", "D", "X"]
).all()

changed = hier_contact_hmm_v1 != reference

diag = pd.DataFrame(diagnostics)

print("=== H/A-HMM DIAGNOSTIK ===")
print("Veränderte Gesamtframes:", int(changed.sum()))
print("Veränderte öffentliche Frames:",
      int(changed[oof_frames].sum()))
print("Unveränderte D-Frames:",
      int(np.sum(status == "D")))

print(
    "Überstimmte Kontaktbeobachtungen:",
    int(diag["contact_observations_overruled"].sum())
)

print(
    "HMM-Teamwechsel an Kontakten:",
    int(diag["decoded_event_switches"].sum())
)

# ------------------------------------------------------------
# 5. Öffentliche H/A-Struktur
# ------------------------------------------------------------

frames = oof_frames
gt = gt_lookup.reindex(frames).to_numpy()

model_comparison = []

for name, labels in {
    "Physics v3": reference,
    "Kontakt-HMM v1": hier_contact_hmm_v1
}.items():

    pred = labels[frames]

    runs = extract_runs(pred, frames)

    direct_switches = sum(
        runs[i-1]["label"] in ("H", "A")
        and runs[i]["label"] in ("H", "A")
        and runs[i-1]["label"] != runs[i]["label"]
        for i in range(1, len(runs))
    )

    model_comparison.append({
        "model": name,
        "HA_runs": sum(
            r["label"] in ("H", "A") for r in runs
        ),
        "direct_HA_switches": direct_switches,
        "wrong_HA_frames": int(np.sum(
            np.isin(gt, ["H", "A"])
            & np.isin(pred, ["H", "A"])
            & (gt != pred)
        )),
        "correct_HA_frames": int(np.sum(
            np.isin(gt, ["H", "A"])
            & (gt == pred)
        ))
    })

print("\n=== BALLBESITZSTRUKTUR ===")
display(pd.DataFrame(model_comparison))

# ------------------------------------------------------------
# 6. Offizielle öffentliche Evaluation
# ------------------------------------------------------------

OUT = PROJECT_DIR / "output/hierarchical_contact_hmm_v1"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

models = {
    "physics_v3_reference": reference,
    "contact_hmm_v1": hier_contact_hmm_v1
}

for name, labels in models.items():

    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUB / f"{name}.csv",
        index=False
    )

ARCHIVE = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root_name

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:
        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\n=== OFFIZIELLER ÖFFENTLICHER SCORE ===")

display(
    scores[
        [
            "submission",
            "macro_f1",
            "spell_f1_mean",
            "f1_H",
            "f1_A",
            "f1_D",
            "poss"
        ]
    ].sort_values(
        "spell_f1_mean",
        ascending=False
    ).round(4)
)

diag.to_csv(
    OUT / "contact_hmm_diagnostics.csv",
    index=False
)


In [ ]:

# ============================================================
# SCHRITT 50: HMM-VERBESSERUNGEN VS. VERLORENE BALLBESITZPHASEN
# ============================================================

import numpy as np
import pandas as pd

frames = np.sort(oof["frame"].to_numpy(dtype=int))
gt = gt_lookup.reindex(frames).to_numpy()

v3 = np.asarray(hier_physics_v3)[frames]
hmm = np.asarray(hier_contact_hmm_v1)[frames]
contacts = np.asarray(contact_candidate_side)[frames]

assert np.isin(gt, ["H", "A", "D"]).all()
assert len(v3) == len(hmm) == len(gt)

def format_time(frame):
    sec = frame / FPS
    return f"{int(sec//3600):02d}:{int(sec%3600//60):02d}:{sec%60:05.2f}"

# ------------------------------------------------------------
# 1. Framekorrekturen und neue Fehler
# ------------------------------------------------------------

live_gt = np.isin(gt, ["H", "A"])

fixed = live_gt & (v3 != gt) & (hmm == gt)
broken = live_gt & (v3 == gt) & (hmm != gt)
both_wrong = live_gt & (v3 != gt) & (hmm != gt)
both_correct = live_gt & (v3 == gt) & (hmm == gt)

print("=== EFFEKT DES HMM AUF TATSÄCHLICHES LIVE ===")

display(pd.DataFrame([{
    "Beide korrekt": int(both_correct.sum()),
    "HMM korrigiert Fehler": int(fixed.sum()),
    "HMM erzeugt neuen Fehler": int(broken.sum()),
    "Beide falsch": int(both_wrong.sum()),
    "Netto verbesserte Frames": int(
        fixed.sum() - broken.sum()
    )
}]))

print(
    "Veränderte Frames bei GT=D:",
    int(np.sum((v3 != hmm) & (gt == "D")))
)

# ------------------------------------------------------------
# 2. Echte H/A-Phasen extrahieren
# ------------------------------------------------------------

def get_runs(labels):
    labels = np.asarray(labels)

    boundaries = np.flatnonzero(
        (labels[1:] != labels[:-1])
        | (np.diff(frames) != 1)
    ) + 1

    return np.split(
        np.arange(len(labels)),
        boundaries
    )

gt_runs = get_runs(gt)

records = []

for group in gt_runs:
    team = gt[group[0]]

    if team not in ("H", "A"):
        continue

    start = int(frames[group[0]])
    end = int(frames[group[-1]])

    v3_correct = np.mean(v3[group] == team)
    hmm_correct = np.mean(hmm[group] == team)

    records.append({
        "start_frame": start,
        "video_time": format_time(start),
        "GT_team": team,
        "duration_s": len(group) / FPS,
        "contacts": int(
            np.isin(contacts[group], ["H", "A"]).sum()
        ),
        "v3_correct_pct": 100 * v3_correct,
        "hmm_correct_pct": 100 * hmm_correct,
        "change_pp": 100 * (
            hmm_correct - v3_correct
        ),
        "v3_D_pct": 100 * np.mean(
            v3[group] == "D"
        ),
        "hmm_D_pct": 100 * np.mean(
            hmm[group] == "D"
        )
    })

df_gt_spell_comparison = pd.DataFrame(records)

print("\n=== ALLE 17 ECHTEN H/A-PHASEN ===")

display(
    df_gt_spell_comparison.round(2)
)

print("\n=== STÄRKSTE VERSCHLECHTERUNGEN ===")

display(
    df_gt_spell_comparison.nsmallest(
        8, "change_pp"
    ).round(2)
)

# ------------------------------------------------------------
# 3. Teamwechselereignisse ermitteln
# ------------------------------------------------------------

def direct_switch_events(labels):
    events = []

    for i in range(1, len(labels)):
        if frames[i] != frames[i-1] + 1:
            continue

        previous = labels[i-1]
        current = labels[i]

        if (
            previous in ("H", "A")
            and current in ("H", "A")
            and previous != current
        ):
            events.append({
                "frame": int(frames[i]),
                "from": previous,
                "to": current
            })

    return events

gt_events = direct_switch_events(gt)
v3_events = direct_switch_events(v3)
hmm_events = direct_switch_events(hmm)

# Eins-zu-eins-Zuordnung mit gleicher Wechselrichtung
def match_switches(reference, predicted, tolerance_frames):

    candidates = []

    for i, r in enumerate(reference):
        for j, p in enumerate(predicted):

            if (
                r["from"] != p["from"]
                or r["to"] != p["to"]
            ):
                continue

            distance = abs(r["frame"] - p["frame"])

            if distance <= tolerance_frames:
                candidates.append((distance, i, j))

    used_ref = set()
    used_pred = set()
    matches = {}

    for distance, i, j in sorted(candidates):
        if i in used_ref or j in used_pred:
            continue

        used_ref.add(i)
        used_pred.add(j)
        matches[i] = j

    return matches

print("\n=== DIREKTE H/A-WECHSEL ===")

switch_summary = []

for tolerance_s in [2, 5]:

    tolerance_frames = int(tolerance_s * FPS)

    for name, events in [
        ("Physics v3", v3_events),
        ("Kontakt-HMM v1", hmm_events)
    ]:

        matches = match_switches(
            gt_events, events, tolerance_frames
        )

        switch_summary.append({
            "Modell": name,
            "Toleranz_s": tolerance_s,
            "GT_Wechsel": len(gt_events),
            "Vorhergesagt": len(events),
            "Zugeordnet": len(matches),
            "GT_nicht_erkannt": (
                len(gt_events) - len(matches)
            ),
            "Vorhersagen_ohne_Match": (
                len(events) - len(matches)
            )
        })

display(pd.DataFrame(switch_summary))

# ------------------------------------------------------------
# 4. Welche GT-Wechsel wurden durch das HMM verloren?
# ------------------------------------------------------------

v3_matches = match_switches(
    gt_events, v3_events, 2 * FPS
)

hmm_matches = match_switches(
    gt_events, hmm_events, 2 * FPS
)

lost_records = []

for i, event in enumerate(gt_events):

    if i not in v3_matches or i in hmm_matches:
        continue

    frame = event["frame"]

    lo = max(0, frame - 2 * FPS)
    hi = min(N_FRAMES, frame + 2 * FPS + 1)

    local_contacts = contact_candidate_side[lo:hi]

    lost_records.append({
        "GT_switch_frame": frame,
        "video_time": format_time(frame),
        "from": event["from"],
        "to": event["to"],
        "v3_timing_error_s": (
            v3_events[v3_matches[i]]["frame"] - frame
        ) / FPS,
        "same_side_contacts_2s": int(
            np.sum(local_contacts == event["to"])
        ),
        "opposite_contacts_2s": int(
            np.sum(local_contacts == event["from"])
        )
    })

print("\n=== GT-WECHSEL: V3 ERKANNT, HMM VERLOREN ===")

df_lost_switches = pd.DataFrame(lost_records)

display(df_lost_switches)

# ------------------------------------------------------------
# 5. Diagnostik speichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_contact_hmm_v1"
    / "diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_gt_spell_comparison.to_csv(
    OUT / "gt_spell_comparison.csv",
    index=False
)

df_lost_switches.to_csv(
    OUT / "lost_gt_switches.csv",
    index=False
)

print("\nDiagnostik gespeichert:", OUT)



In [ ]:

# ============================================================
# SCHRITT 51: PHYSIKALISCHE QUALITÄT DER H/A-WECHSEL
# ============================================================

import numpy as np
import pandas as pd

speed = movement_features["ball_speed_mps"].to_numpy(float)
moving = movement_features["moving_fraction"].to_numpy(float)
coverage = movement_features["ball_coverage"].to_numpy(float)
nearest = np.asarray(nearest_smooth, dtype=float)

contacts = np.asarray(contact_candidate_side)
reference = np.asarray(hier_physics_v3)
hmm = np.asarray(hier_contact_hmm_v1)

assert len(speed) == len(moving) == len(coverage) == N_FRAMES
assert len(nearest) == len(contacts) == N_FRAMES

# Bereits in Schritt 50 berechnete Wechselereignisse.
matches_2s = match_switches(gt_events, v3_events, 2 * FPS)
matches_5s = match_switches(gt_events, v3_events, 5 * FPS)

match_2_by_frame = {
    v3_events[j]["frame"]: gt_events[i]["frame"]
    for i, j in matches_2s.items()
}

match_5_by_frame = {
    v3_events[j]["frame"]: gt_events[i]["frame"]
    for i, j in matches_5s.items()
}

def median_window(arr, start, end):
    a = np.asarray(
        arr[max(0, start):min(N_FRAMES, end)],
        dtype=float
    )
    a = a[np.isfinite(a)]
    return float(np.median(a)) if len(a) else np.nan

public_frames = np.sort(oof["frame"].to_numpy(dtype=int))
gt_public = gt_lookup.reindex(public_frames).to_numpy()
gt_full_diagnostic = np.full(N_FRAMES, "X", dtype="<U1")
gt_full_diagnostic[public_frames] = gt_public

switches = df_ha_spells[
    df_ha_spells["start_cause"] == "Direkter H/A-Wechsel"
].sort_values("start_frame")

records = []

for row in switches.itertuples():

    f = int(row.start_frame)
    side = str(row.team)
    other = "A" if side == "H" else "H"

    # Rein beobachtungsbasierte Kontaktmerkmale
    before_speed = median_window(speed, f - FPS, f)
    after_speed = median_window(speed, f, f + FPS)

    next_contacts = contacts[
        f + 1:min(N_FRAMES, f + 3 * FPS + 1)
    ]

    # Referenz nur für diagnostischen Vergleich
    gt_after = gt_full_diagnostic[
        f:min(N_FRAMES, f + 3 * FPS)
    ]
    gt_after = gt_after[np.isin(gt_after, ["H", "A"])]

    gt_side_pct = (
        100 * np.mean(gt_after == side)
        if len(gt_after) else np.nan
    )

    hmm_next = hmm[
        f:min(N_FRAMES, f + 3 * FPS)
    ]

    gt_match = match_5_by_frame.get(f)

    records.append({
        "frame": f,
        "time_s": f / FPS,
        "to_team": side,
        "pred_spell_s": row.duration_s,
        "ball_speed_before": before_speed,
        "ball_speed_after": after_speed,
        "ball_speed_delta": (
            after_speed - before_speed
        ),
        "moving_at_contact": median_window(
            moving, f - 12, f + 13
        ),
        "ball_coverage": median_window(
            coverage, f - 12, f + 13
        ),
        "nearest_player_m": median_window(
            nearest, f - 12, f + 13
        ),
        "same_contacts_next3s": int(
            np.sum(next_contacts == side)
        ),
        "opposite_contacts_next3s": int(
            np.sum(next_contacts == other)
        ),
        "hmm_agreement_next3s_pct": (
            100 * np.mean(hmm_next == side)
        ),
        "GT_team_share_next3s_pct": gt_side_pct,
        "matched_GT_2s": f in match_2_by_frame,
        "matched_GT_5s": gt_match is not None,
        "GT_timing_error_s": (
            (f - gt_match) / FPS
            if gt_match is not None else np.nan
        )
    })

df_switch_physics = pd.DataFrame(records)

print("=== PHYSIK DER 16 TEAMWECHSEL ===")

display(df_switch_physics.round(2))

print("\n=== VERGLEICH: GT-WECHSEL INNERHALB +/-5 S ===")

summary = (
    df_switch_physics
    .groupby("matched_GT_5s")
    .agg(
        n=("frame", "size"),
        spell_s=("pred_spell_s", "median"),
        speed_before=("ball_speed_before", "median"),
        speed_after=("ball_speed_after", "median"),
        speed_delta=("ball_speed_delta", "median"),
        moving=("moving_at_contact", "median"),
        nearest_player=("nearest_player_m", "median"),
        ball_coverage=("ball_coverage", "median"),
        same_contacts=("same_contacts_next3s", "median"),
        hmm_agreement=(
            "hmm_agreement_next3s_pct", "median"
        )
    )
    .reset_index()
)

display(summary.round(3))

print("\n=== NICHT ZUGEORDNETE WECHSEL ===")

display(
    df_switch_physics[
        ~df_switch_physics["matched_GT_5s"]
    ].round(2)
)

# Abspeichern
out = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

out.mkdir(parents=True, exist_ok=True)

df_switch_physics.to_csv(
    out / "ha_switch_physical_evidence.csv",
    index=False
)


In [ ]:

# ============================================================
# SCHRITT 52: TRACKING-STRUKTUREN IM NOTEBOOK PRÜFEN
# ============================================================

import re
import itertools
import numpy as np
import pandas as pd
from pathlib import Path

name_pattern = re.compile(
    r"track|player|team|home|away|ball|dist|coord|"
    r"position|spatial|pitch|event|frame|match",
    re.IGNORECASE
)

column_pattern = re.compile(
    r"(^x$|^y$|^z$|^x_\w+|^y_\w+|"
    r"ball|player|team|track|home|away|"
    r"coord|position|distance|jersey|shirt)",
    re.IGNORECASE
)

records = []
geometry_records = []

# Snapshot, damit die Untersuchung selbst keine Probleme
# durch neu hinzugefügte Notebook-Variablen erzeugt.
namespace = list(globals().items())

for name, obj in namespace:

    if name.startswith("_") or len(name) > 100:
        continue

    name_match = bool(name_pattern.search(name))

    if isinstance(obj, pd.DataFrame):
        columns = [str(c) for c in obj.columns]

        spatial_cols = [
            c for c in columns
            if column_pattern.search(c)
        ]

        if spatial_cols:
            geometry_records.append({
                "variable": name,
                "rows": len(obj),
                "columns": len(columns),
                "relevant_columns": ", ".join(
                    spatial_cols[:25]
                )
            })

        if not (name_match or spatial_cols):
            continue

        records.append({
            "variable": name,
            "type": "DataFrame",
            "shape_or_size": str(obj.shape),
            "structure": ", ".join(columns[:25])
        })

    elif name_match and isinstance(obj, np.ndarray):
        records.append({
            "variable": name,
            "type": "ndarray",
            "shape_or_size": str(obj.shape),
            "structure": f"dtype={obj.dtype}"
        })

    elif name_match and isinstance(obj, dict):
        keys = list(itertools.islice(obj.keys(), 12))

        records.append({
            "variable": name,
            "type": "dict",
            "shape_or_size": str(len(obj)),
            "structure": "keys: " + str(keys)[:220]
        })

    elif name_match and isinstance(obj, (list, tuple)):
        first_type = (
            type(obj[0]).__name__ if len(obj) else "-"
        )

        records.append({
            "variable": name,
            "type": type(obj).__name__,
            "shape_or_size": str(len(obj)),
            "structure": f"first element: {first_type}"
        })

    elif name_match and isinstance(obj, Path):
        records.append({
            "variable": name,
            "type": "Path",
            "shape_or_size": "-",
            "structure": str(obj)[:220]
        })

# ------------------------------------------------------------
# 1. Relevante Variablen
# ------------------------------------------------------------

df_tracking_inventory = pd.DataFrame(records)

print("=== VORHANDENE TRACKING-/TEAM-VARIABLEN ===")

if not df_tracking_inventory.empty:
    display(
        df_tracking_inventory
        .sort_values(["type", "variable"])
        .head(70)
    )
else:
    print("Keine passenden Variablen gefunden.")

# ------------------------------------------------------------
# 2. Datenframes mit potenziellen Positions-/Teamspalten
# ------------------------------------------------------------

df_geometry_inventory = pd.DataFrame(geometry_records)

print("\n=== POTENZIELLE GEOMETRIEDATEN ===")

if not df_geometry_inventory.empty:
    display(
        df_geometry_inventory
        .sort_values("variable")
        .head(40)
    )
else:
    print("Keine passenden DataFrame-Spalten gefunden.")

# ------------------------------------------------------------
# 3. Bisherige Informationsgrenzen
# ------------------------------------------------------------

print("\n=== BISHERIGE BASISMERKMALE ===")

for name in [
    "nearest_distance",
    "nearest_smooth",
    "contact_candidate_side",
    "movement_features"
]:
    obj = globals().get(name)

    if obj is None:
        print(name, ": nicht vorhanden")

    elif isinstance(obj, pd.DataFrame):
        print(
            name, ":",
            obj.shape,
            "| Spalten:",
            obj.columns.tolist()
        )

    else:
        arr = np.asarray(obj)
        print(
            name, ":",
            arr.shape,
            "| dtype:",
            arr.dtype
        )


In [ ]:

# ============================================================
# SCHRITT 53: TRACKING-GEOMETRIE UND DATENHERKUNFT
# ============================================================

import numpy as np
import pandas as pd
import json
import re
from pathlib import Path

# ------------------------------------------------------------
# 1. Mehrdimensionale NumPy-Arrays identifizieren
# ------------------------------------------------------------

arrays = []

for name, obj in list(globals().items()):

    if name.startswith("_"):
        continue

    if not isinstance(obj, np.ndarray):
        continue

    if obj.ndim < 2:
        continue

    arrays.append({
        "variable": name,
        "shape": str(obj.shape),
        "dtype": str(obj.dtype),
        "full_match": N_FRAMES in obj.shape,
        "possible_player_axis": any(
            10 <= d <= 50 for d in obj.shape
        )
    })

df_arrays = pd.DataFrame(arrays)

print("=== MEHRDIMENSIONALE ARRAYS ===")

if not df_arrays.empty:
    display(
        df_arrays.sort_values(
            ["full_match", "possible_player_axis"],
            ascending=False
        ).head(40)
    )
else:
    print("Keine mehrdimensionalen Arrays gefunden.")

# ------------------------------------------------------------
# 2. Vorhandene Tracking-Datenstrukturen
# ------------------------------------------------------------

keywords = re.compile(
    r"track|coord|position|player|team|"
    r"home|away|ball|raw_data|match_data",
    re.IGNORECASE
)

structures = []

for name, obj in list(globals().items()):

    if not keywords.search(name):
        continue

    if isinstance(obj, dict):
        structures.append({
            "variable": name,
            "type": "dict",
            "size": len(obj),
            "details": str(list(obj.keys())[:15])[:250]
        })

    elif isinstance(obj, pd.DataFrame):
        if (
            len(obj) >= 10000
            or keywords.search(name)
        ):
            structures.append({
                "variable": name,
                "type": "DataFrame",
                "size": len(obj),
                "details": str(obj.columns.tolist())[:250]
            })

print("\n=== MÖGLICHE TRACKING-STRUKTUREN ===")

if structures:
    display(
        pd.DataFrame(structures)
        .sort_values("variable")
        .head(35)
    )

# ------------------------------------------------------------
# 3. Code zur Berechnung der Abstände finden
# ------------------------------------------------------------

patterns = [
    re.compile(r"\bnearest_distance\s*="),
    re.compile(r"\bcontact_candidate_side\s*="),
    re.compile(r"\bdistances\s*="),
    re.compile(r"\bplayer_positions\s*="),
    re.compile(r"\bball_positions\s*="),
    re.compile(r"np\.linalg\.norm"),
]

notebook_files = sorted(
    PROJECT_DIR.rglob("*.ipynb"),
    key=lambda p: (
        0 if p.name.startswith("09_") else
        1 if p.name.startswith(("04_", "06_")) else 2,
        str(p)
    )
)

print("\n=== QUELLCODE DER ABSTANDSBERECHNUNG ===")
print("Gefundene Notebooks:", len(notebook_files))

hits = []

for path in notebook_files[:30]:

    try:
        if path.stat().st_size > 50_000_000:
            continue

        notebook = json.loads(
            path.read_text(encoding="utf-8")
        )

    except Exception:
        continue

    for cell_number, cell in enumerate(
        notebook.get("cells", [])
    ):

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        lines = source.splitlines()

        for pattern in patterns:
            matched = [
                i for i, line in enumerate(lines)
                if pattern.search(line)
            ]

            if not matched:
                continue

            line_number = matched[0]

            excerpt = "\n".join(
                lines[
                    max(0, line_number - 6):
                    min(len(lines), line_number + 16)
                ]
            )

            hits.append({
                "notebook": path.name,
                "cell": cell_number,
                "pattern": pattern.pattern,
                "excerpt": excerpt
            })

            break

# Nur die relevantesten Treffer zeigen
for hit in hits[:8]:
    print("\n" + "=" * 65)
    print(
        f"{hit['notebook']} | "
        f"Codezelle {hit['cell']}"
    )
    print("-" * 65)
    print(hit["excerpt"])

print("\nInsgesamt gefundene Codezellen:", len(hits))

# ------------------------------------------------------------
# 4. Trackingdateien im Projektverzeichnis
# ------------------------------------------------------------

print("\n=== MÖGLICHE TRACKING-DATEIEN ===")

file_pattern = re.compile(
    r"track|player|position|ball|coord",
    re.IGNORECASE
)

tracking_files = []

for extension in ("*.parquet", "*.csv", "*.npz", "*.npy"):

    for path in PROJECT_DIR.rglob(extension):

        if file_pattern.search(path.name):
            tracking_files.append({
                "file": str(path.relative_to(PROJECT_DIR)),
                "size_mb": round(
                    path.stat().st_size / 1_000_000, 2
                )
            })

if tracking_files:
    display(pd.DataFrame(tracking_files).head(40))
else:
    print("Keine Dateien mit passenden Namen gefunden.")



In [ ]:

# ============================================================
# SCHRITT 54: TEAM-SPEZIFISCHE BALLNÄHE
# ============================================================

import numpy as np
import pandas as pd
from collections import Counter

# ------------------------------------------------------------
# 1. Tracking-Strukturen und Teamzuordnung prüfen
# ------------------------------------------------------------

assert player_xy_contact.shape == (N_FRAMES, 46, 2)
assert ball_xy_contact.shape == (N_FRAMES, 2)
assert len(player_team) == 46

print("=== TEAM-MAPPING ===")
print("team_map:", team_map)

def resolve_team(raw_team):
    """Team-ID anhand expliziter Metadaten H/A zuordnen."""
    aliases = [str(raw_team)]

    # team_map kann in beide Richtungen vorliegen
    for key, value in team_map.items():
        if str(key) == str(raw_team):
            aliases.append(str(value))
        if str(value) == str(raw_team):
            aliases.append(str(key))

    for alias in aliases:
        a = alias.strip().lower()

        if a in ("h", "home", "bar", "barca", "barça"):
            return "H"
        if "barcelona" in a:
            return "H"

        if a in ("a", "away", "ath", "bilbao"):
            return "A"
        if "athletic" in a:
            return "A"

    return None

track_rows = []

for track_number in range(1, 47):

    raw_team = player_team.get(
        str(track_number),
        player_team.get(track_number)
    )

    track_rows.append({
        "track_number": track_number,
        "array_index": track_number - 1,
        "raw_team": raw_team,
        "side": resolve_team(raw_team)
    })

df_track_teams = pd.DataFrame(track_rows)

print("\nSpieler-Tracks nach Team:")
display(
    df_track_teams.groupby(
        ["raw_team", "side"],
        dropna=False
    ).size().reset_index(name="tracks")
)

if df_track_teams["side"].isna().any():
    raise ValueError(
        "Teamzuordnung nicht eindeutig. "
        "Bitte die angezeigte Team-Mapping-Tabelle prüfen."
    )

idx_H = df_track_teams.loc[
    df_track_teams["side"] == "H",
    "array_index"
].to_numpy(dtype=int)

idx_A = df_track_teams.loc[
    df_track_teams["side"] == "A",
    "array_index"
].to_numpy(dtype=int)

assert len(idx_H) > 0 and len(idx_A) > 0
assert len(idx_H) + len(idx_A) == 46

# ------------------------------------------------------------
# 2. Minimaldistanz Ball <-> Spieler je Team
#    Blockweise, um Speicherverbrauch gering zu halten
# ------------------------------------------------------------

ball_valid_mask = np.asarray(ball_seen, dtype=bool)

assert len(ball_valid_mask) == N_FRAMES

def team_min_distance(track_indices, chunk_size=10000):

    result = np.full(
        N_FRAMES,
        np.nan,
        dtype=np.float32
    )

    for start in range(0, N_FRAMES, chunk_size):

        end = min(N_FRAMES, start + chunk_size)

        p = np.asarray(
            player_xy_contact[start:end][:, track_indices, :],
            dtype=np.float32
        )

        b = np.asarray(
            ball_xy_contact[start:end],
            dtype=np.float32
        )

        distances = np.hypot(
            p[:, :, 0] - b[:, None, 0],
            p[:, :, 1] - b[:, None, 1]
        )

        distances = np.where(
            np.isfinite(distances),
            distances,
            np.inf
        )

        nearest = distances.min(axis=1)
        nearest[~np.isfinite(nearest)] = np.nan

        result[start:end] = nearest

    # Keine Ballposition -> keine räumliche Evidenz
    result[~ball_valid_mask] = np.nan

    return result

distance_H = team_min_distance(idx_H)
distance_A = team_min_distance(idx_A)

# Positive Werte: Barcelona näher
# Negative Werte: Athletic näher
proximity_margin_H = distance_A - distance_H

both_valid = (
    np.isfinite(distance_H)
    & np.isfinite(distance_A)
)

nearest_team = np.full(
    N_FRAMES, "X", dtype="<U1"
)

nearest_team[
    both_valid & (distance_H < distance_A)
] = "H"

nearest_team[
    both_valid & (distance_A < distance_H)
] = "A"

# ------------------------------------------------------------
# 3. Qualitätsprüfung gegenüber bisherigem Abstand
# ------------------------------------------------------------

nearest_new = np.fmin(distance_H, distance_A)
old = np.asarray(nearest_distance, dtype=float)

comparable = (
    np.isfinite(nearest_new)
    & np.isfinite(old)
)

print("\n=== TRACKING-ABDECKUNG ===")
print("Frames mit gültigem Ball:", int(ball_valid_mask.sum()))
print("Frames mit beiden Teamdistanzen:", int(both_valid.sum()))
print(
    "Anteil am gesamten Video:",
    f"{100 * both_valid.mean():.2f}%"
)

if comparable.any():
    print(
        "Median |neuer - bisheriger Minimalabstand|:",
        round(float(np.median(
            np.abs(nearest_new[comparable] - old[comparable])
        )), 3),
        "m"
    )

print("\n=== DISTANZVERTEILUNG ===")

display(pd.DataFrame({
    "Merkmal": [
        "Barcelona-Minimaldistanz",
        "Athletic-Minimaldistanz",
        "H-Nähevorsprung (A minus H)"
    ],
    "Median_m": [
        np.nanmedian(distance_H),
        np.nanmedian(distance_A),
        np.nanmedian(proximity_margin_H)
    ],
    "P10_m": [
        np.nanpercentile(distance_H, 10),
        np.nanpercentile(distance_A, 10),
        np.nanpercentile(proximity_margin_H, 10)
    ],
    "P90_m": [
        np.nanpercentile(distance_H, 90),
        np.nanpercentile(distance_A, 90),
        np.nanpercentile(proximity_margin_H, 90)
    ]
}).round(3))

# ------------------------------------------------------------
# 4. Distanzlage an den 16 öffentlichen H/A-Wechseln
# ------------------------------------------------------------

def safe_median(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(np.median(values)) if len(values) else np.nan

rows = []

for event in v3_events:

    f = int(event["frame"])
    new_side = event["to"]

    lo = max(0, f - FPS)
    hi = min(N_FRAMES, f + FPS)

    margin_before = safe_median(
        proximity_margin_H[lo:f]
    )

    margin_after = safe_median(
        proximity_margin_H[f:hi]
    )

    rows.append({
        "frame": f,
        "to_team": new_side,
        "distance_H_m": distance_H[f],
        "distance_A_m": distance_A[f],
        "margin_before_1s": margin_before,
        "margin_after_1s": margin_after,
        "margin_change": margin_after - margin_before,
        "nearest_team_at_contact": nearest_team[f],
        "nearest_matches_new_team": (
            nearest_team[f] == new_side
        )
    })

df_team_proximity_switches = pd.DataFrame(rows)

print("\n=== BALLNÄHE AN DEN 16 H/A-WECHSELN ===")
display(df_team_proximity_switches.round(3))

# ------------------------------------------------------------
# 5. Rein diagnostischer Vergleich zur öffentlichen GT
# ------------------------------------------------------------

public_frames = np.sort(
    oof["frame"].to_numpy(dtype=int)
)

public_gt = gt_lookup.reindex(
    public_frames
).to_numpy()

valid_eval = (
    np.isin(public_gt, ["H", "A"])
    & np.isin(nearest_team[public_frames], ["H", "A"])
)

if valid_eval.any():

    accuracy = np.mean(
        nearest_team[public_frames][valid_eval]
        == public_gt[valid_eval]
    )

    print("\n=== NAIVE NÄCHSTES-TEAM-HEURISTIK ===")
    print("Auswertbare öffentliche LIVE-Frames:", int(valid_eval.sum()))
    print(
        "Trefferquote des räumlich nächsten Teams:",
        f"{100 * accuracy:.2f}%"
    )

print("\nKeine Modellvorhersagen verändert.")


In [ ]:

# ============================================================
# SCHRITT 55: ZEITLICHE BALLNÄHE UND KONTAKTQUALITÄT
# ============================================================

import numpy as np
import pandas as pd

margin = np.asarray(proximity_margin_H, dtype=float)
contacts = np.asarray(contact_candidate_side)
pred = np.asarray(hier_physics_v3)

assert len(margin) == N_FRAMES

public_frames = np.sort(
    oof["frame"].to_numpy(dtype=int)
)
public_gt = gt_lookup.reindex(public_frames).to_numpy()

# Ausschließlich diagnostische Referenz
gt_diag = np.full(N_FRAMES, "X", dtype="<U1")
gt_diag[public_frames] = public_gt

# ------------------------------------------------------------
# 1. Zeitliche Nähe für ein bestimmtes Team
# ------------------------------------------------------------

def temporal_proximity(frame, side, start_s, end_s):

    start = max(
        0, frame + int(round(start_s * FPS))
    )
    end = min(
        N_FRAMES, frame + int(round(end_s * FPS))
    )

    if end <= start:
        return {
            "support_pct": np.nan,
            "margin_m": np.nan,
            "coverage_pct": 0.0
        }

    values = margin[start:end]

    # Positive Werte bedeuten:
    # Das angegebene Team ist näher am Ball.
    orientation = 1 if side == "H" else -1

    values = orientation * values
    valid = np.isfinite(values)

    coverage_pct = 100 * valid.mean()

    if not valid.any():
        return {
            "support_pct": np.nan,
            "margin_m": np.nan,
            "coverage_pct": coverage_pct
        }

    values = values[valid]

    return {
        "support_pct": 100 * np.mean(values > 0),
        "margin_m": float(np.median(values)),
        "coverage_pct": coverage_pct
    }


def gt_side_share(frame, side, start_s, end_s):

    start = max(
        0, frame + int(round(start_s * FPS))
    )
    end = min(
        N_FRAMES, frame + int(round(end_s * FPS))
    )

    values = gt_diag[start:end]
    valid = np.isin(values, ["H", "A"])

    return (
        100 * np.mean(values[valid] == side)
        if valid.any() else np.nan
    )

# ------------------------------------------------------------
# 2. Alle 16 vorhergesagten H/A-Wechsel untersuchen
# ------------------------------------------------------------

records = []

for row in df_switch_physics.itertuples():

    frame = int(row.frame)
    side = str(row.to_team)

    before = temporal_proximity(
        frame, side, -1.0, -0.2
    )

    early = temporal_proximity(
        frame, side, 0.2, 0.8
    )

    later = temporal_proximity(
        frame, side, 0.8, 2.0
    )

    records.append({
        "frame": frame,
        "time_s": frame / FPS,
        "team": side,
        "pred_spell_s": row.pred_spell_s,

        "before_support_pct": before["support_pct"],
        "early_support_pct": early["support_pct"],
        "later_support_pct": later["support_pct"],

        "before_margin_m": before["margin_m"],
        "early_margin_m": early["margin_m"],
        "later_margin_m": later["margin_m"],

        "early_coverage_pct": early["coverage_pct"],
        "later_coverage_pct": later["coverage_pct"],

        "GT_at_contact": gt_diag[frame],
        "GT_side_early_pct": gt_side_share(
            frame, side, 0.2, 0.8
        ),

        "matched_GT_2s": bool(row.matched_GT_2s),
        "matched_GT_5s": bool(row.matched_GT_5s)
    })

df_temporal_switches = pd.DataFrame(records)

print("=== ZEITLICHE BALLNÄHE DER 16 WECHSEL ===")

display(
    df_temporal_switches[
        [
            "frame",
            "team",
            "pred_spell_s",
            "before_support_pct",
            "early_support_pct",
            "later_support_pct",
            "early_margin_m",
            "later_margin_m",
            "early_coverage_pct",
            "GT_side_early_pct",
            "matched_GT_2s",
            "matched_GT_5s"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 3. Gruppenvergleich
# ------------------------------------------------------------

print("\n=== MIT / OHNE GT-WECHSEL INNERHALB +/- 5 S ===")

display(
    df_temporal_switches
    .groupby("matched_GT_5s")
    .agg(
        n=("frame", "size"),
        before_support=("before_support_pct", "median"),
        early_support=("early_support_pct", "median"),
        later_support=("later_support_pct", "median"),
        early_margin=("early_margin_m", "median"),
        later_margin=("later_margin_m", "median"),
        early_coverage=("early_coverage_pct", "median"),
        GT_side_early=("GT_side_early_pct", "median")
    )
    .round(2)
)

# ------------------------------------------------------------
# 4. Größere Vergleichsgruppe: Alle 103 öffentlichen Kontakte
# ------------------------------------------------------------

contact_frames = public_frames[
    np.isin(
        contacts[public_frames], ["H", "A"]
    )
]

switch_frames = set(
    df_switch_physics["frame"].astype(int)
)

contact_records = []

for frame in contact_frames:

    side = str(contacts[frame])
    early = temporal_proximity(
        int(frame), side, 0.2, 0.8
    )
    later = temporal_proximity(
        int(frame), side, 0.8, 2.0
    )

    contact_records.append({
        "frame": int(frame),
        "triggered_switch": int(frame) in switch_frames,
        "contact_matches_GT": gt_diag[frame] == side,
        "early_support_pct": early["support_pct"],
        "later_support_pct": later["support_pct"],
        "early_margin_m": early["margin_m"],
        "later_margin_m": later["margin_m"]
    })

df_all_contact_proximity = pd.DataFrame(contact_records)

print("\n=== ALLE ÖFFENTLICHEN KONTAKTKANDIDATEN ===")

display(
    df_all_contact_proximity
    .groupby([
        "triggered_switch",
        "contact_matches_GT"
    ])
    .agg(
        n=("frame", "size"),
        early_support=("early_support_pct", "median"),
        later_support=("later_support_pct", "median"),
        early_margin=("early_margin_m", "median"),
        later_margin=("later_margin_m", "median")
    )
    .round(2)
)

# ------------------------------------------------------------
# 5. Fairer Genauigkeitsvergleich auf denselben Frames
# ------------------------------------------------------------

nearest_public = nearest_team[public_frames]
physics_public = pred[public_frames]

valid = (
    np.isin(public_gt, ["H", "A"])
    & np.isin(nearest_public, ["H", "A"])
)

print("\n=== VERGLEICH AUF IDENTISCHEN LIVE-FRAMES ===")
print("Frames:", int(valid.sum()))

print(
    "Nächstes Team:",
    f"{100 * np.mean(nearest_public[valid] == public_gt[valid]):.2f}%"
)

print(
    "Physics v3:",
    f"{100 * np.mean(physics_public[valid] == public_gt[valid]):.2f}%"
)

# ------------------------------------------------------------
# 6. Diagnose speichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_temporal_switches.to_csv(
    OUT / "temporal_proximity_switches.csv",
    index=False
)

df_all_contact_proximity.to_csv(
    OUT / "temporal_proximity_all_contacts.csv",
    index=False
)

print("\nDiagnostik abgeschlossen.")


In [ ]:

# ============================================================
# SCHRITT 56: BALL-SPIELER-BEWEGUNGSKOPPLUNG
# ============================================================

import numpy as np
import pandas as pd

LAG = 5                 # 0.2 Sekunden
CLOSE_DISTANCE_M = 2.5  # Deskriptive Nähedefinition
LOW_REL_SPEED_MPS = 4.0 # Deskriptive Kopplungsdefinition

players_xy = np.asarray(player_xy_contact)
ball_xy = np.asarray(ball_xy_contact)
ball_observed = np.asarray(ball_seen, dtype=bool)

team_indices = {
    "H": np.asarray(idx_H, dtype=int),
    "A": np.asarray(idx_A, dtype=int)
}

def player_ball_coupling(frame, side, start_s, end_s):
    """Bewegungskopplung zum nächsten Spieler eines Teams."""

    lo = max(LAG, frame + int(round(start_s * FPS)))
    hi = min(N_FRAMES, frame + int(round(end_s * FPS)))

    empty = {
        "valid_pct": 0.0,
        "close_pct": np.nan,
        "coupled_pct": np.nan,
        "rel_speed_close": np.nan
    }

    if hi <= lo:
        return empty

    t = np.arange(lo, hi)
    track_ids = team_indices[side]

    p = players_xy[t][:, track_ids, :]
    b = ball_xy[t]

    distances = np.linalg.norm(
        p - b[:, None, :],
        axis=2
    )

    distances[~np.isfinite(distances)] = np.inf

    nearest_local = np.argmin(distances, axis=1)
    nearest_dist = distances[
        np.arange(len(t)), nearest_local
    ]

    # Derselbe Spieler an t und t-LAG
    nearest_track = track_ids[nearest_local]

    p_now = players_xy[t, nearest_track]
    p_previous = players_xy[t - LAG, nearest_track]

    b_now = ball_xy[t]
    b_previous = ball_xy[t - LAG]

    relative_displacement = (
        (b_now - b_previous)
        - (p_now - p_previous)
    )

    relative_speed = (
        np.linalg.norm(relative_displacement, axis=1)
        * FPS / LAG
    )

    valid = (
        ball_observed[t]
        & ball_observed[t - LAG]
        & np.isfinite(nearest_dist)
        & np.isfinite(relative_speed)
        & np.isfinite(p_previous).all(axis=1)
    )

    if not valid.any():
        return empty

    close = valid & (nearest_dist <= CLOSE_DISTANCE_M)
    coupled = close & (relative_speed <= LOW_REL_SPEED_MPS)

    return {
        "valid_pct": 100 * np.mean(valid),
        "close_pct": 100 * close.sum() / valid.sum(),
        "coupled_pct": 100 * coupled.sum() / valid.sum(),
        "rel_speed_close": (
            float(np.median(relative_speed[close]))
            if close.any() else np.nan
        )
    }

# ------------------------------------------------------------
# 1. Alle 16 vorhergesagten H/A-Wechsel
# ------------------------------------------------------------

records = []

for event in df_switch_physics.itertuples():

    f = int(event.frame)
    new_team = str(event.to_team)
    old_team = "A" if new_team == "H" else "H"

    early_new = player_ball_coupling(
        f, new_team, 0.2, 0.8
    )
    early_old = player_ball_coupling(
        f, old_team, 0.2, 0.8
    )

    later_new = player_ball_coupling(
        f, new_team, 0.8, 2.0
    )
    later_old = player_ball_coupling(
        f, old_team, 0.8, 2.0
    )

    records.append({
        "frame": f,
        "new_team": new_team,
        "matched_GT_5s": bool(event.matched_GT_5s),
        "GT_side_next3s_pct": event.GT_team_share_next3s_pct,
        "new_close_early": early_new["close_pct"],
        "new_coupled_early": early_new["coupled_pct"],
        "old_coupled_early": early_old["coupled_pct"],
        "coupling_adv_early": (
            early_new["coupled_pct"]
            - early_old["coupled_pct"]
        ),
        "new_rel_speed_early": early_new["rel_speed_close"],
        "new_coupled_later": later_new["coupled_pct"],
        "old_coupled_later": later_old["coupled_pct"],
        "coupling_adv_later": (
            later_new["coupled_pct"]
            - later_old["coupled_pct"]
        ),
        "new_rel_speed_later": later_new["rel_speed_close"],
        "early_valid_pct": early_new["valid_pct"],
        "later_valid_pct": later_new["valid_pct"]
    })

df_coupling_switches = pd.DataFrame(records)

print("=== BEWEGUNGSKOPPLUNG: 16 H/A-WECHSEL ===")

display(df_coupling_switches.round(2))

# ------------------------------------------------------------
# 2. Vergleich zugeordnet / nicht zugeordnet
# ------------------------------------------------------------

print("\n=== GRUPPENVERGLEICH ===")

display(
    df_coupling_switches
    .groupby("matched_GT_5s")
    .agg(
        n=("frame", "size"),
        new_coupled_early=("new_coupled_early", "median"),
        old_coupled_early=("old_coupled_early", "median"),
        coupling_adv_early=("coupling_adv_early", "median"),
        new_rel_speed_early=("new_rel_speed_early", "median"),
        new_coupled_later=("new_coupled_later", "median"),
        coupling_adv_later=("coupling_adv_later", "median"),
        valid_early=("early_valid_pct", "median")
    )
    .round(2)
)

# ------------------------------------------------------------
# 3. Alle 103 Kontaktkandidaten
# ------------------------------------------------------------

contact_frames = np.sort(oof["frame"].to_numpy(dtype=int))
contact_frames = contact_frames[
    np.isin(contact_candidate_side[contact_frames], ["H", "A"])
]

public_gt_lookup = gt_lookup
switch_frames_set = set(df_switch_physics["frame"].astype(int))

contact_records = []

for f in contact_frames:

    side = str(contact_candidate_side[f])
    other = "A" if side == "H" else "H"

    new = player_ball_coupling(f, side, 0.2, 0.8)
    old = player_ball_coupling(f, other, 0.2, 0.8)

    contact_records.append({
        "frame": int(f),
        "triggered_switch": int(f) in switch_frames_set,
        "side_matches_GT": (
            public_gt_lookup.loc[f] == side
        ),
        "coupled_contact_side": new["coupled_pct"],
        "coupled_other_side": old["coupled_pct"],
        "coupling_advantage": (
            new["coupled_pct"] - old["coupled_pct"]
        ),
        "rel_speed_contact_side": new["rel_speed_close"],
        "valid_pct": new["valid_pct"]
    })

df_contact_coupling = pd.DataFrame(contact_records)

print("\n=== ALLE 103 KONTAKTKANDIDATEN ===")

display(
    df_contact_coupling
    .groupby(["triggered_switch", "side_matches_GT"])
    .agg(
        n=("frame", "size"),
        coupled_contact=("coupled_contact_side", "median"),
        coupled_other=("coupled_other_side", "median"),
        coupling_advantage=("coupling_advantage", "median"),
        rel_speed=("rel_speed_contact_side", "median"),
        valid_pct=("valid_pct", "median")
    )
    .round(2)
)

# ------------------------------------------------------------
# 4. Ergebnisse speichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_coupling_switches.to_csv(
    OUT / "player_ball_coupling_switches.csv",
    index=False
)

df_contact_coupling.to_csv(
    OUT / "player_ball_coupling_all_contacts.csv",
    index=False
)

print("\nDiagnostik abgeschlossen. Keine Vorhersagen verändert.")


In [ ]:

# ============================================================
# SCHRITT 57: FRAMEWEISE H/A-BEWEGUNGSKOPPLUNG
# ============================================================

import numpy as np
import pandas as pd

LAG = 5
CHUNK = 10000

ball_seen_arr = np.asarray(ball_seen, dtype=bool)
player_pos = player_xy_contact
ball_pos = ball_xy_contact

team_tracks = {
    "H": np.asarray(idx_H, dtype=int),
    "A": np.asarray(idx_A, dtype=int)
}

assert player_pos.shape == (N_FRAMES, 46, 2)
assert ball_pos.shape == (N_FRAMES, 2)

# ------------------------------------------------------------
# 1. Nächstgelegenen Spieler pro Team und Frame finden
# ------------------------------------------------------------

nearest_track_by_team = {}
relative_speed_by_team = {}
control_score_by_team = {}

# Für Geschwindigkeit müssen alle 6 Ballframes
# innerhalb des 0.2-s-Fensters vorhanden sein.
seen_window = np.zeros(N_FRAMES, dtype=bool)
seen_window[LAG:] = (
    np.convolve(
        ball_seen_arr.astype(np.int8),
        np.ones(LAG + 1, dtype=np.int8),
        mode="valid"
    ) == LAG + 1
)

for side, ids in team_tracks.items():

    nearest_track = np.full(
        N_FRAMES, -1, dtype=np.int16
    )

    for start in range(0, N_FRAMES, CHUNK):

        end = min(N_FRAMES, start + CHUNK)

        players = np.asarray(
            player_pos[start:end][:, ids, :],
            dtype=np.float32
        )

        ball_block = np.asarray(
            ball_pos[start:end],
            dtype=np.float32
        )

        distances = np.linalg.norm(
            players - ball_block[:, None, :],
            axis=2
        )

        distances[~np.isfinite(distances)] = np.inf

        best_local = np.argmin(distances, axis=1)
        valid = (
            np.isfinite(distances).any(axis=1)
            & ball_seen_arr[start:end]
        )

        nearest_track[start:end] = np.where(
            valid, ids[best_local], -1
        )

    nearest_track_by_team[side] = nearest_track

    # --------------------------------------------------------
    # 2. Relative Geschwindigkeit zum selben Spieler
    # --------------------------------------------------------

    relative_speed = np.full(
        N_FRAMES, np.nan, dtype=np.float32
    )

    t = np.arange(LAG, N_FRAMES)
    tracked = nearest_track[t]

    valid = (
        (tracked >= 0)
        & seen_window[t]
    )

    valid_t = t[valid]
    valid_tracks = tracked[valid]

    p_now = player_pos[valid_t, valid_tracks]
    p_before = player_pos[
        valid_t - LAG, valid_tracks
    ]

    b_now = ball_pos[valid_t]
    b_before = ball_pos[valid_t - LAG]

    rel = (
        np.linalg.norm(
            (b_now - b_before)
            - (p_now - p_before),
            axis=1
        )
        * FPS / LAG
    )

    relative_speed[valid_t] = np.where(
        np.isfinite(rel), rel, np.nan
    )

    relative_speed_by_team[side] = relative_speed

    # --------------------------------------------------------
    # 3. Kombinierter Kontroll-Evidenzwert
    # --------------------------------------------------------

    distance = np.asarray(
        distance_H if side == "H" else distance_A,
        dtype=float
    )

    # Hohe Werte bei Nähe UND ähnlicher Bewegung.
    # Die Skalen 4 m und 6 m/s sind Testannahmen.
    score = (
        np.exp(-distance / 4.0)
        * np.exp(-relative_speed / 6.0)
    )

    score[
        ~np.isfinite(distance)
        | ~np.isfinite(relative_speed)
    ] = np.nan

    control_score_by_team[side] = score.astype(
        np.float32
    )

# ------------------------------------------------------------
# 4. Rein physikalische Teamentscheidung
# ------------------------------------------------------------

score_H = control_score_by_team["H"]
score_A = control_score_by_team["A"]

valid_control = (
    np.isfinite(score_H)
    & np.isfinite(score_A)
)

coupling_team = np.full(
    N_FRAMES, "X", dtype="<U1"
)

coupling_team[
    valid_control & (score_H > score_A)
] = "H"

coupling_team[
    valid_control & (score_A > score_H)
] = "A"

# Positive Werte: Evidenz für Barcelona
control_margin_H = np.full(
    N_FRAMES, np.nan, dtype=np.float32
)

control_margin_H[valid_control] = (
    (score_H[valid_control] - score_A[valid_control])
    / (
        score_H[valid_control]
        + score_A[valid_control]
        + 1e-9
    )
)

# ------------------------------------------------------------
# 5. Fairer Vergleich auf denselben GT-LIVE-Frames
# ------------------------------------------------------------

public_frames = np.sort(
    oof["frame"].to_numpy(dtype=int)
)

gt = gt_lookup.reindex(
    public_frames
).to_numpy()

nearest_pred = nearest_team[public_frames]
coupling_pred = coupling_team[public_frames]
physics_pred = np.asarray(
    hier_physics_v3
)[public_frames]

valid_public = (
    np.isin(gt, ["H", "A"])
    & valid_control[public_frames]
    & np.isin(nearest_pred, ["H", "A"])
    & np.isin(coupling_pred, ["H", "A"])
)

def accuracy_on_mask(pred, mask):
    return (
        100 * np.mean(pred[mask] == gt[mask])
        if mask.any() else np.nan
    )

results = []

for name, prediction in [
    ("Nächstes Team", nearest_pred),
    ("Bewegungskopplung", coupling_pred),
    ("Physics v3", physics_pred)
]:

    results.append({
        "Modell": name,
        "GT_LIVE_Frames": int(valid_public.sum()),
        "Accuracy_pct": accuracy_on_mask(
            prediction, valid_public
        )
    })

print("=== VERGLEICH AUF IDENTISCHEN FRAMES ===")
display(pd.DataFrame(results).round(2))

# ------------------------------------------------------------
# 6. Liefert die Kopplung zusätzliche Information?
# ------------------------------------------------------------

disagreement = (
    valid_public
    & (nearest_pred != coupling_pred)
)

print("\n=== UNTERSCHIEDE ZUR REINEN BALLNÄHE ===")
print("Frames mit unterschiedlicher Entscheidung:",
      int(disagreement.sum()))

if disagreement.any():

    display(pd.DataFrame([
        {
            "Modell": "Nächstes Team",
            "Accuracy_pct": accuracy_on_mask(
                nearest_pred, disagreement
            )
        },
        {
            "Modell": "Bewegungskopplung",
            "Accuracy_pct": accuracy_on_mask(
                coupling_pred, disagreement
            )
        },
        {
            "Modell": "Physics v3",
            "Accuracy_pct": accuracy_on_mask(
                physics_pred, disagreement
            )
        }
    ]).round(2))

# ------------------------------------------------------------
# 7. Ergebnisse an den 16 H/A-Wechseln
# ------------------------------------------------------------

rows = []

for event in df_switch_physics.itertuples():

    f = int(event.frame)
    side = str(event.to_team)

    lo = f + 5
    hi = min(N_FRAMES, f + 20)

    future = coupling_team[lo:hi]
    valid = np.isin(future, ["H", "A"])

    rows.append({
        "frame": f,
        "new_team": side,
        "GT_match_5s": bool(event.matched_GT_5s),
        "coupling_at_contact": coupling_team[f],
        "control_margin_at_contact": (
            control_margin_H[f]
            if side == "H"
            else -control_margin_H[f]
        ),
        "coupling_new_team_next_0_2_to_0_8s_pct": (
            100 * np.mean(future[valid] == side)
            if valid.any() else np.nan
        ),
        "valid_next_window_pct": 100 * valid.mean()
    })

df_continuous_coupling_switches = pd.DataFrame(rows)

print("\n=== KOPPLUNG AN DEN 16 TEAMWECHSELN ===")
display(df_continuous_coupling_switches.round(2))

# ------------------------------------------------------------
# 8. Abdeckung und Speicherung
# ------------------------------------------------------------

print("\n=== GESAMTE TRACKING-ABDECKUNG ===")
print("Gültige H/A-Kopplungsframes:", int(valid_control.sum()))
print("Anteil Gesamtvideo:",
      f"{100 * valid_control.mean():.2f}%")
print("Öffentlich auswertbare LIVE-Frames:",
      int(valid_public.sum()))

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUT / "continuous_team_coupling.npz",
    score_H=score_H,
    score_A=score_A,
    margin_H=control_margin_H,
    coupling_team=coupling_team
)

df_continuous_coupling_switches.to_csv(
    OUT / "continuous_coupling_switches.csv",
    index=False
)

print("\nDiagnostik abgeschlossen.")
print("Physics-v3-Vorhersage unverändert.")


In [ ]:

# ============================================================
# SCHRITT 58: ZEITLICHE URSACHEN DER H/A-WECHSELFEHLER
# ============================================================

import numpy as np
import pandas as pd

public_frames = np.sort(oof["frame"].to_numpy(dtype=int))
gt_labels = gt_lookup.reindex(public_frames).to_numpy()
pred_labels = np.asarray(hier_physics_v3)[public_frames]
contacts = np.asarray(contact_candidate_side)

assert np.all(np.diff(public_frames) == 1)

# ------------------------------------------------------------
# 1. Direkte H/A-Wechsel aus Labelsequenzen
# ------------------------------------------------------------

def find_direct_switches(labels, frame_ids):
    events = []

    for i in range(1, len(labels)):
        old, new = labels[i-1], labels[i]

        if (
            frame_ids[i] == frame_ids[i-1] + 1
            and old in ("H", "A")
            and new in ("H", "A")
            and old != new
        ):
            events.append({
                "frame": int(frame_ids[i]),
                "from": old,
                "to": new
            })

    return events

gt_events_58 = find_direct_switches(
    gt_labels, public_frames
)

pred_events_58 = find_direct_switches(
    pred_labels, public_frames
)

# ------------------------------------------------------------
# 2. Eins-zu-eins-Matching gleicher Wechselrichtung
# ------------------------------------------------------------

MAX_DELTA = 5 * FPS

candidates = []

for i, g in enumerate(gt_events_58):
    for j, p in enumerate(pred_events_58):

        if (g["from"], g["to"]) != (p["from"], p["to"]):
            continue

        delta = p["frame"] - g["frame"]

        if abs(delta) <= MAX_DELTA:
            candidates.append((abs(delta), i, j, delta))

matched = {}
used_predictions = set()

for _, i, j, delta in sorted(candidates):
    if i in matched or j in used_predictions:
        continue

    matched[i] = (j, delta)
    used_predictions.add(j)

# ------------------------------------------------------------
# 3. Kontakt- und räumliche Wechselzeitpunkte
# ------------------------------------------------------------

contact_frames_by_side = {
    side: np.flatnonzero(contacts == side)
    for side in ("H", "A")
}

# Rein deskriptive 0.44-s-Medianglättung
margin_smooth = (
    pd.Series(np.asarray(proximity_margin_H, dtype=float))
    .rolling(11, center=True, min_periods=8)
    .median()
    .to_numpy()
)

spatial_sign = np.zeros(N_FRAMES, dtype=np.int8)
spatial_sign[margin_smooth > 0] = 1
spatial_sign[margin_smooth < 0] = -1

spatial_crossings = {}

for side, target in [("H", 1), ("A", -1)]:
    spatial_crossings[side] = np.flatnonzero(
        (spatial_sign[1:] == target)
        & (spatial_sign[:-1] == -target)
    ) + 1

def nearest_offset_s(event_frames, reference_frame, radius_s=5):
    candidates = event_frames[
        np.abs(event_frames - reference_frame)
        <= radius_s * FPS
    ]

    if not len(candidates):
        return np.nan

    nearest = candidates[
        np.argmin(np.abs(candidates - reference_frame))
    ]
    return float((nearest - reference_frame) / FPS)

def first_after_s(event_frames, reference_frame, radius_s=5):
    candidates = event_frames[
        (event_frames >= reference_frame)
        & (event_frames <= reference_frame + radius_s * FPS)
    ]

    return (
        float((candidates[0] - reference_frame) / FPS)
        if len(candidates) else np.nan
    )

# ------------------------------------------------------------
# 4. Alle GT-Wechsel vergleichen
# ------------------------------------------------------------

rows = []

for i, event in enumerate(gt_events_58):

    f = event["frame"]
    side = event["to"]

    match = matched.get(i)

    pred_delta = (
        match[1] / FPS if match is not None else np.nan
    )

    rows.append({
        "GT_frame": f,
        "video_time_s": f / FPS,
        "from": event["from"],
        "to": side,
        "v3_switch_error_s": pred_delta,
        "v3_matched_2s": (
            bool(abs(pred_delta) <= 2)
            if np.isfinite(pred_delta) else False
        ),
        "nearest_new_contact_s": nearest_offset_s(
            contact_frames_by_side[side], f
        ),
        "first_new_contact_after_s": first_after_s(
            contact_frames_by_side[side], f
        ),
        "nearest_spatial_crossing_s": nearest_offset_s(
            spatial_crossings[side], f
        ),
        "nearest_team_at_GT": nearest_team[f],
        "distance_H_m": float(distance_H[f]),
        "distance_A_m": float(distance_A[f])
    })

df_gt_transition_timing = pd.DataFrame(rows)

print("=== ZEITPUNKTE ALLER DIREKTEN GT-WECHSEL ===")

display(df_gt_transition_timing.round(2))

# ------------------------------------------------------------
# 5. Zusammenfassung
# ------------------------------------------------------------

print("\n=== ZUSAMMENFASSUNG ===")

print("Tatsächliche direkte GT-Wechsel:", len(gt_events_58))
print("Vorhergesagte direkte Wechsel:", len(pred_events_58))
print("GT-Wechsel mit Physics-Match +/-5s:", len(matched))

timing_cols = [
    "v3_switch_error_s",
    "nearest_new_contact_s",
    "nearest_spatial_crossing_s"
]

summaries = []

for col in timing_cols:

    offsets = df_gt_transition_timing[col].dropna()

    summaries.append({
        "Signal": col,
        "auswertbar": len(offsets),
        "median_offset_s": offsets.median(),
        "median_abs_error_s": offsets.abs().median(),
        "within_1s": int((offsets.abs() <= 1).sum()),
        "within_2s": int((offsets.abs() <= 2).sum()),
        "within_5s": int((offsets.abs() <= 5).sum())
    })

display(pd.DataFrame(summaries).round(3))

print("\nInterpretation der Vorzeichen:")
print("Positiv = Signal kommt NACH dem GT-Wechsel.")
print("Negativ = Signal kommt VOR dem GT-Wechsel.")

# ------------------------------------------------------------
# 6. Speicherung
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_gt_transition_timing.to_csv(
    OUT / "gt_transition_timing.csv",
    index=False
)

print("\nDiagnostik gespeichert.")


In [ ]:

# ============================================================
# SCHRITT 59: UNABHÄNGIGE RÄUMLICHE WECHSELKANDIDATEN
# ============================================================

import numpy as np
import pandas as pd

public_frames = np.sort(
    oof["frame"].to_numpy(dtype=int)
)
first, last = int(public_frames[0]), int(public_frames[-1])

gt_events = gt_events_58
pred_events = pred_events_58

CONTACT_WINDOW = 2 * FPS
ALIGN_WINDOW = 3 * FPS

# ------------------------------------------------------------
# 1. Ereignisse OHNE GT-Information erzeugen
# ------------------------------------------------------------

spatial_events = []

for side in ("H", "A"):
    previous = "A" if side == "H" else "H"

    for frame in spatial_crossings[side]:
        f = int(frame)

        if first <= f <= last:
            spatial_events.append({
                "frame": f,
                "from": previous,
                "to": side
            })

spatial_events.sort(key=lambda x: x["frame"])

spatial_live = [
    e for e in spatial_events
    if status_physics_v3[e["frame"]] == "L"
]

contact_frames = {
    side: np.flatnonzero(
        contact_candidate_side == side
    )
    for side in ("H", "A")
}

def nearest_contact_delta(frame, side):
    arr = contact_frames[side]
    k = np.searchsorted(arr, frame)

    candidates = []

    if k < len(arr):
        candidates.append(arr[k])
    if k > 0:
        candidates.append(arr[k-1])

    if not candidates:
        return np.nan

    return min(
        candidates,
        key=lambda x: abs(x-frame)
    ) - frame

spatial_contact = [
    e for e in spatial_live
    if abs(
        nearest_contact_delta(e["frame"], e["to"])
    ) <= CONTACT_WINDOW
]

# ------------------------------------------------------------
# 2. Ereignis-Matching: eins zu eins
#    GT wird NUR zur Evaluation verwendet
# ------------------------------------------------------------

def match_events(reference, proposals, tolerance_frames):
    candidates = []

    for i, g in enumerate(reference):
        for j, p in enumerate(proposals):

            if (g["from"], g["to"]) != (p["from"], p["to"]):
                continue

            delta = p["frame"] - g["frame"]

            if abs(delta) <= tolerance_frames:
                candidates.append(
                    (abs(delta), i, j, delta)
                )

    used_ref = set()
    used_pred = set()
    matches = []

    for _, i, j, delta in sorted(candidates):
        if i in used_ref or j in used_pred:
            continue

        used_ref.add(i)
        used_pred.add(j)
        matches.append((i, j, delta))

    return matches

event_sets = {
    "Physics v3": pred_events,
    "Räumlich, alle": spatial_events,
    "Räumlich, LIVE": spatial_live,
    "Räumlich + Kontakt": spatial_contact
}

rows = []

for name, proposals in event_sets.items():
    for seconds in [1, 2, 5]:

        matches = match_events(
            gt_events, proposals, seconds * FPS
        )

        n = len(matches)

        rows.append({
            "Verfahren": name,
            "Toleranz_s": seconds,
            "Kandidaten": len(proposals),
            "GT_Wechsel": len(gt_events),
            "Matches": n,
            "Precision": (
                n / len(proposals)
                if proposals else np.nan
            ),
            "Recall": n / len(gt_events),
            "Event_F1": (
                2 * n / (len(proposals) + len(gt_events))
                if proposals else 0.0
            )
        })

print("=== EREIGNISQUALITÄT DER RÄUMLICHEN WECHSEL ===")
display(pd.DataFrame(rows).round(3))

# ------------------------------------------------------------
# 3. Räumliche Kandidaten an Physics-v3-Wechsel koppeln
#    Ausrichtung erfolgt ausschließlich anhand Physics v3.
#    GT spielt bei der Auswahl KEINE Rolle.
# ------------------------------------------------------------

candidates = []

for i, p in enumerate(pred_events):
    for j, s in enumerate(spatial_live):

        if (p["from"], p["to"]) != (s["from"], s["to"]):
            continue

        delta = s["frame"] - p["frame"]

        if abs(delta) <= ALIGN_WINDOW:
            candidates.append(
                (abs(delta), i, j, delta)
            )

used_pred = set()
used_spatial = set()
alignment = {}

for _, i, j, delta in sorted(candidates):
    if i in used_pred or j in used_spatial:
        continue

    used_pred.add(i)
    used_spatial.add(j)
    alignment[i] = (j, delta)

aligned_events = []

for i, p in enumerate(pred_events):
    new_frame = (
        spatial_live[alignment[i][0]]["frame"]
        if i in alignment
        else p["frame"]
    )

    aligned_events.append({
        **p,
        "frame": new_frame
    })

aligned_events.sort(key=lambda e: e["frame"])

# ------------------------------------------------------------
# 4. Diagnostischer Vergleich nach Zuordnung
# ------------------------------------------------------------

print("\n=== ZEITLICHE AUSRICHTUNG ===")
print("Physics-v3-Wechsel:", len(pred_events))
print("Räumlich zugeordnet:", len(alignment))
print("Ohne räumliche Zuordnung:",
      len(pred_events) - len(alignment))

alignment_rows = []

for i, p in enumerate(pred_events):

    if i in alignment:
        j, delta = alignment[i]
        spatial_frame = spatial_live[j]["frame"]
    else:
        delta = np.nan
        spatial_frame = np.nan

    alignment_rows.append({
        "contact_frame": p["frame"],
        "time_s": p["frame"] / FPS,
        "from": p["from"],
        "to": p["to"],
        "spatial_frame": spatial_frame,
        "spatial_minus_contact_s": (
            delta / FPS
            if np.isfinite(delta) else np.nan
        ),
        "has_alignment": i in alignment
    })

df_spatial_alignment = pd.DataFrame(alignment_rows)

display(df_spatial_alignment.round(2))

print("\n=== MATCHING VOR / NACH AUSRICHTUNG ===")

comparison_rows = []

for name, events in [
    ("Physics v3", pred_events),
    ("Kontakt + räumliche Ausrichtung", aligned_events)
]:
    for seconds in [1, 2, 5]:
        matches = match_events(
            gt_events, events, seconds * FPS
        )

        comparison_rows.append({
            "Modell": name,
            "Toleranz_s": seconds,
            "Matches": len(matches),
            "Median_Abweichung_s": (
                np.median([
                    abs(delta) / FPS
                    for _, _, delta in matches
                ])
                if matches else np.nan
            )
        })

display(pd.DataFrame(comparison_rows).round(3))

# ------------------------------------------------------------
# 5. Ergebnisse sichern
# ------------------------------------------------------------

OUT = (
    PROJECT_DIR
    / "output/hierarchical_physics_v3"
    / "possession_diagnostics"
)

OUT.mkdir(parents=True, exist_ok=True)

df_spatial_alignment.to_csv(
    OUT / "spatial_contact_alignment.csv",
    index=False
)

print("\nNur Diagnose – keine Vorhersagen verändert.")


In [ ]:

# ============================================================
# SCHRITT 60: KONTAKT + RÄUMLICHE ZEITAUSRICHTUNG
# ============================================================

import numpy as np
import pandas as pd
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

MAX_SHIFT_SECONDS = 3.0
MAX_SHIFT_FRAMES = round(MAX_SHIFT_SECONDS * FPS)

reference = np.asarray(hier_physics_v3)
status = np.asarray(status_physics_v3)

hier_spatial_hybrid_v1 = reference.copy()

# Nur tatsächliche Spielhälften und LIVE-Abschnitte
active = np.zeros(N_FRAMES, dtype=bool)
for _, start, end in HALVES:
    active[start:end + 1] = True

live_mask = active & (status == "L")

# Bereits in Schritt 58 berechnete räumliche Wechsel
crossings = {
    "H": np.sort(np.asarray(spatial_crossings["H"], dtype=int)),
    "A": np.sort(np.asarray(spatial_crossings["A"], dtype=int))
}

# ------------------------------------------------------------
# 1. Zusammenhängende LIVE-Segmente
# ------------------------------------------------------------

edges = np.flatnonzero(
    np.diff(
        np.r_[False, live_mask, False].astype(np.int8)
    )
)

starts = edges[0::2]
stops = edges[1::2]

records = []
n_original_switches = 0

for seg_start, seg_stop in zip(starts, stops):

    if seg_stop - seg_start < 2:
        continue

    # Bestehende direkte H/A-Wechsel
    f = np.arange(seg_start + 1, seg_stop)

    previous = reference[f - 1]
    current = reference[f]

    is_switch = (
        np.isin(previous, ["H", "A"])
        & np.isin(current, ["H", "A"])
        & (previous != current)
    )

    switches = f[is_switch]
    n_original_switches += len(switches)

    for i, frame in enumerate(switches):

        frame = int(frame)

        old_side = str(reference[frame - 1])
        new_side = str(reference[frame])

        # Grenzen der beiden benachbarten Wechsel
        left = (
            int(switches[i - 1])
            if i > 0 else int(seg_start)
        )

        right = (
            int(switches[i + 1])
            if i + 1 < len(switches)
            else int(seg_stop)
        )

        # Maximal die halbe Distanz zu den Nachbarwechseln.
        # Dadurch bleiben Reihenfolge und alle kurzen
        # Ballbesitzphasen erhalten.
        radius = min(
            MAX_SHIFT_FRAMES,
            (frame - left - 1) // 2,
            (right - frame - 1) // 2
        )

        if radius <= 0:
            continue

        possible = crossings[new_side]

        lo = np.searchsorted(
            possible, frame - radius, side="left"
        )

        hi = np.searchsorted(
            possible, frame + radius, side="right"
        )

        candidates = possible[lo:hi]

        if not len(candidates):
            continue

        # Ausschließlich räumlich nächster Kandidat
        # zum MODELLWECHSEL, nicht zur Ground Truth!
        spatial_frame = int(
            candidates[
                np.argmin(np.abs(candidates - frame))
            ]
        )

        shift = spatial_frame - frame

        if shift == 0:
            continue

        # Früherer Wechsel: neues Team früher beginnen
        if shift < 0:
            hier_spatial_hybrid_v1[
                spatial_frame:frame
            ] = new_side

        # Späterer Wechsel: vorheriges Team länger behalten
        else:
            hier_spatial_hybrid_v1[
                frame:spatial_frame
            ] = old_side

        records.append({
            "original_frame": frame,
            "aligned_frame": spatial_frame,
            "shift_seconds": shift / FPS,
            "from": old_side,
            "to": new_side,
            "segment_start": int(seg_start),
            "segment_end": int(seg_stop - 1),
            "public": bool(
                np.isin(
                    frame, oof_frames
                )
            )
        })

df_hybrid_alignments = pd.DataFrame(records)

# ------------------------------------------------------------
# 2. Sicherheitsprüfungen
# ------------------------------------------------------------

assert len(hier_spatial_hybrid_v1) == N_FRAMES

# Keine Veränderung außerhalb von LIVE
assert np.array_equal(
    hier_spatial_hybrid_v1[~live_mask],
    reference[~live_mask]
)

# D und X vollständig unverändert
assert np.array_equal(
    hier_spatial_hybrid_v1[
        np.isin(reference, ["D", "X"])
    ],
    reference[
        np.isin(reference, ["D", "X"])
    ]
)

# Auf LIVE-Fragments ausschließlich H/A
assert np.isin(
    hier_spatial_hybrid_v1[live_mask],
    ["H", "A"]
).all()

# Anzahl und Abfolge der H/A-Wechsel müssen gleich sein.
def transition_sequence(labels, starts, stops):
    sequence = []

    for a, b in zip(starts, stops):
        if b <= a:
            continue

        segment = labels[a:b]
        changes = np.r_[
            True, segment[1:] != segment[:-1]
        ]

        sequence.append(
            tuple(segment[changes])
        )

    return sequence

assert transition_sequence(
    reference, starts, stops
) == transition_sequence(
    hier_spatial_hybrid_v1, starts, stops
), "Wechselreihenfolge oder Ballbesitzphasen verändert!"

changed = hier_spatial_hybrid_v1 != reference

print("=== HYBRID-DIAGNOSTIK ===")
print("Direkte H/A-Wechsel gesamtes Spiel:",
      n_original_switches)
print("Räumlich verschobene Wechsel:",
      len(df_hybrid_alignments))
print("Veränderte H/A-Frames:", int(changed.sum()))
print("Veränderte öffentliche Frames:",
      int(changed[oof_frames].sum()))
print("Veränderte D-Frames:", int(np.sum(
    (reference == "D") & changed
)))

print("\nÖffentliche Wechselverschiebungen:")

display(
    df_hybrid_alignments[
        df_hybrid_alignments["public"]
    ].round(3)
)

# ------------------------------------------------------------
# 3. Vergleich der direkten H/A-Wechsel
# ------------------------------------------------------------

public_frames = np.sort(oof_frames)
gt_public = gt_lookup.reindex(public_frames).to_numpy()

comparison = []

for name, labels in [
    ("Physics v3", reference),
    ("Spatial Hybrid v1", hier_spatial_hybrid_v1)
]:

    public_pred = labels[public_frames]

    events = find_direct_switches(
        public_pred, public_frames
    )

    wrong_team = int(np.sum(
        np.isin(gt_public, ["H", "A"])
        & np.isin(public_pred, ["H", "A"])
        & (gt_public != public_pred)
    ))

    for tol in [1, 2, 5]:

        matches = match_events(
            gt_events_58, events, tol * FPS
        )

        comparison.append({
            "Modell": name,
            "Toleranz_s": tol,
            "GT_Matches": len(matches),
            "Vorhergesagte_Wechsel": len(events),
            "Falsche_HA_Frames": wrong_team
        })

print("\n=== DIREKTE WECHSEL ===")
display(pd.DataFrame(comparison))

# ------------------------------------------------------------
# 4. Welche öffentlichen H/A-Frames wurden korrigiert?
# ------------------------------------------------------------

old_public = reference[public_frames]
new_public = hier_spatial_hybrid_v1[public_frames]

fixed = (
    (old_public != gt_public)
    & (new_public == gt_public)
)

broken = (
    (old_public == gt_public)
    & (new_public != gt_public)
)

print("\nÖffentliche Frameänderungen:")
print("Vorher falsch, jetzt richtig:", int(fixed.sum()))
print("Vorher richtig, jetzt falsch:", int(broken.sum()))
print("Netto:", int(fixed.sum() - broken.sum()))

# ------------------------------------------------------------
# 5. Offizielle Evaluation
# ------------------------------------------------------------

OUT = PROJECT_DIR / "output/hierarchical_spatial_hybrid_v1"
SUB = OUT / "submissions"
EVAL = OUT / "evaluation"

SUB.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

models = {
    "physics_v3_reference": reference,
    "spatial_hybrid_v1": hier_spatial_hybrid_v1
}

for name, labels in models.items():
    pd.DataFrame({
        "frame": np.arange(N_FRAMES),
        "label": labels
    }).to_csv(
        SUB / f"{name}.csv",
        index=False
    )

ARCHIVE = (
    PROJECT_DIR
    / "reference/mtah2027_official_3de6b60.zip"
)

with TemporaryDirectory() as tmp:

    with ZipFile(ARCHIVE) as archive:
        root_name = archive.namelist()[0]
        archive.extractall(tmp)

    official = Path(tmp) / root_name

    validator = official / "evaluation/validate_submission.py"
    evaluator = official / "evaluation/run_evaluation.py"

    for name in models:
        subprocess.run(
            [
                sys.executable,
                str(validator),
                str(SUB / f"{name}.csv")
            ],
            check=True,
            capture_output=True,
            text=True
        )

    subprocess.run(
        [
            sys.executable,
            str(evaluator),
            "--gt", "public",
            "--results", str(SUB),
            "--out", str(EVAL),
            "--no-detail"
        ],
        check=True,
        capture_output=True,
        text=True
    )

scores = pd.read_csv(EVAL / "scoreboard.csv")

print("\n=== OFFIZIELLER ÖFFENTLICHER SCORE ===")

display(
    scores[
        [
            "submission",
            "macro_f1",
            "spell_f1_mean",
            "f1_H",
            "f1_A",
            "f1_D",
            "poss"
        ]
    ].sort_values(
        "spell_f1_mean",
        ascending=False
    ).round(4)
)

# ------------------------------------------------------------
# 6. Ergebnisse sichern
# ------------------------------------------------------------

df_hybrid_alignments.to_csv(
    OUT / "spatial_alignment_diagnostics.csv",
    index=False
)

print("\nHybrid-Test abgeschlossen.")


In [ ]:

# ============================================================
# SCHRITT 61: SPATIAL HYBRID – STABILITÄT UND VIDEO-AUDIT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

align = df_hybrid_alignments.copy()
align["abs_shift_s"] = align["shift_seconds"].abs()

# ------------------------------------------------------------
# 1. Verteilung der Verschiebungen
# ------------------------------------------------------------

print("=== GLOBALE VERSCHIEBUNGEN ===")
print("Anzahl:", len(align))
print("Früher:", int((align["shift_seconds"] < 0).sum()))
print("Später:", int((align["shift_seconds"] > 0).sum()))
print("Median absolut:",
      round(align["abs_shift_s"].median(), 3), "s")
print("90. Perzentil:",
      round(align["abs_shift_s"].quantile(0.90), 3), "s")
print("Maximum:",
      round(align["abs_shift_s"].max(), 3), "s")

align["half"] = np.where(
    align["original_frame"] <= HALVES[0][2],
    "1. Halbzeit",
    "2. Halbzeit"
)

align["shift_direction"] = np.where(
    align["shift_seconds"] < 0,
    "Früher",
    "Später"
)

print("\n=== NACH HALBZEIT ===")
display(pd.crosstab(
    align["half"],
    align["shift_direction"],
    margins=True
))

# ------------------------------------------------------------
# 2. Bereits videogeprüfte Szenen ausschließen
# ------------------------------------------------------------

reviewed = np.array([
    18026, 19772, 21884, 31170, 42853,
    53142, 54785, 67524, 69845,
    83578, 86237, 136650, 132924,
    142570, 143920, 147953,
    151893, 164019
], dtype=int)

PUBLIC_FIRST = int(oof_frames.min())
PUBLIC_LAST = int(oof_frames.max())

candidates = align.copy()

# Keine Wechsel im öffentlichen GT-Bereich
candidates = candidates[
    (candidates["original_frame"] < PUBLIC_FIRST)
    | (candidates["original_frame"] > PUBLIC_LAST)
].copy()

# Bereits bekannte Videosequenzen und Nachbarschaft ausschließen
candidates = candidates[
    candidates["original_frame"].apply(
        lambda frame: np.all(
            np.abs(reviewed - int(frame)) > 12 * FPS
        )
    )
].copy()

# ------------------------------------------------------------
# 3. Nach Verschiebungsgröße schichten
# ------------------------------------------------------------

candidates["shift_group"] = pd.cut(
    candidates["abs_shift_s"],
    bins=[0.0, 0.5, 1.5, np.inf],
    labels=["Klein", "Mittel", "Groß"],
    right=False
)

print("\n=== AUDIT-GRUNDMENGE ===")
display(
    candidates["shift_group"]
    .value_counts()
    .rename_axis("Verschiebung")
    .reset_index(name="Kandidaten")
)

# ------------------------------------------------------------
# 4. Sechs Szenen reproduzierbar auswählen
# ------------------------------------------------------------

rng = np.random.default_rng(2027)
selected = []
selected_centers = []

for group_name in ["Groß", "Mittel", "Klein"]:

    group = candidates[
        candidates["shift_group"] == group_name
    ]

    for idx in rng.permutation(group.index.to_numpy()):

        row = candidates.loc[idx]
        center = int(row.original_frame)

        # Keine unmittelbar benachbarten Auditclips
        if any(
            abs(center - previous) < 15 * FPS
            for previous in selected_centers
        ):
            continue

        selected.append(row.to_dict())
        selected_centers.append(center)

        if sum(
            r["shift_group"] == group_name
            for r in selected
        ) >= 2:
            break

df_spatial_video_audit = pd.DataFrame(selected)

def format_video_time(frame):
    sec = int(frame) / FPS
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

if len(df_spatial_video_audit):

    # 5 Sekunden Kontext auf beiden Seiten
    df_spatial_video_audit["clip_start_frame"] = (
        df_spatial_video_audit[
            ["original_frame", "aligned_frame"]
        ].min(axis=1).astype(int) - 5 * FPS
    ).clip(lower=0)

    df_spatial_video_audit["clip_end_frame"] = (
        df_spatial_video_audit[
            ["original_frame", "aligned_frame"]
        ].max(axis=1).astype(int) + 5 * FPS
    ).clip(upper=N_FRAMES - 1)

    df_spatial_video_audit = (
        df_spatial_video_audit
        .sort_values("clip_start_frame")
        .reset_index(drop=True)
    )

    df_spatial_video_audit["audit_id"] = [
        f"SH{i+1:02d}"
        for i in range(len(df_spatial_video_audit))
    ]

    df_spatial_video_audit["clip_start"] = (
        df_spatial_video_audit[
            "clip_start_frame"
        ].apply(format_video_time)
    )

    df_spatial_video_audit["clip_end"] = (
        df_spatial_video_audit[
            "clip_end_frame"
        ].apply(format_video_time)
    )

    print("\n=== NEUE VIDEO-AUDIT-SZENEN ===")

    # Keine ursprünglichen oder verschobenen
    # Wechselzeitpunkte anzeigen.
    display(
        df_spatial_video_audit[
            ["audit_id", "clip_start", "clip_end"]
        ]
    )

# ------------------------------------------------------------
# 5. Vollständige Zuordnung lokal speichern
# ------------------------------------------------------------

OUT = PROJECT_DIR / "output/hierarchical_spatial_hybrid_v1"
OUT.mkdir(parents=True, exist_ok=True)

df_spatial_video_audit.to_csv(
    OUT / "spatial_video_audit_blinded.csv",
    index=False
)

print("\nAudit-Auswahl gespeichert.")
print("Keine Vorhersagen verändert.")
